# FloodShift — Preprocessing

**Canonical notebook:** `03_FloodShift_Preprocessing`

This cleaned notebook contains one execution path only. Superseded DP-SSD normalization/DataLoader cells and emergency recovery cells have been removed.

**Execution rule:** run from top to bottom. The canonical FloodShift validation and test splits are not used for DP-SSD training or hyperparameter selection.


## 1. Canonical data connection and raw SAR audit


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C01
# SECTION: P3-S0 — SETUP
#
# PURPOSE:
#   Locate the frozen canonical raw FloodShift dataset.
#   Load the master manifest.
#
# NO preprocessing is performed in this cell.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# Locate canonical manifest automatically
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")

manifest_candidates = list(
    INPUT_ROOT.rglob("master_manifest.csv")
)

if not manifest_candidates:
    raise FileNotFoundError(
        "master_manifest.csv not found.\n"
        "Attach floodshift-raw-canonical-v1 through Add Data."
    )

print("Found manifest candidates:")

for p in manifest_candidates:
    print(" ", p)

MANIFEST_PATH = manifest_candidates[0]

master = pd.read_csv(MANIFEST_PATH)

print("\n" + "=" * 72)
print("P3-C01 — CANONICAL DATASET CONNECTION")
print("=" * 72)

print("Manifest :", MANIFEST_PATH)
print("Scenes   :", len(master))

print("\nSplit counts:")
print(master["split"].value_counts())

print("\nRegions:")
print(master["region"].value_counts().sort_index())

assert len(master) == 440
assert master["scene_id"].is_unique

print("\nCanonical raw dataset connected ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C02
# SECTION: P3-S1 — RAW SAR UNDERSTANDING
#
# PURPOSE:
#   Inspect one Sentinel-1 scene BEFORE touching the data.
#
# Questions answered:
#   - How many bands?
#   - VV/VH value ranges?
#   - dtype?
#   - non-finite pixels?
#   - spatial dimensions?
# ============================================================

import rasterio
import numpy as np

# Use the first TRAIN scene only
sample = master[
    master["split"] == "train"
].iloc[0]

scene_id = sample["scene_id"]

# Important:
# Paths saved inside the old manifest may point to the
# previous /kaggle/working location.
#
# Therefore locate the file by filename inside /kaggle/input.

sar_filename = Path(
    sample["sar_raw_path"]
).name

sar_candidates = list(
    INPUT_ROOT.rglob(sar_filename)
)

if not sar_candidates:
    raise FileNotFoundError(
        f"Could not locate {sar_filename}"
    )

sar_path = sar_candidates[0]

with rasterio.open(sar_path) as src:

    sar = src.read()

    print("=" * 72)
    print("P3-C02 — RAW SENTINEL-1 INSPECTION")
    print("=" * 72)

    print("Scene      :", scene_id)
    print("Path       :", sar_path)
    print("Shape      :", sar.shape)
    print("Dtype      :", sar.dtype)
    print("CRS        :", src.crs)
    print("Resolution :", src.res)

print("\nOverall finite:")
print(np.isfinite(sar).all())

for band_idx, band_name in enumerate(["VV", "VH"]):

    x = sar[band_idx]

    finite = np.isfinite(x)

    print("\n" + "-" * 40)
    print(band_name)
    print("-" * 40)

    print("Finite pixels    :", finite.sum())
    print("Non-finite pixels:", (~finite).sum())

    if finite.any():

        values = x[finite]

        print("Minimum :", values.min())
        print("Maximum :", values.max())
        print("Mean    :", values.mean())
        print("Std     :", values.std())

        print(
            "Percentiles:",
            np.percentile(
                values,
                [0.5, 1, 5, 50, 95, 99, 99.5]
            )
        )


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C03
# SECTION: P3-S2 — SAR VALIDITY / NO-DATA AUDIT
#
# PURPOSE:
#   1. Check whether VV and VH share the same valid footprint.
#   2. Inspect GeoTIFF no-data metadata.
#   3. Compare SAR validity against the hand-label validity.
#   4. Create the canonical SAR validity mask.
#
# NO filtering is performed here.
# ============================================================

from pathlib import Path
import numpy as np
import rasterio

# ------------------------------------------------------------
# Resolve matching label file from canonical input
# ------------------------------------------------------------

label_filename = Path(
    sample["flood_label_path"]
).name

label_candidates = list(
    INPUT_ROOT.rglob(label_filename)
)

if not label_candidates:
    raise FileNotFoundError(
        f"Could not locate {label_filename}"
    )

label_path = label_candidates[0]


# ------------------------------------------------------------
# Read SAR + metadata
# ------------------------------------------------------------

with rasterio.open(sar_path) as src:

    sar = src.read()

    print("=" * 72)
    print("P3-C03 — SAR VALIDITY / NO-DATA AUDIT")
    print("=" * 72)

    print("Scene          :", scene_id)
    print("Raster nodata  :", src.nodata)
    print("Nodata values  :", src.nodatavals)
    print("Dataset mask   :", np.unique(src.dataset_mask(), return_counts=True))


# ------------------------------------------------------------
# Read hand label
# ------------------------------------------------------------

with rasterio.open(label_path) as src:
    label = src.read(1)

# ------------------------------------------------------------
# Construct masks
# ------------------------------------------------------------

vv_valid = np.isfinite(sar[0])
vh_valid = np.isfinite(sar[1])

# Canonical SAR pixel is valid only if BOTH polarizations exist
sar_valid = vv_valid & vh_valid

# Supervised-label validity
label_valid = label != -1


# ------------------------------------------------------------
# Compare VV and VH footprints
# ------------------------------------------------------------

same_footprint = np.array_equal(
    vv_valid,
    vh_valid
)

vv_only = vv_valid & ~vh_valid
vh_only = vh_valid & ~vv_valid


# ------------------------------------------------------------
# Compare SAR validity with supervision
# ------------------------------------------------------------

both_valid = sar_valid & label_valid

label_valid_sar_invalid = label_valid & ~sar_valid
sar_valid_label_invalid = sar_valid & ~label_valid


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

N = sar_valid.size

def pct(n):
    return 100.0 * n / N


print("\n" + "-" * 72)
print("POLARIZATION VALIDITY")
print("-" * 72)

print(
    f"VV valid              : {vv_valid.sum():,} "
    f"({pct(vv_valid.sum()):.2f}%)"
)

print(
    f"VH valid              : {vh_valid.sum():,} "
    f"({pct(vh_valid.sum()):.2f}%)"
)

print(
    f"Both VV + VH valid    : {sar_valid.sum():,} "
    f"({pct(sar_valid.sum()):.2f}%)"
)

print("Same VV/VH footprint  :", same_footprint)
print("VV-only valid pixels  :", int(vv_only.sum()))
print("VH-only valid pixels  :", int(vh_only.sum()))


print("\n" + "-" * 72)
print("SAR vs LABEL VALIDITY")
print("-" * 72)

print(
    f"Label-valid pixels          : {label_valid.sum():,} "
    f"({pct(label_valid.sum()):.2f}%)"
)

print(
    f"SAR + label both valid      : {both_valid.sum():,} "
    f"({pct(both_valid.sum()):.2f}%)"
)

print(
    f"Label valid, SAR invalid     : "
    f"{label_valid_sar_invalid.sum():,} "
    f"({pct(label_valid_sar_invalid.sum()):.2f}%)"
)

print(
    f"SAR valid, label invalid     : "
    f"{sar_valid_label_invalid.sum():,} "
    f"({pct(sar_valid_label_invalid.sum()):.2f}%)"
)


# ------------------------------------------------------------
# Safety assertions
# ------------------------------------------------------------

assert sar.shape == (2, 512, 512)

print("\nCanonical SAR validity mask created ✅")
print("sar_valid shape:", sar_valid.shape)
print("sar_valid dtype:", sar_valid.dtype)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C04
# SECTION: P3-S2 — SAFE dB <-> LINEAR CONVERSION
#
# PURPOSE:
#   Verify mathematically correct conversion between
#   Sentinel-1 dB values and linear intensity.
#
# IMPORTANT:
#   - Original NaN/no-data pixels remain NaN.
#   - No filtering is performed.
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Conversion functions
# ------------------------------------------------------------

def db_to_linear(x_db, valid_mask=None):
    """
    Convert SAR dB intensity to linear intensity.

    Invalid positions remain NaN.
    """
    x_db = np.asarray(x_db, dtype=np.float32)

    if valid_mask is None:
        valid_mask = np.isfinite(x_db)

    out = np.full_like(
        x_db,
        np.nan,
        dtype=np.float32
    )

    out[valid_mask] = np.power(
        10.0,
        x_db[valid_mask] / 10.0
    )

    return out


def linear_to_db(x_linear, valid_mask=None, eps=1e-12):
    """
    Convert linear SAR intensity back to dB.

    Invalid positions remain NaN.
    """
    x_linear = np.asarray(
        x_linear,
        dtype=np.float32
    )

    if valid_mask is None:
        valid_mask = np.isfinite(x_linear)

    out = np.full_like(
        x_linear,
        np.nan,
        dtype=np.float32
    )

    out[valid_mask] = (
        10.0
        * np.log10(
            np.maximum(
                x_linear[valid_mask],
                eps
            )
        )
    )

    return out


# ------------------------------------------------------------
# Convert both polarizations
# ------------------------------------------------------------

sar_linear = np.empty_like(
    sar,
    dtype=np.float32
)

sar_roundtrip = np.empty_like(
    sar,
    dtype=np.float32
)

for band_idx in range(2):

    valid = sar_valid

    sar_linear[band_idx] = db_to_linear(
        sar[band_idx],
        valid
    )

    sar_roundtrip[band_idx] = linear_to_db(
        sar_linear[band_idx],
        valid
    )


# ------------------------------------------------------------
# Verify reconstruction accuracy
# ------------------------------------------------------------

print("=" * 72)
print("P3-C04 — dB ↔ LINEAR ROUND-TRIP TEST")
print("=" * 72)

for idx, name in enumerate(["VV", "VH"]):

    valid = sar_valid

    original = sar[idx][valid]
    restored = sar_roundtrip[idx][valid]

    abs_error = np.abs(
        original - restored
    )

    print(f"\n{name}")

    print(
        "Linear range:",
        float(np.nanmin(sar_linear[idx])),
        "to",
        float(np.nanmax(sar_linear[idx]))
    )

    print(
        "Max round-trip error :",
        float(abs_error.max())
    )

    print(
        "Mean round-trip error:",
        float(abs_error.mean())
    )

    print(
        "Invalid pixels preserved:",
        np.isnan(
            sar_roundtrip[idx][~valid]
        ).all()
    )


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert np.isnan(
    sar_roundtrip[:, ~sar_valid]
).all()

assert np.allclose(
    sar[:, sar_valid],
    sar_roundtrip[:, sar_valid],
    atol=1e-5
)

print("\nSafe dB ↔ linear conversion verified ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C05
# SECTION: P3-S3 — TRAIN SAR DISTRIBUTION AUDIT
#
# PURPOSE:
#   Characterize raw Sentinel-1 VV/VH across TRAIN scenes only.
#
#   We measure:
#   - valid pixel coverage
#   - non-finite coverage
#   - global train distribution
#   - robust percentiles
#
# IMPORTANT:
#   No validation/test statistics are used.
#   No filtering is performed.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

train_df = master[master["split"] == "train"].copy()

print("Training scenes:", len(train_df))

# ------------------------------------------------------------
# Helper: locate file inside attached Kaggle input
# ------------------------------------------------------------

def locate_input_file(saved_path):
    filename = Path(saved_path).name
    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(filename)

    return matches[0]


# ------------------------------------------------------------
# Collect statistics
# ------------------------------------------------------------

scene_rows = []

vv_chunks = []
vh_chunks = []

# To avoid storing every pixel from all 314 scenes,
# sample a maximum number of valid pixels per scene.
MAX_PIXELS_PER_SCENE = 50_000

rng = np.random.default_rng(42)

for idx, row in train_df.iterrows():

    sar_path_i = locate_input_file(
        row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        x = src.read().astype(np.float32)

    vv = x[0]
    vh = x[1]

    valid = (
        np.isfinite(vv)
        &
        np.isfinite(vh)
    )

    valid_count = int(valid.sum())
    total_count = int(valid.size)

    if valid_count > 0:

        vv_valid = vv[valid]
        vh_valid = vh[valid]

        # --------------------------------------------
        # Random subsample for global distribution
        # --------------------------------------------

        if valid_count > MAX_PIXELS_PER_SCENE:

            sample_idx = rng.choice(
                valid_count,
                size=MAX_PIXELS_PER_SCENE,
                replace=False
            )

            vv_sample = vv_valid[sample_idx]
            vh_sample = vh_valid[sample_idx]

        else:

            vv_sample = vv_valid
            vh_sample = vh_valid

        vv_chunks.append(vv_sample)
        vh_chunks.append(vh_sample)

        vv_min = float(vv_valid.min())
        vv_max = float(vv_valid.max())
        vv_mean = float(vv_valid.mean())
        vv_std = float(vv_valid.std())

        vh_min = float(vh_valid.min())
        vh_max = float(vh_valid.max())
        vh_mean = float(vh_valid.mean())
        vh_std = float(vh_valid.std())

    else:

        vv_min = vv_max = vv_mean = vv_std = np.nan
        vh_min = vh_max = vh_mean = vh_std = np.nan

    scene_rows.append({
        "scene_id": row["scene_id"],
        "region": row["region"],

        "valid_pixels": valid_count,
        "invalid_pixels": total_count - valid_count,
        "valid_pct": 100.0 * valid_count / total_count,

        "vv_min_db": vv_min,
        "vv_max_db": vv_max,
        "vv_mean_db": vv_mean,
        "vv_std_db": vv_std,

        "vh_min_db": vh_min,
        "vh_max_db": vh_max,
        "vh_mean_db": vh_mean,
        "vh_std_db": vh_std,
    })


scene_stats = pd.DataFrame(scene_rows)

vv_global = np.concatenate(vv_chunks)
vh_global = np.concatenate(vh_chunks)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("P3-C05 — TRAIN SAR DISTRIBUTION AUDIT")
print("=" * 72)

print("\nVALID COVERAGE")
print("-" * 72)

print(
    scene_stats["valid_pct"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

print("\nScenes with < 25% SAR coverage:",
      int((scene_stats["valid_pct"] < 25).sum()))

print("Scenes with < 50% SAR coverage:",
      int((scene_stats["valid_pct"] < 50).sum()))

print("Scenes with 100% SAR coverage:",
      int((scene_stats["valid_pct"] == 100).sum()))


# ------------------------------------------------------------
# Robust global distribution
# ------------------------------------------------------------

percentiles = [
    0.1,
    0.5,
    1,
    2,
    5,
    25,
    50,
    75,
    95,
    98,
    99,
    99.5,
    99.9
]

print("\nVV TRAIN DISTRIBUTION [dB]")
print("-" * 72)

for p, v in zip(
    percentiles,
    np.percentile(vv_global, percentiles)
):
    print(f"{p:>5.1f}% : {v:>10.4f}")


print("\nVH TRAIN DISTRIBUTION [dB]")
print("-" * 72)

for p, v in zip(
    percentiles,
    np.percentile(vh_global, percentiles)
):
    print(f"{p:>5.1f}% : {v:>10.4f}")


print("\nGLOBAL SAMPLE SIZE")
print("-" * 72)

print("VV sampled pixels:", f"{len(vv_global):,}")
print("VH sampled pixels:", f"{len(vh_global):,}")


# ------------------------------------------------------------
# Save audit locally in this notebook
# ------------------------------------------------------------

OUTPUT_ROOT = Path(
    "/kaggle/working/FloodShift_Preprocessing"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

scene_stats.to_csv(
    OUTPUT_ROOT / "train_raw_sar_scene_statistics.csv",
    index=False
)

print(
    "\nSaved:",
    OUTPUT_ROOT / "train_raw_sar_scene_statistics.csv"
)

print("\nTRAIN-only SAR audit complete ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C06
# SECTION: P3-S3 — REPRESENTATIVE RAW SAR VISUALIZATION
#
# PURPOSE:
#   Inspect representative training scenes before despeckling.
#
#   Automatically selects:
#     1. low-texture scene
#     2. medium-texture scene
#     3. high-texture scene
#
#   Selection uses SAR statistics only — NOT labels.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import rasterio

# ------------------------------------------------------------
# Use only scenes with complete SAR coverage
# ------------------------------------------------------------

full_coverage = scene_stats[
    scene_stats["valid_pct"] == 100
].copy()

# VV standard deviation is used only as a simple
# scene-level texture proxy.
texture_values = full_coverage["vv_std_db"].values

targets = {
    "LOW texture": np.percentile(texture_values, 10),
    "MEDIUM texture": np.percentile(texture_values, 50),
    "HIGH texture": np.percentile(texture_values, 90),
}

selected_rows = []

for label_name, target_std in targets.items():

    temp = full_coverage.copy()

    temp["distance"] = np.abs(
        temp["vv_std_db"] - target_std
    )

    row = temp.sort_values(
        "distance"
    ).iloc[0]

    selected_rows.append(
        (label_name, row)
    )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

fig, axes = plt.subplots(
    len(selected_rows),
    2,
    figsize=(12, 14)
)

for i, (category, row) in enumerate(selected_rows):

    original_row = train_df[
        train_df["scene_id"] == row["scene_id"]
    ].iloc[0]

    path = locate_input_file(
        original_row["sar_raw_path"]
    )

    with rasterio.open(path) as src:
        x = src.read().astype(np.float32)

    vv = x[0]
    vh = x[1]

    # --------------------------------------------------------
    # Display limits only
    #
    # These DO NOT alter the underlying SAR.
    # Robust percentiles simply improve visualization.
    # --------------------------------------------------------

    vv_valid = vv[np.isfinite(vv)]
    vh_valid = vh[np.isfinite(vh)]

    vv_lo, vv_hi = np.percentile(
        vv_valid,
        [2, 98]
    )

    vh_lo, vh_hi = np.percentile(
        vh_valid,
        [2, 98]
    )

    axes[i, 0].imshow(
        vv,
        cmap="gray",
        vmin=vv_lo,
        vmax=vv_hi
    )

    axes[i, 0].set_title(
        f"{category} — {row['scene_id']} — VV\n"
        f"scene std = {row['vv_std_db']:.2f} dB"
    )

    axes[i, 0].axis("off")

    axes[i, 1].imshow(
        vh,
        cmap="gray",
        vmin=vh_lo,
        vmax=vh_hi
    )

    axes[i, 1].set_title(
        f"{category} — {row['scene_id']} — VH\n"
        f"scene std = {row['vh_std_db']:.2f} dB"
    )

    axes[i, 1].axis("off")


plt.tight_layout()
plt.show()


print("=" * 72)
print("P3-C06 — REPRESENTATIVE SCENES")
print("=" * 72)

for category, row in selected_rows:

    print(
        f"{category:<16} "
        f"{row['scene_id']:<25} "
        f"VV std={row['vv_std_db']:.3f} dB   "
        f"VH std={row['vh_std_db']:.3f} dB"
    )

print("\nRaw SAR visualization complete ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C07
# SECTION: P3-S3 — AUTOMATIC HOMOGENEOUS PATCH SELECTION
#
# PURPOSE:
#   Automatically identify low-variation SAR regions that can
#   later be reused for objective speckle evaluation.
#
# IMPORTANT:
#   - Uses SAR only.
#   - Uses TRAIN scenes only.
#   - Does NOT use flood labels.
#   - Patch coordinates are frozen once selected.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

PATCH_SIZE = 32
STRIDE = 8

patch_records = []


def patch_statistics(x_linear):
    """
    Calculate mean, std, coefficient of variation and ENL
    for a finite linear-intensity patch.
    """

    values = x_linear[np.isfinite(x_linear)]

    if len(values) == 0:
        return None

    mean = float(values.mean())
    std = float(values.std())

    if mean <= 0:
        return None

    cv = std / mean

    enl = (
        (mean ** 2) / (std ** 2)
        if std > 0
        else np.inf
    )

    return mean, std, cv, enl


# ------------------------------------------------------------
# Search each representative scene
# ------------------------------------------------------------

for category, stats_row in selected_rows:

    scene_id_i = stats_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        x_db = src.read().astype(np.float32)

    vv_db = x_db[0]
    vh_db = x_db[1]

    valid = (
        np.isfinite(vv_db)
        &
        np.isfinite(vh_db)
    )

    # --------------------------------------------------------
    # Convert to linear intensity
    # --------------------------------------------------------

    vv_linear = db_to_linear(
        vv_db,
        valid
    )

    vh_linear = db_to_linear(
        vh_db,
        valid
    )

    candidates = []

    H, W = valid.shape

    for y in range(
        0,
        H - PATCH_SIZE + 1,
        STRIDE
    ):

        for x in range(
            0,
            W - PATCH_SIZE + 1,
            STRIDE
        ):

            valid_patch = valid[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            # Require complete valid SAR coverage
            if not valid_patch.all():
                continue

            vv_patch = vv_linear[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            vh_patch = vh_linear[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            vv_stats = patch_statistics(vv_patch)
            vh_stats = patch_statistics(vh_patch)

            if vv_stats is None or vh_stats is None:
                continue

            vv_mean, vv_std, vv_cv, vv_enl = vv_stats
            vh_mean, vh_std, vh_cv, vh_enl = vh_stats

            # ------------------------------------------------
            # Combined score:
            # low CV in BOTH VV and VH is preferred.
            # ------------------------------------------------

            score = (vv_cv + vh_cv) / 2.0

            candidates.append({
                "scene_id": scene_id_i,
                "category": category,

                "x": x,
                "y": y,
                "patch_size": PATCH_SIZE,

                "vv_cv": vv_cv,
                "vh_cv": vh_cv,

                "vv_enl_raw": vv_enl,
                "vh_enl_raw": vh_enl,

                "score": score,
            })

    if not candidates:
        raise RuntimeError(
            f"No valid homogeneous patch found for {scene_id_i}"
        )

    candidates_df = pd.DataFrame(candidates)

    # Lowest combined CV = most homogeneous candidate
    best = candidates_df.sort_values(
        "score"
    ).iloc[0]

    patch_records.append(
        best.to_dict()
    )


# ------------------------------------------------------------
# Final frozen patch table
# ------------------------------------------------------------

homogeneous_patches = pd.DataFrame(
    patch_records
)

print("=" * 72)
print("P3-C07 — AUTOMATIC HOMOGENEOUS PATCHES")
print("=" * 72)

print(
    homogeneous_patches[
        [
            "category",
            "scene_id",
            "x",
            "y",
            "patch_size",
            "vv_cv",
            "vh_cv",
            "vv_enl_raw",
            "vh_enl_raw",
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Visual verification
# ------------------------------------------------------------

fig, axes = plt.subplots(
    len(homogeneous_patches),
    2,
    figsize=(11, 13)
)

for i, row in homogeneous_patches.iterrows():

    scene_id_i = row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        x_db = src.read().astype(np.float32)

    x0 = int(row["x"])
    y0 = int(row["y"])
    ps = int(row["patch_size"])

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        band = x_db[band_idx]

        finite = band[np.isfinite(band)]

        lo, hi = np.percentile(
            finite,
            [2, 98]
        )

        ax = axes[i, band_idx]

        ax.imshow(
            band,
            cmap="gray",
            vmin=lo,
            vmax=hi
        )

        rect = plt.Rectangle(
            (x0, y0),
            ps,
            ps,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rect)

        ax.set_title(
            f"{row['category']} — {scene_id_i}\n"
            f"{band_name} homogeneous patch"
        )

        ax.axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Save frozen patch coordinates
# ------------------------------------------------------------

patch_file = (
    OUTPUT_ROOT /
    "sar_homogeneous_evaluation_patches.csv"
)

homogeneous_patches.to_csv(
    patch_file,
    index=False
)

print("\nSaved:", patch_file)

print(
    "\nThese coordinates are now reusable across "
    "all SAR preprocessing methods ✅"
)


## 2. Existing SAR despeckling baselines


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C08
# SECTION: P3-S4 — REFINED LEE DESPECKLING
#
# PURPOSE:
#   1. Implement a directional Refined Lee filter.
#   2. Apply it in LINEAR intensity space.
#   3. Convert the result back to dB.
#   4. Compare Raw vs Refined Lee on the SAME frozen
#      homogeneous evaluation patches.
#
# IMPORTANT:
#   - TRAIN scenes only.
#   - No labels used.
#   - No normalization yet.
#   - No files are permanently generated yet.
# ============================================================

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from scipy.ndimage import (
    uniform_filter,
    convolve
)


# ------------------------------------------------------------
# Helper: reflect-safe spatial shift
# ------------------------------------------------------------

def shift_reflect(arr, dy, dx):
    """
    Shift array spatially without wrap-around.
    Reflective padding is used at boundaries.
    """

    H, W = arr.shape

    pad = max(
        abs(dy),
        abs(dx)
    )

    padded = np.pad(
        arr,
        pad_width=pad,
        mode="reflect"
    )

    y0 = pad + dy
    x0 = pad + dx

    return padded[
        y0:y0 + H,
        x0:x0 + W
    ]


# ------------------------------------------------------------
# Refined Lee
# ------------------------------------------------------------

def refined_lee_filter(
    image_linear,
    eps=1e-12
):
    """
    Directional Refined Lee filter for a positive
    SAR linear-intensity image.

    Input:
        image_linear : 2D linear-intensity SAR

    Returns:
        filtered linear-intensity SAR
    """

    img = np.asarray(
        image_linear,
        dtype=np.float64
    )

    if not np.isfinite(img).all():
        raise ValueError(
            "refined_lee_filter currently expects "
            "a fully valid test image."
        )

    # --------------------------------------------------------
    # STEP 1
    # 3x3 local mean and variance
    # --------------------------------------------------------

    mean3 = uniform_filter(
        img,
        size=3,
        mode="reflect"
    )

    mean_sq3 = uniform_filter(
        img ** 2,
        size=3,
        mode="reflect"
    )

    var3 = np.maximum(
        mean_sq3 - mean3 ** 2,
        0.0
    )


    # --------------------------------------------------------
    # STEP 2
    # Sample the 3x3 statistics at 9 locations distributed
    # across the surrounding 7x7 neighborhood.
    #
    # Layout:
    #
    #   0  1  2
    #   3  4  5
    #   6  7  8
    #
    # --------------------------------------------------------

    offsets = [
        (-2, -2), (-2,  0), (-2,  2),
        ( 0, -2), ( 0,  0), ( 0,  2),
        ( 2, -2), ( 2,  0), ( 2,  2),
    ]

    sample_mean = np.stack(
        [
            shift_reflect(
                mean3,
                dy,
                dx
            )
            for dy, dx in offsets
        ],
        axis=0
    )

    sample_var = np.stack(
        [
            shift_reflect(
                var3,
                dy,
                dx
            )
            for dy, dx in offsets
        ],
        axis=0
    )


    # --------------------------------------------------------
    # STEP 3
    # Four principal gradient magnitudes.
    # --------------------------------------------------------

    gradients = np.stack(
        [
            np.abs(
                sample_mean[1]
                -
                sample_mean[7]
            ),

            np.abs(
                sample_mean[6]
                -
                sample_mean[2]
            ),

            np.abs(
                sample_mean[3]
                -
                sample_mean[5]
            ),

            np.abs(
                sample_mean[0]
                -
                sample_mean[8]
            ),
        ],
        axis=0
    )

    max_gradient = np.argmax(
        gradients,
        axis=0
    )


    # --------------------------------------------------------
    # STEP 4
    # Determine one of 8 local directions.
    # --------------------------------------------------------

    d1 = (
        sample_mean[1] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[7]
    )

    d2 = (
        sample_mean[6] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[2]
    )

    d3 = (
        sample_mean[3] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[5]
    )

    d4 = (
        sample_mean[0] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[8]
    )

    first_directions = np.stack(
        [
            np.where(d1, 1, 5),
            np.where(d2, 2, 6),
            np.where(d3, 3, 7),
            np.where(d4, 4, 8),
        ],
        axis=0
    )

    yy, xx = np.indices(
        img.shape
    )

    direction = first_directions[
        max_gradient,
        yy,
        xx
    ]


    # --------------------------------------------------------
    # STEP 5
    # Estimate local speckle variance.
    #
    # sigmaV is estimated from the five lowest
    # normalized local variances among the 9 samples.
    # --------------------------------------------------------

    normalized_variance = (
        sample_var
        /
        (
            sample_mean ** 2
            +
            eps
        )
    )

    normalized_sorted = np.sort(
        normalized_variance,
        axis=0
    )

    sigmaV = np.mean(
        normalized_sorted[:5],
        axis=0
    )


    # --------------------------------------------------------
    # STEP 6
    # Directional kernels
    # --------------------------------------------------------

    rect_kernel = np.array(
        [
            [0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0],
            [1,1,1,1,1,1,1],
            [1,1,1,1,1,1,1],
            [1,1,1,1,1,1,1],
            [1,1,1,1,1,1,1],
        ],
        dtype=np.float64
    )

    diag_kernel = np.array(
        [
            [1,0,0,0,0,0,0],
            [1,1,0,0,0,0,0],
            [1,1,1,0,0,0,0],
            [1,1,1,1,0,0,0],
            [1,1,1,1,1,0,0],
            [1,1,1,1,1,1,0],
            [1,1,1,1,1,1,1],
        ],
        dtype=np.float64
    )

    kernels = []

    for rot in range(4):

        kernels.append(
            np.rot90(
                rect_kernel,
                rot
            )
        )

        kernels.append(
            np.rot90(
                diag_kernel,
                rot
            )
        )


    # --------------------------------------------------------
    # STEP 7
    # Calculate directional mean/variance.
    # --------------------------------------------------------

    dir_mean = np.zeros_like(
        img
    )

    dir_var = np.zeros_like(
        img
    )

    for direction_id, kernel in enumerate(
        kernels,
        start=1
    ):

        weight = kernel.sum()

        local_mean = convolve(
            img,
            kernel,
            mode="reflect"
        ) / weight

        local_mean_sq = convolve(
            img ** 2,
            kernel,
            mode="reflect"
        ) / weight

        local_var = np.maximum(
            local_mean_sq
            -
            local_mean ** 2,
            0.0
        )

        mask = (
            direction
            ==
            direction_id
        )

        dir_mean[mask] = local_mean[mask]
        dir_var[mask] = local_var[mask]


    # --------------------------------------------------------
    # STEP 8
    # Refined Lee adaptive weighting
    # --------------------------------------------------------

    var_x = (
        dir_var
        -
        (
            dir_mean ** 2
            *
            sigmaV
        )
    ) / (
        sigmaV
        +
        1.0
    )

    var_x = np.maximum(
        var_x,
        0.0
    )

    b = (
        var_x
        /
        (
            dir_var
            +
            eps
        )
    )

    b = np.clip(
        b,
        0.0,
        1.0
    )

    filtered = (
        dir_mean
        +
        b
        *
        (
            img
            -
            dir_mean
        )
    )

    filtered = np.maximum(
        filtered,
        eps
    )

    return filtered.astype(
        np.float32
    )


# ============================================================
# APPLY TO THE THREE FROZEN REFERENCE SCENES
# ============================================================

comparison_records = []

visual_records = []


for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]

    path = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(path) as src:

        raw_db = src.read().astype(
            np.float32
        )

    # These three reference scenes were selected
    # specifically from complete-coverage scenes.
    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    assert valid.all(), (
        f"{scene_id_i} unexpectedly contains invalid SAR."
    )

    filtered_db = np.empty_like(
        raw_db
    )


    for band_idx in range(2):

        # dB → linear
        linear = db_to_linear(
            raw_db[band_idx],
            valid
        )

        # Refined Lee in linear domain
        filtered_linear = refined_lee_filter(
            linear
        )

        # linear → dB
        filtered_db[band_idx] = linear_to_db(
            filtered_linear,
            valid
        )


    # --------------------------------------------------------
    # SAME frozen homogeneous patch
    # --------------------------------------------------------

    x0 = int(patch_row["x"])
    y0 = int(patch_row["y"])
    ps = int(patch_row["patch_size"])

    record = {
        "scene_id": scene_id_i,
        "category": patch_row["category"]
    }


    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        raw_patch_db = raw_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]

        lee_patch_db = filtered_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]

        # ENL must be calculated in linear intensity.
        raw_patch_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )

        lee_patch_linear = np.power(
            10.0,
            lee_patch_db / 10.0
        )

        raw_stats = patch_statistics(
            raw_patch_linear
        )

        lee_stats = patch_statistics(
            lee_patch_linear
        )

        raw_enl = raw_stats[3]
        lee_enl = lee_stats[3]

        record[
            f"{band_name.lower()}_raw_enl"
        ] = raw_enl

        record[
            f"{band_name.lower()}_refined_lee_enl"
        ] = lee_enl

        record[
            f"{band_name.lower()}_enl_gain"
        ] = lee_enl / raw_enl


    comparison_records.append(
        record
    )

    visual_records.append(
        (
            scene_id_i,
            patch_row["category"],
            raw_db,
            filtered_db
        )
    )


refined_lee_results = pd.DataFrame(
    comparison_records
)


# ============================================================
# REPORT
# ============================================================

print("=" * 78)
print("P3-C08 — REFINED LEE FIRST EVALUATION")
print("=" * 78)

print(
    refined_lee_results.to_string(
        index=False
    )
)


# ============================================================
# VISUAL COMPARISON
# ============================================================

fig, axes = plt.subplots(
    len(visual_records),
    4,
    figsize=(16, 13)
)


for i, (
    scene_id_i,
    category,
    raw_db,
    filtered_db
) in enumerate(visual_records):

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        raw_band = raw_db[band_idx]

        finite = raw_band[
            np.isfinite(raw_band)
        ]

        lo, hi = np.percentile(
            finite,
            [2, 98]
        )

        # RAW
        ax_raw = axes[
            i,
            band_idx * 2
        ]

        ax_raw.imshow(
            raw_band,
            cmap="gray",
            vmin=lo,
            vmax=hi
        )

        ax_raw.set_title(
            f"{category}\n"
            f"{scene_id_i} {band_name} RAW"
        )

        ax_raw.axis("off")


        # REFINED LEE
        ax_lee = axes[
            i,
            band_idx * 2 + 1
        ]

        ax_lee.imshow(
            filtered_db[band_idx],
            cmap="gray",
            vmin=lo,
            vmax=hi
        )

        ax_lee.set_title(
            f"{category}\n"
            f"{scene_id_i} {band_name} Refined Lee"
        )

        ax_lee.axis("off")


plt.tight_layout()
plt.show()


print(
    "\nRefined Lee test completed ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C09
# SECTION: P3-S4 — REFINED LEE EDGE PRESERVATION
#
# PURPOSE:
#   Quantify whether Refined Lee preserves strong SAR structure.
#
# METRICS:
#   EGR = Edge Gradient Retention
#   EGC = Edge Gradient Correlation
#
# IMPORTANT:
#   - Same three frozen TRAIN scenes.
#   - No labels used.
#   - Gaussian smoothing is ONLY for locating structural edges.
# ============================================================

import numpy as np
import pandas as pd

from scipy.ndimage import (
    gaussian_filter,
    sobel
)


# ------------------------------------------------------------
# Gradient magnitude
# ------------------------------------------------------------

def gradient_magnitude(image):
    gx = sobel(
        image,
        axis=1,
        mode="reflect"
    )

    gy = sobel(
        image,
        axis=0,
        mode="reflect"
    )

    return np.sqrt(
        gx**2 + gy**2
    )


def edge_preservation_metrics(
    raw_db,
    filtered_db,
    edge_percentile=90
):
    """
    Evaluate preservation of strong structural edges.

    Returns:
        EGR  - gradient strength retention
        EGC  - gradient correlation
        n_edges
    """

    # --------------------------------------------------------
    # Light smoothing ONLY to identify reliable structures.
    # This suppresses tiny speckle-induced gradients.
    # --------------------------------------------------------

    raw_for_edges = gaussian_filter(
        raw_db,
        sigma=1.0,
        mode="reflect"
    )

    structural_gradient = gradient_magnitude(
        raw_for_edges
    )

    threshold = np.percentile(
        structural_gradient,
        edge_percentile
    )

    edge_mask = (
        structural_gradient >= threshold
    )


    # --------------------------------------------------------
    # Gradients of actual raw + filtered images
    # --------------------------------------------------------

    raw_gradient = gradient_magnitude(
        raw_db
    )

    filtered_gradient = gradient_magnitude(
        filtered_db
    )


    raw_edge_grad = raw_gradient[
        edge_mask
    ]

    filtered_edge_grad = filtered_gradient[
        edge_mask
    ]


    # --------------------------------------------------------
    # Edge Gradient Retention
    # --------------------------------------------------------

    egr = (
        filtered_edge_grad.sum()
        /
        (raw_edge_grad.sum() + 1e-12)
    )


    # --------------------------------------------------------
    # Edge Gradient Correlation
    # --------------------------------------------------------

    if (
        raw_edge_grad.std() > 0
        and
        filtered_edge_grad.std() > 0
    ):

        egc = np.corrcoef(
            raw_edge_grad,
            filtered_edge_grad
        )[0, 1]

    else:

        egc = np.nan


    return {
        "egr": float(egr),
        "egc": float(egc),
        "edge_pixels": int(edge_mask.sum())
    }


# ============================================================
# CALCULATE FOR THE SAME THREE REFERENCE SCENES
# ============================================================

edge_records = []

for (
    scene_id_i,
    category,
    raw_db,
    filtered_db
) in visual_records:

    record = {
        "scene_id": scene_id_i,
        "category": category
    }

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        metrics = edge_preservation_metrics(
            raw_db[band_idx],
            filtered_db[band_idx]
        )

        record[
            f"{band_name.lower()}_egr"
        ] = metrics["egr"]

        record[
            f"{band_name.lower()}_egc"
        ] = metrics["egc"]

        record[
            f"{band_name.lower()}_edge_pixels"
        ] = metrics["edge_pixels"]


    edge_records.append(
        record
    )


refined_lee_edge_results = pd.DataFrame(
    edge_records
)


# ============================================================
# REPORT
# ============================================================

print("=" * 78)
print("P3-C09 — REFINED LEE EDGE PRESERVATION")
print("=" * 78)

print(
    refined_lee_edge_results.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nSUMMARY")
print("-" * 78)

print(
    "Mean VV EGR:",
    refined_lee_edge_results[
        "vv_egr"
    ].mean()
)

print(
    "Mean VV EGC:",
    refined_lee_edge_results[
        "vv_egc"
    ].mean()
)

print(
    "Mean VH EGR:",
    refined_lee_edge_results[
        "vh_egr"
    ].mean()
)

print(
    "Mean VH EGC:",
    refined_lee_edge_results[
        "vh_egc"
    ].mean()
)

print(
    "\nRefined Lee edge-preservation audit complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C10
# SECTION: P3-S5 — LEE SIGMA DESPECKLING
#
# PURPOSE:
#   Compare Lee Sigma against Raw and Refined Lee using:
#       - same scenes
#       - same homogeneous patches
#       - same ENL metric
#       - same edge-preservation metric
#
# IMPORTANT:
#   - Processing is performed in LINEAR intensity.
#   - No labels are used.
#   - No normalization yet.
# ============================================================

import numpy as np
import pandas as pd
import rasterio

from scipy.ndimage import uniform_filter


# ------------------------------------------------------------
# Lee Sigma filter
# ------------------------------------------------------------

def lee_sigma_filter(
    image_linear,
    window_size=7,
    sigma_factor=0.9,
    eps=1e-12
):
    """
    Practical Lee-Sigma implementation.

    Steps:
      1. estimate local mean/std
      2. retain neighbors lying inside a sigma interval
      3. calculate statistics from selected pixels
      4. apply adaptive Lee/MMSE weighting

    Input must be positive linear SAR intensity.
    """

    img = np.asarray(
        image_linear,
        dtype=np.float64
    )

    if not np.isfinite(img).all():
        raise ValueError(
            "Lee Sigma test expects a fully valid image."
        )

    radius = window_size // 2

    H, W = img.shape

    padded = np.pad(
        img,
        radius,
        mode="reflect"
    )

    output = np.empty_like(img)

    # --------------------------------------------------------
    # Initial local statistics
    # --------------------------------------------------------

    local_mean = uniform_filter(
        img,
        size=window_size,
        mode="reflect"
    )

    local_sq_mean = uniform_filter(
        img ** 2,
        size=window_size,
        mode="reflect"
    )

    local_var = np.maximum(
        local_sq_mean
        -
        local_mean ** 2,
        0.0
    )

    local_std = np.sqrt(
        local_var
    )

    # --------------------------------------------------------
    # Pixel-by-pixel sigma-range selection
    # --------------------------------------------------------

    for y in range(H):

        for x in range(W):

            window = padded[
                y:y + window_size,
                x:x + window_size
            ]

            mu = local_mean[y, x]
            std = local_std[y, x]

            lower = max(
                mu - sigma_factor * std,
                eps
            )

            upper = (
                mu
                +
                sigma_factor * std
            )

            selected = window[
                (window >= lower)
                &
                (window <= upper)
            ]

            # ------------------------------------------------
            # Fallback if too few pixels survive
            # ------------------------------------------------

            if selected.size < 5:
                selected = window.reshape(-1)

            selected_mean = selected.mean()

            selected_var = selected.var()

            # ------------------------------------------------
            # Adaptive Lee/MMSE weight
            # ------------------------------------------------

            noise_var = max(
                selected_var,
                eps
            )

            total_var = max(
                local_var[y, x],
                eps
            )

            weight = (
                total_var - noise_var
            ) / total_var

            weight = np.clip(
                weight,
                0.0,
                1.0
            )

            output[y, x] = (
                selected_mean
                +
                weight
                *
                (
                    img[y, x]
                    -
                    selected_mean
                )
            )

    return np.maximum(
        output,
        eps
    ).astype(np.float32)


# ============================================================
# APPLY TO SAME REFERENCE SCENES
# ============================================================

lee_sigma_records = []

lee_sigma_visual_records = []


for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    path = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(path) as src:

        raw_db = src.read().astype(
            np.float32
        )

    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    assert valid.all()

    filtered_db = np.empty_like(
        raw_db
    )


    for band_idx in range(2):

        linear = db_to_linear(
            raw_db[band_idx],
            valid
        )

        filtered_linear = lee_sigma_filter(
            linear,
            window_size=7,
            sigma_factor=0.9
        )

        filtered_db[band_idx] = linear_to_db(
            filtered_linear,
            valid
        )


    # --------------------------------------------------------
    # Frozen homogeneous patch
    # --------------------------------------------------------

    x0 = int(patch_row["x"])
    y0 = int(patch_row["y"])
    ps = int(patch_row["patch_size"])

    record = {
        "scene_id": scene_id_i,
        "category": patch_row["category"]
    }


    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        raw_patch_db = raw_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]

        sigma_patch_db = filtered_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]


        raw_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )

        sigma_linear = np.power(
            10.0,
            sigma_patch_db / 10.0
        )


        raw_enl = patch_statistics(
            raw_linear
        )[3]

        sigma_enl = patch_statistics(
            sigma_linear
        )[3]


        # ----------------------------------------------------
        # Edge preservation
        # ----------------------------------------------------

        edge_metrics = edge_preservation_metrics(
            raw_db[band_idx],
            filtered_db[band_idx]
        )


        prefix = band_name.lower()

        record[
            f"{prefix}_raw_enl"
        ] = raw_enl

        record[
            f"{prefix}_lee_sigma_enl"
        ] = sigma_enl

        record[
            f"{prefix}_enl_gain"
        ] = sigma_enl / raw_enl

        record[
            f"{prefix}_egr"
        ] = edge_metrics["egr"]

        record[
            f"{prefix}_egc"
        ] = edge_metrics["egc"]


    lee_sigma_records.append(
        record
    )

    lee_sigma_visual_records.append(
        (
            scene_id_i,
            patch_row["category"],
            raw_db,
            filtered_db
        )
    )


lee_sigma_results = pd.DataFrame(
    lee_sigma_records
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 82)
print("P3-C10 — LEE SIGMA RESULTS")
print("=" * 82)

print(
    lee_sigma_results.to_string(
        index=False
    )
)


print("\nSUMMARY")
print("-" * 82)

print(
    "Mean VV ENL gain:",
    lee_sigma_results[
        "vv_enl_gain"
    ].mean()
)

print(
    "Mean VH ENL gain:",
    lee_sigma_results[
        "vh_enl_gain"
    ].mean()
)

print(
    "Mean VV EGR:",
    lee_sigma_results[
        "vv_egr"
    ].mean()
)

print(
    "Mean VH EGR:",
    lee_sigma_results[
        "vh_egr"
    ].mean()
)

print(
    "Mean VV EGC:",
    lee_sigma_results[
        "vv_egc"
    ].mean()
)

print(
    "Mean VH EGC:",
    lee_sigma_results[
        "vh_egc"
    ].mean()
)

print("\nLee Sigma comparison complete ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C11
# SECTION: P3-S5 — FREEZE CLASSICAL DESPECKLING RESULTS
#
# PURPOSE:
#   Save the current Raw / Refined Lee / Lee Sigma comparison
#   before moving to deep-learning despeckling.
# ============================================================

from pathlib import Path
import pandas as pd

RESULT_ROOT = (
    OUTPUT_ROOT /
    "sar_despeckling_results"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save individual detailed results
# ------------------------------------------------------------

refined_lee_results.to_csv(
    RESULT_ROOT /
    "refined_lee_enl_results.csv",
    index=False
)

refined_lee_edge_results.to_csv(
    RESULT_ROOT /
    "refined_lee_edge_results.csv",
    index=False
)

lee_sigma_results.to_csv(
    RESULT_ROOT /
    "lee_sigma_results.csv",
    index=False
)


# ------------------------------------------------------------
# Build compact summary
# ------------------------------------------------------------

refined_summary = {
    "method": "Refined Lee",

    "vv_enl_gain":
        refined_lee_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        refined_lee_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        refined_lee_edge_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        refined_lee_edge_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        refined_lee_edge_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        refined_lee_edge_results[
            "vh_egc"
        ].mean(),
}


sigma_summary = {
    "method": "Lee Sigma",

    "vv_enl_gain":
        lee_sigma_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        lee_sigma_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        lee_sigma_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        lee_sigma_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        lee_sigma_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        lee_sigma_results[
            "vh_egc"
        ].mean(),
}


classical_summary = pd.DataFrame(
    [
        refined_summary,
        sigma_summary
    ]
)

classical_summary.to_csv(
    RESULT_ROOT /
    "classical_despeckling_summary.csv",
    index=False
)


print("=" * 78)
print("P3-C11 — CLASSICAL SAR RESULTS FROZEN")
print("=" * 78)

print(
    classical_summary.to_string(
        index=False
    )
)

print("\nSaved to:")
print(RESULT_ROOT)

print(
    "\nCurrent classical leader "
    "(structure/noise trade-off): Lee Sigma ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C12
# SECTION: P3-S6 — SAR2SAR SETUP + PYTORCH COMPATIBILITY
#
# PURPOSE:
#   Install and verify the maintained SAR2SAR implementation.
#   Apply a narrow compatibility wrapper only while the bundled
#   trusted checkpoint is instantiated.
#
# IMPORTANT:
#   - The compatibility wrapper is active only during model load.
#   - torch.load is restored immediately afterward.
#   - No image is denoised in this cell.
# ============================================================

import sys
import subprocess
import torch

print("=" * 78)
print("P3-C12 — SAR2SAR SETUP + COMPATIBILITY")
print("=" * 78)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)


# ------------------------------------------------------------
# Install package if needed
# ------------------------------------------------------------

try:
    import deepdespeckling
    print("\ndeepdespeckling already installed ✅")

except ImportError:

    print("\nInstalling deepdespeckling...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "deepdespeckling"
        ]
    )

    import deepdespeckling

    print("deepdespeckling installed ✅")


# ------------------------------------------------------------
# Import SAR2SAR implementation
# ------------------------------------------------------------

from deepdespeckling.sar2sar.sar2sar_denoiser import (
    Sar2SarDenoiser
)

from deepdespeckling.utils.constants import (
    PATCH_SIZE,
    STRIDE_SIZE
)

print("\nSAR2SAR imports successful ✅")
print("PATCH_SIZE :", PATCH_SIZE)
print("STRIDE_SIZE:", STRIDE_SIZE)


# ------------------------------------------------------------
# PyTorch 2.6+ compatibility for the trusted bundled checkpoint
#
# Older deepdespeckling checkpoints assume:
#     torch.load(..., weights_only=False)
#
# The wrapper is deliberately temporary so it cannot affect
# later FloodShift code.
# ------------------------------------------------------------

_original_torch_load = torch.load

def _sar2sar_compat_load(*args, **kwargs):

    kwargs.setdefault(
        "weights_only",
        False
    )

    return _original_torch_load(
        *args,
        **kwargs
    )


torch.load = _sar2sar_compat_load

try:

    denoiser = Sar2SarDenoiser()

finally:

    torch.load = _original_torch_load


print("\nSAR2SAR denoiser instantiated ✅")
print("torch.load restored ✅")

print("\nP3-C12 COMPLETE")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C13
# SECTION: P3-S7 — SAR2SAR FIRST EVALUATION
#
# PURPOSE:
#   Apply pretrained SAR2SAR to the SAME three reference scenes
#   used for Refined Lee and Lee Sigma.
#
#   Evaluate:
#       - ENL gain
#       - Edge Gradient Retention (EGR)
#       - Edge Gradient Correlation (EGC)
#       - Runtime
#
# IMPORTANT:
#   SAR2SAR expects AMPLITUDE, not dB.
#
#   dB
#    -> linear intensity
#    -> sqrt = amplitude
#    -> SAR2SAR
#    -> denoised amplitude
#    -> square = intensity
#    -> dB
#
# PYTORCH 2.6+ NOTE:
#   deepdespeckling loads its bundled legacy SAR2SAR checkpoint
#   lazily inside denoise_image().
#
#   PyTorch >= 2.6 defaults torch.load(weights_only=True),
#   which is incompatible with this trusted bundled checkpoint.
#
#   Therefore torch.load is temporarily wrapped ONLY during
#   SAR2SAR inference and restored immediately afterward.
# ============================================================

import time
import numpy as np
import pandas as pd
import rasterio
import torch


print("=" * 82)
print("P3-C13 — SAR2SAR FIRST EVALUATION")
print("=" * 82)

# ------------------------------------------------------------
# Check execution device
# ------------------------------------------------------------

print("SAR2SAR device:", denoiser.device)


# ============================================================
# PYTORCH 2.6+ SAR2SAR COMPATIBILITY HELPER
# ============================================================

def sar2sar_denoise_with_compat(
    denoiser,
    amplitude,
    patch_size,
    stride_size
):
    """
    Run SAR2SAR while temporarily restoring the legacy
    torch.load(weights_only=False) behaviour required by the
    trusted checkpoint bundled with deepdespeckling.

    torch.load is ALWAYS restored afterward.
    """

    original_torch_load = torch.load


    def compatible_torch_load(
        *args,
        **kwargs
    ):

        kwargs["weights_only"] = False

        return original_torch_load(
            *args,
            **kwargs
        )


    torch.load = compatible_torch_load


    try:

        output = denoiser.denoise_image(
            amplitude,
            patch_size=patch_size,
            stride_size=stride_size
        )


    finally:

        # ----------------------------------------------------
        # CRITICAL:
        # Never leave global torch.load patched.
        # ----------------------------------------------------

        torch.load = original_torch_load


    return output


print(
    "PyTorch 2.6+ SAR2SAR compatibility helper ready ✅"
)


# ============================================================
# STORAGE
# ============================================================

sar2sar_records = []

sar2sar_visual_records = []


# ============================================================
# PROCESS SAME FROZEN REFERENCE SCENES
# ============================================================

for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row[
        "scene_id"
    ]


    manifest_row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]


    sar_path_i = locate_input_file(
        manifest_row[
            "sar_raw_path"
        ]
    )


    with rasterio.open(
        sar_path_i
    ) as src:

        raw_db = src.read().astype(
            np.float32
        )


    # --------------------------------------------------------
    # These reference scenes were intentionally selected from
    # scenes with complete SAR coverage.
    # --------------------------------------------------------

    valid = (
        np.isfinite(
            raw_db[0]
        )
        &
        np.isfinite(
            raw_db[1]
        )
    )


    assert valid.all(), (
        f"{scene_id_i} unexpectedly contains invalid SAR."
    )


    filtered_db = np.empty_like(
        raw_db,
        dtype=np.float32
    )


    runtimes = []


    # ========================================================
    # PROCESS VV AND VH SEPARATELY
    # ========================================================

    for band_idx, band_name in enumerate(
        [
            "VV",
            "VH"
        ]
    ):

        print(
            f"\nProcessing "
            f"{scene_id_i} — {band_name}"
        )


        # ----------------------------------------------------
        # STEP 1
        # dB -> linear intensity
        # ----------------------------------------------------

        intensity = db_to_linear(
            raw_db[
                band_idx
            ],
            valid
        )


        # ----------------------------------------------------
        # STEP 2
        # intensity -> amplitude
        #
        # intensity = amplitude^2
        # ----------------------------------------------------

        amplitude = np.sqrt(
            np.maximum(
                intensity,
                1e-12
            )
        ).astype(
            np.float32
        )


        # ----------------------------------------------------
        # STEP 3
        # SAR2SAR inference
        #
        # Compatibility wrapper is active ONLY while
        # denoise_image() loads/runs the bundled checkpoint.
        # ----------------------------------------------------

        start = time.perf_counter()


        output = sar2sar_denoise_with_compat(

            denoiser=
                denoiser,

            amplitude=
                amplitude,

            patch_size=
                PATCH_SIZE,

            stride_size=
                STRIDE_SIZE
        )


        elapsed = (
            time.perf_counter()
            -
            start
        )


        runtimes.append(
            elapsed
        )


        # ----------------------------------------------------
        # Current deepdespeckling implementation returns:
        #
        # {
        #     "noisy": ...,
        #     "denoised": ...
        # }
        # ----------------------------------------------------

        if not isinstance(
            output,
            dict
        ):

            raise TypeError(
                "Unexpected SAR2SAR output type: "
                f"{type(output)}"
            )


        if "denoised" not in output:

            raise KeyError(
                "SAR2SAR output does not contain "
                "'denoised'. Keys: "
                f"{output.keys()}"
            )


        denoised_amplitude = np.asarray(
            output[
                "denoised"
            ],
            dtype=np.float32
        )


        # ----------------------------------------------------
        # Validate output
        # ----------------------------------------------------

        assert (
            denoised_amplitude.shape
            ==
            amplitude.shape
        ), (
            f"Shape mismatch: "
            f"{denoised_amplitude.shape} vs "
            f"{amplitude.shape}"
        )


        if not np.isfinite(
            denoised_amplitude
        ).all():

            raise ValueError(
                f"SAR2SAR produced non-finite values "
                f"for {scene_id_i} {band_name}"
            )


        # ----------------------------------------------------
        # SAR amplitude must be non-negative
        # ----------------------------------------------------

        denoised_amplitude = np.maximum(
            denoised_amplitude,
            1e-12
        )


        # ----------------------------------------------------
        # STEP 4
        # amplitude -> intensity
        # ----------------------------------------------------

        denoised_intensity = (
            denoised_amplitude
            **
            2
        )


        # ----------------------------------------------------
        # STEP 5
        # intensity -> dB
        # ----------------------------------------------------

        filtered_db[
            band_idx
        ] = linear_to_db(
            denoised_intensity,
            valid
        )


        print(
            f"  Runtime: "
            f"{elapsed:.3f} sec"
        )


        print(
            "  Output amplitude range:",
            float(
                denoised_amplitude.min()
            ),
            "to",
            float(
                denoised_amplitude.max()
            )
        )


    # ========================================================
    # EVALUATE SAME FROZEN HOMOGENEOUS PATCH
    # ========================================================

    x0 = int(
        patch_row[
            "x"
        ]
    )

    y0 = int(
        patch_row[
            "y"
        ]
    )

    ps = int(
        patch_row[
            "patch_size"
        ]
    )


    record = {

        "scene_id":
            scene_id_i,

        "category":
            patch_row[
                "category"
            ],

        "runtime_sec_total":
            sum(
                runtimes
            )
    }


    for band_idx, band_name in enumerate(
        [
            "VV",
            "VH"
        ]
    ):

        prefix = (
            band_name.lower()
        )


        # ----------------------------------------------------
        # Frozen patch
        # ----------------------------------------------------

        raw_patch_db = raw_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]


        sar2sar_patch_db = filtered_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]


        # ----------------------------------------------------
        # ENL in linear-intensity domain
        # ----------------------------------------------------

        raw_patch_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )


        sar2sar_patch_linear = np.power(
            10.0,
            sar2sar_patch_db / 10.0
        )


        raw_enl = patch_statistics(
            raw_patch_linear
        )[3]


        sar2sar_enl = patch_statistics(
            sar2sar_patch_linear
        )[3]


        # ----------------------------------------------------
        # Same edge-preservation metric used for all methods
        # ----------------------------------------------------

        edge_metrics = (
            edge_preservation_metrics(

                raw_db[
                    band_idx
                ],

                filtered_db[
                    band_idx
                ]
            )
        )


        record[
            f"{prefix}_raw_enl"
        ] = raw_enl


        record[
            f"{prefix}_sar2sar_enl"
        ] = sar2sar_enl


        record[
            f"{prefix}_enl_gain"
        ] = (
            sar2sar_enl
            /
            raw_enl
        )


        record[
            f"{prefix}_egr"
        ] = edge_metrics[
            "egr"
        ]


        record[
            f"{prefix}_egc"
        ] = edge_metrics[
            "egc"
        ]


    sar2sar_records.append(
        record
    )


    sar2sar_visual_records.append(
        (
            scene_id_i,
            patch_row[
                "category"
            ],
            raw_db,
            filtered_db
        )
    )


# ============================================================
# RESULTS TABLE
# ============================================================

sar2sar_results = pd.DataFrame(
    sar2sar_records
)


print(
    "\n"
    +
    "=" * 82
)

print(
    "SAR2SAR RESULTS"
)

print(
    "=" * 82
)


print(
    sar2sar_results.to_string(
        index=False
    )
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\nSUMMARY"
)

print(
    "-" * 82
)


print(
    "Mean VV ENL gain:",
    sar2sar_results[
        "vv_enl_gain"
    ].mean()
)


print(
    "Mean VH ENL gain:",
    sar2sar_results[
        "vh_enl_gain"
    ].mean()
)


print(
    "Mean VV EGR:",
    sar2sar_results[
        "vv_egr"
    ].mean()
)


print(
    "Mean VH EGR:",
    sar2sar_results[
        "vh_egr"
    ].mean()
)


print(
    "Mean VV EGC:",
    sar2sar_results[
        "vv_egc"
    ].mean()
)


print(
    "Mean VH EGC:",
    sar2sar_results[
        "vh_egc"
    ].mean()
)


print(
    "Mean runtime per 2-band scene:",
    sar2sar_results[
        "runtime_sec_total"
    ].mean(),
    "sec"
)


# ============================================================
# FINAL SAFETY CHECK
# ============================================================

assert len(
    sar2sar_results
) == len(
    homogeneous_patches
)


assert np.isfinite(
    sar2sar_results.select_dtypes(
        include=[
            np.number
        ]
    ).to_numpy()
).all()


print(
    "\ntorch.load currently restored:",
    torch.load.__name__
    !=
    "compatible_torch_load"
)


print(
    "\nSAR2SAR first evaluation complete ✅"
)

In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C14
# SECTION: P3-S6 — HOMOMORPHIC BM3D SETUP
#
# PURPOSE:
#   Install and verify the modern reference BM3D package.
#
# IMPORTANT:
#   This is NOT labelled "SAR-BM3D".
#   We will use BM3D in a homomorphic/log-domain SAR pipeline.
# ============================================================

import sys
import subprocess

print("=" * 72)
print("P3-C14 — HOMOMORPHIC BM3D SETUP")
print("=" * 72)

try:
    import bm3d

    print("bm3d already installed ✅")

except ImportError:

    print("Installing bm3d...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "bm3d"
        ]
    )

    import bm3d

    print("bm3d installed ✅")


print("\nBM3D module:", bm3d)

print(
    "\nHomomorphic BM3D environment ready ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C15
# SECTION: P3-S6 — TRAIN-ONLY BM3D NOISE ESTIMATION
#
# PURPOSE:
#   Estimate a robust log-domain noise scale from the frozen
#   homogeneous TRAIN patches.
#
# NO filtering is performed yet.
# ============================================================

import numpy as np
import pandas as pd
import rasterio

noise_records = []

for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        raw_db = src.read().astype(np.float32)

    x0 = int(patch_row["x"])
    y0 = int(patch_row["y"])
    ps = int(patch_row["patch_size"])

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        patch_db = raw_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]

        # --------------------------------------------
        # dB -> linear intensity
        # --------------------------------------------

        patch_linear = np.power(
            10.0,
            patch_db / 10.0
        )

        # --------------------------------------------
        # log-intensity domain
        # --------------------------------------------

        log_patch = np.log(
            np.maximum(
                patch_linear,
                1e-12
            )
        )

        # Robust scale estimate using MAD
        median = np.median(log_patch)

        mad = np.median(
            np.abs(
                log_patch - median
            )
        )

        sigma_mad = (
            1.4826 * mad
        )

        noise_records.append({
            "scene_id": scene_id_i,
            "category": patch_row["category"],
            "band": band_name,
            "log_sigma_mad": float(sigma_mad),
            "log_std": float(log_patch.std())
        })


bm3d_noise_stats = pd.DataFrame(
    noise_records
)


print("=" * 76)
print("P3-C15 — LOG-SPECKLE SCALE ESTIMATION")
print("=" * 76)

print(
    bm3d_noise_stats.to_string(
        index=False
    )
)


print("\nSUMMARY")
print("-" * 76)

print(
    bm3d_noise_stats.groupby("band")[
        ["log_sigma_mad", "log_std"]
    ].median()
)


BM3D_SIGMA_VV = (
    bm3d_noise_stats[
        bm3d_noise_stats["band"] == "VV"
    ]["log_sigma_mad"].median()
)

BM3D_SIGMA_VH = (
    bm3d_noise_stats[
        bm3d_noise_stats["band"] == "VH"
    ]["log_sigma_mad"].median()
)


print("\nSelected TRAIN-only robust estimates:")

print(
    f"VV sigma = {BM3D_SIGMA_VV:.6f}"
)

print(
    f"VH sigma = {BM3D_SIGMA_VH:.6f}"
)

print(
    "\nNoise estimation complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C16
# SECTION: P3-S6 — HOMOMORPHIC BM3D
#
# PURPOSE:
#   Apply BM3D in log-intensity space to the SAME three
#   reference scenes.
#
# PIPELINE:
#   dB
#    -> linear intensity
#    -> natural log
#    -> BM3D
#    -> exp
#    -> radiometric mean correction
#    -> dB
#
# EVALUATION:
#   ENL
#   EGR
#   EGC
#   runtime
#
# IMPORTANT:
#   This is called "Homomorphic BM3D", NOT "SAR-BM3D".
# ============================================================

import time
import numpy as np
import pandas as pd
import rasterio
import bm3d


print("=" * 82)
print("P3-C16 — HOMOMORPHIC BM3D EVALUATION")
print("=" * 82)

BM3D_SIGMAS = {
    "VV": float(BM3D_SIGMA_VV),
    "VH": float(BM3D_SIGMA_VH),
}

print("\nScreening log-domain sigma values:")
print(BM3D_SIGMAS)


bm3d_records = []
bm3d_visual_records = []


# ============================================================
# PROCESS SAME THREE REFERENCE SCENES
# ============================================================

for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        raw_db = src.read().astype(np.float32)


    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    # These three reference scenes were chosen from
    # complete-coverage scenes.
    assert valid.all()


    filtered_db = np.empty_like(
        raw_db,
        dtype=np.float32
    )

    runtimes = []


    # ========================================================
    # VV / VH separately
    # ========================================================

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        print(
            f"\nProcessing {scene_id_i} — {band_name}"
        )

        sigma = BM3D_SIGMAS[
            band_name
        ]


        # ----------------------------------------------------
        # STEP 1: dB -> linear intensity
        # ----------------------------------------------------

        intensity = db_to_linear(
            raw_db[band_idx],
            valid
        ).astype(np.float64)


        # ----------------------------------------------------
        # STEP 2: intensity -> log intensity
        # ----------------------------------------------------

        log_intensity = np.log(
            np.maximum(
                intensity,
                1e-12
            )
        )


        # ----------------------------------------------------
        # STEP 3: BM3D in log domain
        # ----------------------------------------------------

        start = time.perf_counter()

        denoised_log = bm3d.bm3d(
            log_intensity,
            sigma_psd=sigma,
            stage_arg=bm3d.BM3DStages.ALL_STAGES
        )

        elapsed = (
            time.perf_counter()
            -
            start
        )

        runtimes.append(
            elapsed
        )


        denoised_log = np.asarray(
            denoised_log,
            dtype=np.float64
        )


        # ----------------------------------------------------
        # STEP 4: log -> linear intensity
        # ----------------------------------------------------

        denoised_intensity = np.exp(
            denoised_log
        )


        # ----------------------------------------------------
        # STEP 5: preserve scene-level mean backscatter
        #
        # Homomorphic processing can introduce a multiplicative
        # radiometric bias after exponentiation.
        # ----------------------------------------------------

        raw_mean = float(
            intensity.mean()
        )

        den_mean = float(
            denoised_intensity.mean()
        )

        radiometric_scale = (
            raw_mean
            /
            max(den_mean, 1e-12)
        )

        denoised_intensity *= (
            radiometric_scale
        )


        # ----------------------------------------------------
        # STEP 6: linear -> dB
        # ----------------------------------------------------

        filtered_db[
            band_idx
        ] = linear_to_db(
            denoised_intensity,
            valid
        )


        print(
            f"  sigma             : {sigma:.6f}"
        )

        print(
            f"  runtime           : {elapsed:.3f} sec"
        )

        print(
            f"  radiometric scale : {radiometric_scale:.6f}"
        )


    # ========================================================
    # SAME FROZEN PATCH EVALUATION
    # ========================================================

    x0 = int(
        patch_row["x"]
    )

    y0 = int(
        patch_row["y"]
    )

    ps = int(
        patch_row["patch_size"]
    )


    record = {
        "scene_id": scene_id_i,
        "category": patch_row["category"],
        "runtime_sec_total": sum(runtimes)
    }


    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        prefix = band_name.lower()


        raw_patch_db = raw_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]

        bm3d_patch_db = filtered_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]


        # ----------------------------------------------------
        # ENL in linear intensity
        # ----------------------------------------------------

        raw_patch_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )

        bm3d_patch_linear = np.power(
            10.0,
            bm3d_patch_db / 10.0
        )


        raw_enl = patch_statistics(
            raw_patch_linear
        )[3]

        bm3d_enl = patch_statistics(
            bm3d_patch_linear
        )[3]


        # ----------------------------------------------------
        # Same edge metrics
        # ----------------------------------------------------

        edge_metrics = edge_preservation_metrics(
            raw_db[band_idx],
            filtered_db[band_idx]
        )


        record[
            f"{prefix}_raw_enl"
        ] = raw_enl

        record[
            f"{prefix}_bm3d_enl"
        ] = bm3d_enl

        record[
            f"{prefix}_enl_gain"
        ] = (
            bm3d_enl / raw_enl
        )

        record[
            f"{prefix}_egr"
        ] = edge_metrics["egr"]

        record[
            f"{prefix}_egc"
        ] = edge_metrics["egc"]


    bm3d_records.append(
        record
    )

    bm3d_visual_records.append(
        (
            scene_id_i,
            patch_row["category"],
            raw_db,
            filtered_db
        )
    )


# ============================================================
# RESULTS
# ============================================================

bm3d_results = pd.DataFrame(
    bm3d_records
)


print("\n" + "=" * 82)
print("HOMOMORPHIC BM3D RESULTS")
print("=" * 82)

print(
    bm3d_results.to_string(
        index=False
    )
)


print("\nSUMMARY")
print("-" * 82)

print(
    "Mean VV ENL gain:",
    bm3d_results[
        "vv_enl_gain"
    ].mean()
)

print(
    "Mean VH ENL gain:",
    bm3d_results[
        "vh_enl_gain"
    ].mean()
)

print(
    "Mean VV EGR:",
    bm3d_results[
        "vv_egr"
    ].mean()
)

print(
    "Mean VH EGR:",
    bm3d_results[
        "vh_egr"
    ].mean()
)

print(
    "Mean VV EGC:",
    bm3d_results[
        "vv_egc"
    ].mean()
)

print(
    "Mean VH EGC:",
    bm3d_results[
        "vh_egc"
    ].mean()
)

print(
    "Mean runtime per 2-band scene:",
    bm3d_results[
        "runtime_sec_total"
    ].mean(),
    "sec"
)

print(
    "\nHomomorphic BM3D evaluation complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C17
# SECTION: P3-S6 — FREEZE EXISTING SAR DESPECKLING SCREENING
#
# PURPOSE:
#   Build one canonical comparison table containing:
#       Refined Lee
#       Lee Sigma
#       SAR2SAR
#       Homomorphic BM3D
#
# IMPORTANT:
#   This screening does NOT select the final preprocessing
#   winner. Downstream segmentation will make that decision.
# ============================================================

import pandas as pd

print("=" * 90)
print("P3-C17 — EXISTING SAR DESPECKLING METHODS")
print("=" * 90)


# ------------------------------------------------------------
# SAR2SAR summary
# ------------------------------------------------------------

sar2sar_summary = {

    "method":
        "SAR2SAR",

    "vv_enl_gain":
        sar2sar_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        sar2sar_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        sar2sar_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        sar2sar_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        sar2sar_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        sar2sar_results[
            "vh_egc"
        ].mean(),

    "runtime_sec_per_scene":
        sar2sar_results[
            "runtime_sec_total"
        ].mean()
}


# ------------------------------------------------------------
# Homomorphic BM3D summary
# ------------------------------------------------------------

bm3d_summary = {

    "method":
        "Homomorphic BM3D",

    "vv_enl_gain":
        bm3d_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        bm3d_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        bm3d_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        bm3d_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        bm3d_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        bm3d_results[
            "vh_egc"
        ].mean(),

    "runtime_sec_per_scene":
        bm3d_results[
            "runtime_sec_total"
        ].mean()
}


# ------------------------------------------------------------
# Classical table + deep/non-local comparators
# ------------------------------------------------------------

classical_extended = classical_summary.copy()

classical_extended[
    "runtime_sec_per_scene"
] = pd.NA


existing_methods_summary = pd.concat(
    [
        classical_extended,
        pd.DataFrame(
            [
                sar2sar_summary,
                bm3d_summary
            ]
        )
    ],
    ignore_index=True
)


print(
    existing_methods_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Save canonical screening outputs
# ------------------------------------------------------------

existing_methods_path = (
    RESULT_ROOT /
    "sar_existing_methods_comparison_v2.csv"
)

existing_methods_summary.to_csv(
    existing_methods_path,
    index=False
)

sar2sar_results.to_csv(
    RESULT_ROOT /
    "sar2sar_detailed_results.csv",
    index=False
)

bm3d_results.to_csv(
    RESULT_ROOT /
    "homomorphic_bm3d_detailed_results.csv",
    index=False
)


print("\nSaved:")
print(existing_methods_path)

print(
    "\nExisting-method screening frozen ✅"
)

print(
    "No final despeckling winner selected yet."
)


## 3. Proposed DP-SSD self-supervised preprocessing


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C18
# SECTION: P3-S8 — SELF-SUPERVISED SAMPLE CONSTRUCTION
#
# PURPOSE:
#   Construct ONE valid dual-polarization self-supervised
#   training sample before building any neural network.
#
# PIPELINE:
#   SAR dB
#      ↓
#   linear intensity
#      ↓
#   natural log
#      ↓
#   2×2 interleaved sub-images
#      ↓
#   shared VV/VH masking
#      ↓
#   masked input + original masked targets
#
# IMPORTANT:
#   - TRAIN data only
#   - NO segmentation labels
#   - NO clean SAR targets
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MASK_RATIO = 0.10
RANDOM_SEED = 42

rng = np.random.default_rng(
    RANDOM_SEED
)


# ------------------------------------------------------------
# Select a fully valid TRAIN scene automatically
# ------------------------------------------------------------

sample_scene_id = (
    homogeneous_patches
    .iloc[0]["scene_id"]
)

manifest_row = train_df[
    train_df["scene_id"] == sample_scene_id
].iloc[0]

sar_path_i = locate_input_file(
    manifest_row["sar_raw_path"]
)


with rasterio.open(sar_path_i) as src:

    raw_db = src.read().astype(
        np.float32
    )


print("=" * 78)
print("P3-C18 — SELF-SUPERVISED SAMPLE CONSTRUCTION")
print("=" * 78)

print("Scene:", sample_scene_id)
print("Raw shape:", raw_db.shape)


# ------------------------------------------------------------
# Validity
# ------------------------------------------------------------

valid = (
    np.isfinite(raw_db[0])
    &
    np.isfinite(raw_db[1])
)

assert valid.all(), (
    "Reference scene should have complete SAR coverage."
)


# ============================================================
# STEP 1 — dB → linear intensity
# ============================================================

linear = np.empty_like(
    raw_db,
    dtype=np.float32
)

for band_idx in range(2):

    linear[band_idx] = db_to_linear(
        raw_db[band_idx],
        valid
    )


# ============================================================
# STEP 2 — linear intensity → natural log intensity
# ============================================================

log_sar = np.log(
    np.maximum(
        linear,
        1e-12
    )
).astype(np.float32)


print(
    "\nLog-SAR shape:",
    log_sar.shape
)


# ============================================================
# STEP 3 — construct four interleaved sub-images
#
# Original:
#
# A B A B
# C D C D
# A B A B
# C D C D
#
# gives:
#
# A = even/even
# B = even/odd
# C = odd/even
# D = odd/odd
#
# Each output is 256 × 256.
# ============================================================

def make_interleaved_views(x):

    views = {}

    offsets = {
        "EE": (0, 0),
        "EO": (0, 1),
        "OE": (1, 0),
        "OO": (1, 1),
    }

    for name, (row_offset, col_offset) in offsets.items():

        views[name] = x[
            :,
            row_offset::2,
            col_offset::2
        ]

    return views


views = make_interleaved_views(
    log_sar
)


print("\nInterleaved views:")

for name, view in views.items():

    print(
        f"  {name}: {view.shape}"
    )


# ------------------------------------------------------------
# Use EE only for this demonstration.
#
# During training ALL four views will be used.
# ------------------------------------------------------------

original = views["EE"].copy()

C, H, W = original.shape

assert C == 2


# ============================================================
# STEP 4 — create ONE shared masking map
#
# The SAME pixels are hidden in VV and VH.
# ============================================================

mask = (
    rng.random(
        (H, W)
    )
    <
    MASK_RATIO
)


print(
    "\nMasked spatial positions:",
    f"{mask.sum():,}"
)

print(
    "Actual mask percentage:",
    f"{100 * mask.mean():.2f}%"
)


# ============================================================
# STEP 5 — replace masked pixels with neighboring observations
#
# We must NOT leave the original target visible.
#
# Each masked position receives a value from one of:
#
#   ↑ ↓ ← →
#
# The replacement direction is random.
# ============================================================

def neighbour_replacement(
    image,
    mask,
    rng
):
    """
    image:
        [C, H, W]

    mask:
        [H, W]

    Returns:
        masked copy of image.
    """

    C, H, W = image.shape

    masked_image = image.copy()


    # Reflect padding prevents wrap-around leakage.
    padded = np.pad(
        image,
        (
            (0, 0),
            (1, 1),
            (1, 1)
        ),
        mode="reflect"
    )


    # Four neighboring candidates
    neighbours = np.stack(
        [
            # up
            padded[
                :,
                0:H,
                1:W+1
            ],

            # down
            padded[
                :,
                2:H+2,
                1:W+1
            ],

            # left
            padded[
                :,
                1:H+1,
                0:W
            ],

            # right
            padded[
                :,
                1:H+1,
                2:W+2
            ],

        ],
        axis=0
    )

    # One random neighbour direction for every spatial position
    direction_map = rng.integers(
        0,
        4,
        size=(H, W)
    )


    yy, xx = np.indices(
        (H, W)
    )


    for channel in range(C):

        replacements = neighbours[
            direction_map,
            channel,
            yy,
            xx
        ]

        masked_image[
            channel,
            mask
        ] = replacements[
            mask
        ]


    return masked_image


masked_input = neighbour_replacement(
    original,
    mask,
    rng
)


# ============================================================
# STEP 6 — target
#
# Target values are simply the ORIGINAL observations at the
# locations we intentionally hid.
#
# Loss will later be calculated ONLY on these positions.
# ============================================================

target = original.copy()


# ============================================================
# SANITY CHECKS
# ============================================================

assert masked_input.shape == original.shape

assert target.shape == original.shape

assert mask.shape == (
    original.shape[1],
    original.shape[2]
)

# Both polarizations use exactly the same spatial mask.
assert mask.dtype == bool


# Unmasked pixels must remain unchanged.
assert np.allclose(
    masked_input[:, ~mask],
    original[:, ~mask]
)


print("\nTensor structure:")

print(
    "Original     :",
    original.shape
)

print(
    "Masked input :",
    masked_input.shape
)

print(
    "Target       :",
    target.shape
)

print(
    "Mask         :",
    mask.shape
)


# ============================================================
# VISUAL CHECK
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)


band_names = [
    "VV",
    "VH"
]


for band_idx, band_name in enumerate(
    band_names
):

    original_band = original[
        band_idx
    ]

    masked_band = masked_input[
        band_idx
    ]


    lo, hi = np.percentile(
        original_band,
        [2, 98]
    )


    axes[
        band_idx,
        0
    ].imshow(
        original_band,
        cmap="gray",
        vmin=lo,
        vmax=hi
    )

    axes[
        band_idx,
        0
    ].set_title(
        f"{band_name} — original log-intensity"
    )


    axes[
        band_idx,
        1
    ].imshow(
        masked_band,
        cmap="gray",
        vmin=lo,
        vmax=hi
    )

    axes[
        band_idx,
        1
    ].set_title(
        f"{band_name} — masked input"
    )


    axes[
        band_idx,
        2
    ].imshow(
        mask,
        cmap="gray"
    )

    axes[
        band_idx,
        2
    ].set_title(
        "Shared masked locations"
    )


    for j in range(3):

        axes[
            band_idx,
            j
        ].axis("off")


plt.tight_layout()

plt.show()


print(
    "\nSelf-supervised sample construction verified ✅"
)

print(
    "No clean SAR target or segmentation label was used."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C19
# SECTION: P3-S8 — DP-SSD VALID-CROP AUDIT
#
# PURPOSE:
#   Determine which TRAIN scenes / interleaved views contain
#   at least one fully valid 128x128 crop.
#
# WHY:
#   DP-SSD should never learn from fabricated values inserted
#   into SAR no-data regions.
#
# IMPORTANT:
#   - TRAIN only
#   - SAR validity only
#   - no labels
#   - no validation/test
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio


# ------------------------------------------------------------
# Canonical DP-SSD artifact directory
# ------------------------------------------------------------

DPSSD_ROOT = (
    OUTPUT_ROOT /
    "dpssd"
)

DPSSD_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CROP_SIZE = 128

VIEW_OFFSETS = {
    "EE": (0, 0),
    "EO": (0, 1),
    "OE": (1, 0),
    "OO": (1, 1),
}


# ------------------------------------------------------------
# Integral-image helper
# ------------------------------------------------------------

def valid_crop_locations(valid_mask, crop_size):
    """
    Return top-left coordinates of all crop_size x crop_size
    windows that are 100% valid.

    valid_mask:
        [H, W] boolean
    """

    x = valid_mask.astype(np.int32)

    # Integral image with zero border
    integral = np.pad(
        x,
        ((1, 0), (1, 0)),
        mode="constant"
    ).cumsum(0).cumsum(1)

    window_sum = (
        integral[crop_size:, crop_size:]
        - integral[:-crop_size, crop_size:]
        - integral[crop_size:, :-crop_size]
        + integral[:-crop_size, :-crop_size]
    )

    required = crop_size * crop_size

    yy, xx = np.where(
        window_sum == required
    )

    return yy, xx


# ============================================================
# AUDIT ALL TRAIN SCENES
# ============================================================

crop_audit_records = []

for scene_num, (_, row) in enumerate(
    train_df.iterrows(),
    start=1
):

    sar_path_i = locate_input_file(
        row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:

        x = src.read().astype(
            np.float32
        )

    valid_full = (
        np.isfinite(x[0])
        &
        np.isfinite(x[1])
    )

    scene_has_any_valid_crop = False

    for view_name, (
        row_offset,
        col_offset
    ) in VIEW_OFFSETS.items():

        valid_view = valid_full[
            row_offset::2,
            col_offset::2
        ]

        yy, xx = valid_crop_locations(
            valid_view,
            CROP_SIZE
        )

        n_valid_crops = len(yy)

        if n_valid_crops > 0:
            scene_has_any_valid_crop = True

        crop_audit_records.append({
            "scene_id": row["scene_id"],
            "region": row["region"],
            "view": view_name,

            "view_height": valid_view.shape[0],
            "view_width": valid_view.shape[1],

            "valid_pixels_view": int(
                valid_view.sum()
            ),

            "valid_pct_view": float(
                100.0 * valid_view.mean()
            ),

            "valid_crop_count": int(
                n_valid_crops
            ),

            "eligible": bool(
                n_valid_crops > 0
            ),
        })


    if (
        scene_num % 50 == 0
        or scene_num == len(train_df)
    ):
        print(
            f"Processed "
            f"{scene_num:>3}/{len(train_df)} scenes"
        )


# ============================================================
# RESULTS
# ============================================================

dpssd_crop_audit = pd.DataFrame(
    crop_audit_records
)

eligible_pairs = dpssd_crop_audit[
    dpssd_crop_audit["eligible"]
].copy()


eligible_scene_ids = set(
    eligible_pairs["scene_id"]
)

ineligible_scene_ids = (
    set(train_df["scene_id"])
    -
    eligible_scene_ids
)


print("\n" + "=" * 78)
print("P3-C19 — DP-SSD VALID-CROP AUDIT")
print("=" * 78)

print(
    "TRAIN scenes                  :",
    len(train_df)
)

print(
    "Possible scene-view pairs     :",
    len(train_df) * 4
)

print(
    "Eligible scene-view pairs     :",
    len(eligible_pairs)
)

print(
    "Eligible TRAIN scenes         :",
    len(eligible_scene_ids)
)

print(
    "Scenes with no valid 128 crop :",
    len(ineligible_scene_ids)
)


print("\nEligibility by interleaved view:")
print(
    dpssd_crop_audit.groupby(
        "view"
    )["eligible"].agg(
        ["sum", "count"]
    )
)


print("\nValid crop counts for eligible pairs:")

print(
    eligible_pairs[
        "valid_crop_count"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


if ineligible_scene_ids:

    print("\nScenes with no fully valid crop:")

    for scene_id_i in sorted(
        ineligible_scene_ids
    ):

        print(
            " ",
            scene_id_i
        )


# ============================================================
# SAVE
# ============================================================

crop_audit_path = (
    DPSSD_ROOT /
    "dpssd_valid_crop_audit_v1.csv"
)

eligible_path = (
    DPSSD_ROOT /
    "dpssd_eligible_scene_views_v1.csv"
)


dpssd_crop_audit.to_csv(
    crop_audit_path,
    index=False
)

eligible_pairs.to_csv(
    eligible_path,
    index=False
)


print("\nSaved:")
print(crop_audit_path)
print(eligible_path)

print(
    "\nDP-SSD crop eligibility audit complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C20
# SECTION: P3-S8 — DP-SSD INTERNAL TRAIN/VAL SPLIT
#
# PURPOSE:
#   Split the 308 eligible TRAIN scenes into:
#
#       DP-SSD train
#       DP-SSD internal validation
#
# IMPORTANT:
#   - Both subsets come ONLY from canonical TRAIN.
#   - Split is scene-level, never view-level.
#   - All four views of a scene stay together.
#   - Main FloodShift validation/test remain untouched.
# ============================================================

import numpy as np
import pandas as pd

DPSSD_VAL_FRACTION = 0.10
DPSSD_SPLIT_SEED = 2026

rng_split = np.random.default_rng(
    DPSSD_SPLIT_SEED
)


# ------------------------------------------------------------
# One row per eligible scene
# ------------------------------------------------------------

eligible_scene_table = (
    eligible_pairs[
        ["scene_id", "region"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

assert (
    len(eligible_scene_table)
    ==
    308
)


# ------------------------------------------------------------
# Region-aware scene split
# ------------------------------------------------------------

train_scene_ids = []
val_scene_ids = []

for region, group in eligible_scene_table.groupby(
    "region"
):

    scene_ids = group[
        "scene_id"
    ].tolist()

    rng_split.shuffle(
        scene_ids
    )

    # At least 1 internal-validation scene per region
    # whenever the region contains enough eligible scenes.
    n_val = max(
        1,
        int(
            round(
                len(scene_ids)
                *
                DPSSD_VAL_FRACTION
            )
        )
    )

    # Never consume an entire region as validation.
    n_val = min(
        n_val,
        len(scene_ids) - 1
    )

    val_scene_ids.extend(
        scene_ids[:n_val]
    )

    train_scene_ids.extend(
        scene_ids[n_val:]
    )


train_scene_ids = set(
    train_scene_ids
)

val_scene_ids = set(
    val_scene_ids
)


# ------------------------------------------------------------
# Safety
# ------------------------------------------------------------

assert train_scene_ids.isdisjoint(
    val_scene_ids
)

assert (
    train_scene_ids
    |
    val_scene_ids
) == set(
    eligible_scene_table[
        "scene_id"
    ]
)


# ------------------------------------------------------------
# Expand back to scene-view pairs
# ------------------------------------------------------------

dpssd_train_pairs = eligible_pairs[
    eligible_pairs[
        "scene_id"
    ].isin(
        train_scene_ids
    )
].reset_index(drop=True)


dpssd_val_pairs = eligible_pairs[
    eligible_pairs[
        "scene_id"
    ].isin(
        val_scene_ids
    )
].reset_index(drop=True)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 78)
print("P3-C20 — DP-SSD INTERNAL SPLIT")
print("=" * 78)

print(
    "Eligible scenes       :",
    len(eligible_scene_table)
)

print(
    "DP-SSD train scenes   :",
    len(train_scene_ids)
)

print(
    "DP-SSD val scenes     :",
    len(val_scene_ids)
)

print(
    "\nTrain scene-view pairs:",
    len(dpssd_train_pairs)
)

print(
    "Val scene-view pairs  :",
    len(dpssd_val_pairs)
)


print("\nTRAIN regions:")

print(
    dpssd_train_pairs[
        ["scene_id", "region"]
    ]
    .drop_duplicates()[
        "region"
    ]
    .value_counts()
    .sort_index()
)


print("\nINTERNAL VAL regions:")

print(
    dpssd_val_pairs[
        ["scene_id", "region"]
    ]
    .drop_duplicates()[
        "region"
    ]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

dpssd_train_pairs.to_csv(
    DPSSD_ROOT /
    "dpssd_train_pairs_v1.csv",
    index=False
)

dpssd_val_pairs.to_csv(
    DPSSD_ROOT /
    "dpssd_internal_val_pairs_v1.csv",
    index=False
)

print(
    "\nDP-SSD internal split frozen ✅"
)

print(
    "Main FloodShift validation/test were NOT touched."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C21
# SECTION: P3-S7 — LEAKAGE-SAFE DP-SSD NORMALIZATION
#
# PURPOSE:
#   Compute DP-SSD normalization statistics using ONLY the
#   277 DP-SSD internal-training scenes created in P3-C20.
#
# DOMAIN:
#   dB -> linear intensity -> natural log intensity
#
# IMPORTANT:
#   - DP-SSD internal validation is NOT used.
#   - Canonical FloodShift validation/test are NOT used.
#   - No segmentation labels are used.
# ============================================================

from pathlib import Path
import json
import numpy as np
import rasterio

print("=" * 78)
print("P3-C21 — LEAKAGE-SAFE DP-SSD NORMALIZATION")
print("=" * 78)


DPSSD_TRAIN_SCENE_IDS = sorted(
    train_scene_ids
)

assert len(
    DPSSD_TRAIN_SCENE_IDS
) == 277

assert set(
    DPSSD_TRAIN_SCENE_IDS
).isdisjoint(
    val_scene_ids
)


print(
    "DP-SSD training scenes:",
    len(DPSSD_TRAIN_SCENE_IDS)
)

print(
    "DP-SSD internal-val scenes:",
    len(val_scene_ids)
)


# ------------------------------------------------------------
# Exact streaming statistics
# ------------------------------------------------------------

running = {

    "VV": {
        "count": 0,
        "sum": 0.0,
        "sum_sq": 0.0
    },

    "VH": {
        "count": 0,
        "sum": 0.0,
        "sum_sq": 0.0
    }
}

band_names = [
    "VV",
    "VH"
]


for scene_num, scene_id_i in enumerate(
    DPSSD_TRAIN_SCENE_IDS,
    start=1
):

    row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        row["sar_raw_path"]
    )

    with rasterio.open(
        sar_path_i
    ) as src:

        raw_db = src.read().astype(
            np.float32
        )


    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    if not valid.any():
        continue


    for band_idx, band_name in enumerate(
        band_names
    ):

        db_values = raw_db[
            band_idx
        ][valid].astype(
            np.float64
        )

        intensity = np.power(
            10.0,
            db_values / 10.0
        )

        log_values = np.log(
            np.maximum(
                intensity,
                1e-12
            )
        )

        running[
            band_name
        ]["count"] += log_values.size

        running[
            band_name
        ]["sum"] += float(
            log_values.sum()
        )

        running[
            band_name
        ]["sum_sq"] += float(
            np.square(
                log_values
            ).sum()
        )


    if (
        scene_num % 50 == 0
        or
        scene_num
        ==
        len(DPSSD_TRAIN_SCENE_IDS)
    ):

        print(
            f"Processed "
            f"{scene_num:>3}/"
            f"{len(DPSSD_TRAIN_SCENE_IDS)} scenes"
        )


# ------------------------------------------------------------
# Finalize statistics
# ------------------------------------------------------------

dpssd_norm_v2 = {}

for band_name in band_names:

    n = running[
        band_name
    ]["count"]

    total = running[
        band_name
    ]["sum"]

    total_sq = running[
        band_name
    ]["sum_sq"]

    mean = total / n

    variance = (
        total_sq / n
        -
        mean ** 2
    )

    std = np.sqrt(
        max(
            variance,
            0.0
        )
    )

    dpssd_norm_v2[
        band_name
    ] = {

        "valid_training_pixels":
            int(n),

        "mean_log_intensity":
            float(mean),

        "std_log_intensity":
            float(std)
    }


dpssd_norm_v2[
    "metadata"
] = {

    "version":
        "v2",

    "purpose":
        "DP-SSD self-supervised despeckler normalization",

    "scene_count":
        len(DPSSD_TRAIN_SCENE_IDS),

    "source_split":
        "DP-SSD internal training only",

    "canonical_validation_used":
        False,

    "canonical_test_used":
        False,

    "dpssd_internal_validation_used":
        False
}


# ------------------------------------------------------------
# Freeze normalization constants for later cells
# ------------------------------------------------------------

VV_MEAN = dpssd_norm_v2[
    "VV"
]["mean_log_intensity"]

VV_STD = dpssd_norm_v2[
    "VV"
]["std_log_intensity"]

VH_MEAN = dpssd_norm_v2[
    "VH"
]["mean_log_intensity"]

VH_STD = dpssd_norm_v2[
    "VH"
]["std_log_intensity"]


print("\n" + "-" * 78)
print("FINAL DP-SSD NORMALIZATION")
print("-" * 78)

print(
    f"VV mean={VV_MEAN:.6f}, "
    f"std={VV_STD:.6f}"
)

print(
    f"VH mean={VH_MEAN:.6f}, "
    f"std={VH_STD:.6f}"
)

print("\nValid training pixels:")

print(
    "VV:",
    f"{dpssd_norm_v2['VV']['valid_training_pixels']:,}"
)

print(
    "VH:",
    f"{dpssd_norm_v2['VH']['valid_training_pixels']:,}"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

norm_v2_path = (
    DPSSD_ROOT /
    "dpssd_train_only_log_normalization_v2.json"
)

with open(
    norm_v2_path,
    "w"
) as f:

    json.dump(
        dpssd_norm_v2,
        f,
        indent=2
    )


print("\nSaved:")
print(norm_v2_path)

print(
    "\nLeakage-safe DP-SSD normalization frozen ✅"
)

print(
    "Internal validation and canonical validation/test "
    "remain unseen."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C22
# SECTION: P3-S7 — FINAL DP-SSD TRAIN / VALIDATION LOADERS
#
# PURPOSE:
#   Build the final leakage-safe self-supervised data pipeline.
#
# TRAIN:
#   - stochastic valid crop
#   - stochastic shared VV/VH masking
#
# INTERNAL VALIDATION:
#   - fixed valid crop
#   - fixed mask
#
# IMPORTANT:
#   - normalization comes ONLY from 277 DP-SSD train scenes
#   - train/validation scenes do not overlap
#   - canonical FloodShift validation/test remain untouched
# ============================================================

import hashlib
import numpy as np
import pandas as pd
import rasterio
import torch

from torch.utils.data import Dataset, DataLoader


# ============================================================
# CONFIG
# ============================================================

DPSSD_CROP_SIZE = 128
DPSSD_MASK_RATIO = 0.10

TRAIN_BATCH_SIZE = 16
VAL_BATCH_SIZE = 16

TRAIN_SEED = 42
VAL_SEED = 2026


print("=" * 80)
print("P3-C22 — FINAL DP-SSD TRAIN / VALIDATION LOADERS")
print("=" * 80)

print("\nNormalization used:")

print(
    f"VV mean={VV_MEAN:.6f}, "
    f"std={VV_STD:.6f}"
)

print(
    f"VH mean={VH_MEAN:.6f}, "
    f"std={VH_STD:.6f}"
)


# ============================================================
# INDEX RAW SAR FILES
# ============================================================

sar_files = list(
    INPUT_ROOT.rglob(
        "*_S1Hand.tif"
    )
)

sar_file_index = {
    p.name: p
    for p in sar_files
}

print(
    "\nIndexed raw SAR files:",
    len(sar_file_index)
)

assert len(
    sar_file_index
) == 446


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_log_sar(
    log_sar
):

    out = log_sar.copy().astype(
        np.float32
    )

    out[0] = (
        out[0] - VV_MEAN
    ) / VV_STD

    out[1] = (
        out[1] - VH_MEAN
    ) / VH_STD

    return out


# ============================================================
# SHARED VV/VH NEIGHBOUR MASKING
# ============================================================

def shared_neighbour_masking(
    target,
    mask_ratio,
    rng
):

    C, H, W = target.shape

    mask = (
        rng.random(
            (H, W)
        )
        <
        mask_ratio
    )

    y, x = np.where(
        mask
    )

    direction = rng.integers(
        0,
        4,
        size=len(y)
    )

    ny = y.copy()
    nx = x.copy()

    ny[
        direction == 0
    ] -= 1

    ny[
        direction == 1
    ] += 1

    nx[
        direction == 2
    ] -= 1

    nx[
        direction == 3
    ] += 1


    # Boundary correction: always use a genuine neighbour.
    bad = ny < 0
    ny[bad] = y[bad] + 1

    bad = ny >= H
    ny[bad] = y[bad] - 1

    bad = nx < 0
    nx[bad] = x[bad] + 1

    bad = nx >= W
    nx[bad] = x[bad] - 1


    masked = target.copy()

    for channel in range(C):

        masked[
            channel,
            y,
            x
        ] = target[
            channel,
            ny,
            nx
        ]


    return (
        masked.astype(
            np.float32
        ),
        mask.astype(
            bool
        )
    )


# ============================================================
# STABLE SEED FOR FIXED VALIDATION
# ============================================================

def stable_seed(
    scene_id,
    view_name,
    base_seed
):

    text = (
        f"{scene_id}|"
        f"{view_name}|"
        f"{base_seed}"
    )

    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False
    ) % (2**32)


# ============================================================
# CREATE FIXED INTERNAL-VALIDATION PLAN
# ============================================================

val_plan_records = []

for _, row in dpssd_val_pairs.iterrows():

    scene_id_i = row[
        "scene_id"
    ]

    view_name = row[
        "view"
    ]

    filename = row.get(
        "sar_filename",
        None
    )

    if (
        filename is None
        or
        pd.isna(filename)
    ):

        manifest_row = train_df[
            train_df["scene_id"]
            ==
            scene_id_i
        ].iloc[0]

        filename = Path(
            manifest_row[
                "sar_raw_path"
            ]
        ).name


    sar_path_i = sar_file_index[
        filename
    ]

    with rasterio.open(
        sar_path_i
    ) as src:

        raw_db = src.read().astype(
            np.float32
        )


    valid_full = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    ro, co = VIEW_OFFSETS[
        view_name
    ]

    valid_view = valid_full[
        ro::2,
        co::2
    ]

    yy, xx = valid_crop_locations(
        valid_view,
        DPSSD_CROP_SIZE
    )

    if len(yy) == 0:

        raise RuntimeError(
            f"No valid validation crop for "
            f"{scene_id_i}/{view_name}"
        )


    rng = np.random.default_rng(
        stable_seed(
            scene_id_i,
            view_name,
            VAL_SEED
        )
    )

    choice = int(
        rng.integers(
            0,
            len(yy)
        )
    )


    val_plan_records.append(
        {
            "scene_id":
                scene_id_i,

            "region":
                row["region"],

            "view":
                view_name,

            "sar_filename":
                filename,

            "crop_y":
                int(
                    yy[choice]
                ),

            "crop_x":
                int(
                    xx[choice]
                ),

            "mask_seed":
                int(
                    stable_seed(
                        scene_id_i,
                        view_name,
                        VAL_SEED + 100_000
                    )
                )
        }
    )


dpssd_val_plan = pd.DataFrame(
    val_plan_records
)

assert len(
    dpssd_val_plan
) == len(
    dpssd_val_pairs
)


# ============================================================
# FINAL DATASET
# ============================================================

class DPSSDFinalDataset(Dataset):

    def __init__(
        self,
        table,
        sar_file_index,
        mode="train",
        crop_size=128,
        mask_ratio=0.10,
        seed=42
    ):

        self.table = table.reset_index(
            drop=True
        )

        self.sar_file_index = (
            sar_file_index
        )

        self.mode = mode
        self.crop_size = crop_size
        self.mask_ratio = mask_ratio

        if mode not in {
            "train",
            "val"
        }:

            raise ValueError(
                "mode must be 'train' or 'val'"
            )

        # num_workers=0 below makes this reproducible within
        # a run while still evolving across training samples.
        self.rng = np.random.default_rng(
            seed
        )


    def __len__(
        self
    ):

        return len(
            self.table
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.table.iloc[
            idx
        ]

        scene_id = row[
            "scene_id"
        ]

        view_name = row[
            "view"
        ]

        filename = row.get(
            "sar_filename",
            None
        )

        if (
            filename is None
            or
            pd.isna(filename)
        ):

            manifest_row = train_df[
                train_df["scene_id"]
                ==
                scene_id
            ].iloc[0]

            filename = Path(
                manifest_row[
                    "sar_raw_path"
                ]
            ).name


        sar_path = self.sar_file_index[
            filename
        ]

        with rasterio.open(
            sar_path
        ) as src:

            raw_db = src.read().astype(
                np.float32
            )


        valid_full = (
            np.isfinite(
                raw_db[0]
            )
            &
            np.isfinite(
                raw_db[1]
            )
        )


        ro, co = VIEW_OFFSETS[
            view_name
        ]

        view_db = raw_db[
            :,
            ro::2,
            co::2
        ]

        view_valid = valid_full[
            ro::2,
            co::2
        ]

        H, W = view_valid.shape
        crop = self.crop_size


        # ----------------------------------------------------
        # TRAIN: stochastic valid crop
        # ----------------------------------------------------

        if self.mode == "train":

            rng = self.rng
            found = False

            for _ in range(100):

                y = int(
                    rng.integers(
                        0,
                        H - crop + 1
                    )
                )

                x = int(
                    rng.integers(
                        0,
                        W - crop + 1
                    )
                )

                if view_valid[
                    y:y+crop,
                    x:x+crop
                ].all():

                    found = True
                    break


            if not found:

                yy, xx = valid_crop_locations(
                    view_valid,
                    crop
                )

                if len(yy) == 0:

                    raise RuntimeError(
                        f"No valid crop for "
                        f"{scene_id}/{view_name}"
                    )

                choice = int(
                    rng.integers(
                        0,
                        len(yy)
                    )
                )

                y = int(
                    yy[choice]
                )

                x = int(
                    xx[choice]
                )


        # ----------------------------------------------------
        # VAL: frozen crop and frozen mask
        # ----------------------------------------------------

        else:

            y = int(
                row[
                    "crop_y"
                ]
            )

            x = int(
                row[
                    "crop_x"
                ]
            )

            rng = np.random.default_rng(
                int(
                    row[
                        "mask_seed"
                    ]
                )
            )


        crop_db = view_db[
            :,
            y:y+crop,
            x:x+crop
        ]

        if not np.isfinite(
            crop_db
        ).all():

            raise RuntimeError(
                f"Non-finite crop: "
                f"{scene_id}/{view_name}"
            )


        # dB -> linear intensity -> log intensity
        intensity = np.power(
            10.0,
            crop_db / 10.0
        )

        log_crop = np.log(
            np.maximum(
                intensity,
                1e-12
            )
        ).astype(
            np.float32
        )


        target = normalize_log_sar(
            log_crop
        )

        masked_input, mask = (
            shared_neighbour_masking(
                target,
                self.mask_ratio,
                rng
            )
        )


        return {

            "input":
                torch.from_numpy(
                    masked_input
                ).float(),

            "target":
                torch.from_numpy(
                    target
                ).float(),

            "mask":
                torch.from_numpy(
                    mask[None]
                ).bool(),

            "scene_id":
                scene_id,

            "view":
                view_name,

            "crop_y":
                y,

            "crop_x":
                x
        }


# ============================================================
# DATASETS + SEEDED DATALOADERS
# ============================================================

dpssd_train_dataset = DPSSDFinalDataset(

    table=
        dpssd_train_pairs,

    sar_file_index=
        sar_file_index,

    mode=
        "train",

    crop_size=
        DPSSD_CROP_SIZE,

    mask_ratio=
        DPSSD_MASK_RATIO,

    seed=
        TRAIN_SEED
)


dpssd_val_dataset = DPSSDFinalDataset(

    table=
        dpssd_val_plan,

    sar_file_index=
        sar_file_index,

    mode=
        "val",

    crop_size=
        DPSSD_CROP_SIZE,

    mask_ratio=
        DPSSD_MASK_RATIO,

    seed=
        VAL_SEED
)


loader_generator = torch.Generator()

loader_generator.manual_seed(
    TRAIN_SEED
)


dpssd_train_loader = DataLoader(

    dpssd_train_dataset,

    batch_size=
        TRAIN_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),

    drop_last=True,

    generator=
        loader_generator
)


dpssd_val_loader = DataLoader(

    dpssd_val_dataset,

    batch_size=
        VAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),

    drop_last=False
)


# ============================================================
# LEAKAGE + DETERMINISM CHECKS
# ============================================================

train_ids_final = set(
    dpssd_train_pairs[
        "scene_id"
    ]
)

val_ids_final = set(
    dpssd_val_plan[
        "scene_id"
    ]
)

assert train_ids_final.isdisjoint(
    val_ids_final
)

assert len(
    train_ids_final
) == 277

assert len(
    val_ids_final
) == 31

assert len(
    dpssd_train_dataset
) == 1108

assert len(
    dpssd_val_dataset
) == 124


val_a = dpssd_val_dataset[0]
val_b = dpssd_val_dataset[0]

assert torch.equal(
    val_a[
        "input"
    ],
    val_b[
        "input"
    ]
)

assert torch.equal(
    val_a[
        "target"
    ],
    val_b[
        "target"
    ]
)

assert torch.equal(
    val_a[
        "mask"
    ],
    val_b[
        "mask"
    ]
)


# ============================================================
# BATCH SANITY
# ============================================================

batch = next(
    iter(
        dpssd_train_loader
    )
)

val_batch = next(
    iter(
        dpssd_val_loader
    )
)


print("\nDataset sizes:")
print(
    "Train:",
    len(
        dpssd_train_dataset
    )
)
print(
    "Val  :",
    len(
        dpssd_val_dataset
    )
)


print("\nTRAIN batch")
print(
    "Input :",
    tuple(
        batch[
            "input"
        ].shape
    )
)
print(
    "Target:",
    tuple(
        batch[
            "target"
        ].shape
    )
)
print(
    "Mask  :",
    tuple(
        batch[
            "mask"
        ].shape
    )
)
print(
    "Mask ratio:",
    f"{batch['mask'].float().mean().item()*100:.2f}%"
)


print("\nVAL batch")
print(
    "Input :",
    tuple(
        val_batch[
            "input"
        ].shape
    )
)
print(
    "Target:",
    tuple(
        val_batch[
            "target"
        ].shape
    )
)
print(
    "Mask  :",
    tuple(
        val_batch[
            "mask"
        ].shape
    )
)
print(
    "Mask ratio:",
    f"{val_batch['mask'].float().mean().item()*100:.2f}%"
)


# ------------------------------------------------------------
# Save frozen validation plan
# ------------------------------------------------------------

val_plan_path = (
    DPSSD_ROOT /
    "dpssd_fixed_internal_validation_plan_v1.csv"
)

dpssd_val_plan.to_csv(
    val_plan_path,
    index=False
)


print(
    "\nValidation deterministic: True"
)

print("\nSaved:")
print(val_plan_path)

print(
    "\nFinal DP-SSD data pipeline ready ✅"
)


## 4. DP-SSD model and sanity checks


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C23
# SECTION: P3-S8 — DP-SSD NETWORK
#
# PURPOSE:
#   Define the proposed lightweight dual-polarization
#   self-supervised despeckling network.
#
# DESIGN:
#   separate VV/VH stems
#       ↓
#   fused residual encoder
#       ↓
#   bottleneck
#       ↓
#   decoder with ordinary skip connections
#       ↓
#   2-channel clean log-SAR prediction
#
# NO attention / Transformer is used here.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# BASIC RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(
        self,
        channels
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.act = nn.GELU()


    def forward(
        self,
        x
    ):

        residual = x

        x = self.conv1(x)
        x = self.act(x)

        x = self.conv2(x)

        x = x + residual

        return self.act(x)


# ============================================================
# ENCODER STAGE
# ============================================================

class EncoderStage(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.down = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=2,
            padding=1
        )

        self.block = ResidualBlock(
            out_channels
        )


    def forward(
        self,
        x
    ):

        x = self.down(x)
        x = self.block(x)

        return x


# ============================================================
# DECODER STAGE
# ============================================================

class DecoderStage(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels
    ):

        super().__init__()

        self.project = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.fuse = nn.Conv2d(
            out_channels
            +
            skip_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.block = ResidualBlock(
            out_channels
        )


    def forward(
        self,
        x,
        skip
    ):

        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        x = self.project(x)

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.fuse(x)

        x = self.block(x)

        return x


# ============================================================
# DP-SSD
# ============================================================

class DPSSDNet(nn.Module):

    def __init__(
        self,
        stem_channels=16,
        base_channels=32
    ):

        super().__init__()


        # ----------------------------------------------------
        # Polarization-specific stems
        # ----------------------------------------------------

        self.vv_stem = nn.Sequential(

            nn.Conv2d(
                1,
                stem_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GELU(),

            ResidualBlock(
                stem_channels
            )
        )


        self.vh_stem = nn.Sequential(

            nn.Conv2d(
                1,
                stem_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GELU(),

            ResidualBlock(
                stem_channels
            )
        )


        # ----------------------------------------------------
        # Polarization fusion
        # ----------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Conv2d(
                stem_channels * 2,
                base_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GELU(),

            ResidualBlock(
                base_channels
            )
        )


        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        self.enc1 = EncoderStage(
            base_channels,
            base_channels * 2
        )

        self.enc2 = EncoderStage(
            base_channels * 2,
            base_channels * 4
        )


        # ----------------------------------------------------
        # Bottleneck
        # ----------------------------------------------------

        self.bottleneck = nn.Sequential(

            ResidualBlock(
                base_channels * 4
            ),

            ResidualBlock(
                base_channels * 4
            )
        )


        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.dec1 = DecoderStage(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 2,
            out_channels=base_channels * 2
        )


        self.dec2 = DecoderStage(
            in_channels=base_channels * 2,
            skip_channels=base_channels,
            out_channels=base_channels
        )


        # ----------------------------------------------------
        # Output
        #
        # Direct clean normalized log-SAR prediction.
        # No global input→output residual shortcut.
        # ----------------------------------------------------

        self.output_head = nn.Conv2d(
            base_channels,
            2,
            kernel_size=1
        )


    def forward(
        self,
        x
    ):

        assert x.shape[1] == 2


        # Separate polarizations
        vv = x[:, 0:1]
        vh = x[:, 1:2]


        # Polarization-specific features
        vv = self.vv_stem(
            vv
        )

        vh = self.vh_stem(
            vh
        )


        # Fusion
        x0 = torch.cat(
            [vv, vh],
            dim=1
        )

        x0 = self.fusion(
            x0
        )


        # Encoder
        x1 = self.enc1(
            x0
        )

        x2 = self.enc2(
            x1
        )


        # Bottleneck
        x2 = self.bottleneck(
            x2
        )


        # Decoder
        x = self.dec1(
            x2,
            x1
        )

        x = self.dec2(
            x,
            x0
        )


        # Prediction
        out = self.output_head(
            x
        )

        return out


# ============================================================
# SANITY TEST
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = DPSSDNet().to(
    device
)


test_input = batch[
    "input"
].to(
    device
)


with torch.no_grad():

    test_output = model(
        test_input
    )


print("=" * 78)
print("P3-C23 — DP-SSD NETWORK SANITY TEST")
print("=" * 78)

print("Device      :", device)

print(
    "Input shape :",
    tuple(
        test_input.shape
    )
)

print(
    "Output shape:",
    tuple(
        test_output.shape
    )
)

print(
    "Output finite:",
    bool(
        torch.isfinite(
            test_output
        ).all()
    )
)


parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"\nParameters          : "
    f"{parameter_count:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_count:,}"
)


assert (
    test_output.shape
    ==
    test_input.shape
)

assert torch.isfinite(
    test_output
).all()


print(
    "\nDP-SSD architecture verified ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C24
# SECTION: P3-S8 — MASKED SELF-SUPERVISED LOSS
#
# PURPOSE:
#   Implement and verify the DP-SSD masked reconstruction loss.
#
# LOSS:
#   SmoothL1(prediction, target)
#   ONLY at intentionally hidden spatial positions.
#
# IMPORTANT:
#   The same spatial mask applies to VV and VH.
# ============================================================

import torch
import torch.nn.functional as F


# ============================================================
# MASKED SMOOTHL1 LOSS
# ============================================================

def masked_smooth_l1_loss(
    prediction,
    target,
    mask,
    beta=1.0
):
    """
    prediction : [B, 2, H, W]
    target     : [B, 2, H, W]
    mask       : [B, 1, H, W] boolean

    Loss is evaluated ONLY at masked pixels.
    """

    if prediction.shape != target.shape:
        raise ValueError(
            f"Prediction/target mismatch: "
            f"{prediction.shape} vs {target.shape}"
        )

    if mask.ndim != 4 or mask.shape[1] != 1:
        raise ValueError(
            f"Expected mask [B,1,H,W], got {mask.shape}"
        )

    # --------------------------------------------------------
    # Expand same spatial mask over VV and VH
    # --------------------------------------------------------

    mask_2ch = mask.expand(
        -1,
        prediction.shape[1],
        -1,
        -1
    )

    # --------------------------------------------------------
    # Pixel-wise SmoothL1
    # --------------------------------------------------------

    loss_map = F.smooth_l1_loss(
        prediction,
        target,
        reduction="none",
        beta=beta
    )

    masked_values = loss_map[
        mask_2ch
    ]

    if masked_values.numel() == 0:
        raise RuntimeError(
            "No masked pixels found in batch."
        )

    return masked_values.mean()


# ============================================================
# MOVE TEST BATCH + MODEL TO CURRENT DEVICE
# ============================================================

model = model.to(device)

input_batch = batch[
    "input"
].to(device)

target_batch = batch[
    "target"
].to(device)

mask_batch = batch[
    "mask"
].to(device)


# ============================================================
# FORWARD PASS
# ============================================================

model.train()

prediction = model(
    input_batch
)


# ============================================================
# MASKED LOSS
# ============================================================

loss = masked_smooth_l1_loss(
    prediction,
    target_batch,
    mask_batch
)


print("=" * 78)
print("P3-C24 — MASKED SELF-SUPERVISED LOSS")
print("=" * 78)

print(
    "Prediction shape:",
    tuple(prediction.shape)
)

print(
    "Target shape    :",
    tuple(target_batch.shape)
)

print(
    "Mask shape      :",
    tuple(mask_batch.shape)
)

print(
    "Mask ratio      :",
    f"{100 * mask_batch.float().mean().item():.2f}%"
)

print(
    "Masked loss     :",
    float(loss.item())
)


# ============================================================
# VERIFY THAT UNMASKED PIXELS DO NOT AFFECT LOSS
#
# We deliberately corrupt target values ONLY outside the mask.
# Correct implementation should produce identical loss.
# ============================================================

target_corrupted_unmasked = (
    target_batch.clone()
)

mask_2ch = mask_batch.expand(
    -1,
    2,
    -1,
    -1
)

target_corrupted_unmasked[
    ~mask_2ch
] += 1000.0


loss_after_unmasked_corruption = (
    masked_smooth_l1_loss(
        prediction,
        target_corrupted_unmasked,
        mask_batch
    )
)


difference = abs(
    loss.item()
    -
    loss_after_unmasked_corruption.item()
)


print(
    "\nLoss after corrupting UNMASKED targets:",
    float(
        loss_after_unmasked_corruption.item()
    )
)

print(
    "Difference:",
    difference
)


assert difference < 1e-7, (
    "Loss is incorrectly depending on "
    "unmasked pixels."
)


# ============================================================
# SECOND CHECK:
# Corrupt MASKED targets.
# Loss MUST change.
# ============================================================

target_corrupted_masked = (
    target_batch.clone()
)

target_corrupted_masked[
    mask_2ch
] += 5.0


loss_after_masked_corruption = (
    masked_smooth_l1_loss(
        prediction,
        target_corrupted_masked,
        mask_batch
    )
)


print(
    "\nLoss after corrupting MASKED targets:",
    float(
        loss_after_masked_corruption.item()
    )
)


assert not torch.isclose(
    loss,
    loss_after_masked_corruption
), (
    "Loss should change when masked "
    "targets are modified."
)


# ============================================================
# BACKPROP TEST
# ============================================================

model.zero_grad(
    set_to_none=True
)

loss.backward()


gradient_parameters = 0
finite_gradients = True

for p in model.parameters():

    if p.grad is not None:

        gradient_parameters += 1

        if not torch.isfinite(
            p.grad
        ).all():

            finite_gradients = False


print(
    "\nParameters receiving gradients:",
    gradient_parameters
)

print(
    "All gradients finite:",
    finite_gradients
)


assert gradient_parameters > 0
assert finite_gradients


print(
    "\nMasked self-supervised loss verified ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C25
# SECTION: P3-S7 — DP-SSD TINY OVERFIT TEST
#
# PURPOSE:
#   Verify that DP-SSD can learn the masked reconstruction task
#   before launching a real training run.
#
# STRATEGY:
#   Train a fresh model repeatedly on ONE FIXED 4-sample batch.
#
# IMPORTANT:
#   This is a debugging sanity test, not a scientific result.
# ============================================================

import torch
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Fresh model + fixed batch
# ------------------------------------------------------------

overfit_model = DPSSDNet().to(
    device
)

fixed_batch = next(
    iter(
        dpssd_train_loader
    )
)

fixed_input = fixed_batch[
    "input"
][:4].to(
    device
)

fixed_target = fixed_batch[
    "target"
][:4].to(
    device
)

fixed_mask = fixed_batch[
    "mask"
][:4].to(
    device
)


optimizer = torch.optim.AdamW(
    overfit_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

N_STEPS = 200
loss_history = []


print("=" * 78)
print("P3-C25 — DP-SSD TINY OVERFIT TEST")
print("=" * 78)

print("Device :", device)
print("Steps  :", N_STEPS)
print(
    "Batch  :",
    tuple(
        fixed_input.shape
    )
)


overfit_model.train()

for step in range(
    1,
    N_STEPS + 1
):

    optimizer.zero_grad(
        set_to_none=True
    )

    prediction = overfit_model(
        fixed_input
    )

    loss = masked_smooth_l1_loss(
        prediction,
        fixed_target,
        fixed_mask
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        overfit_model.parameters(),
        max_norm=5.0
    )

    optimizer.step()

    loss_history.append(
        float(
            loss.item()
        )
    )

    if (
        step == 1
        or
        step % 20 == 0
    ):

        print(
            f"Step {step:>3}/{N_STEPS}  "
            f"Loss = {loss.item():.6f}"
        )


initial_loss = loss_history[0]
final_loss = loss_history[-1]

reduction_pct = (
    100.0
    *
    (
        initial_loss
        -
        final_loss
    )
    /
    initial_loss
)


print("\n" + "-" * 78)

print(
    f"Initial loss : "
    f"{initial_loss:.6f}"
)

print(
    f"Final loss   : "
    f"{final_loss:.6f}"
)

print(
    f"Reduction    : "
    f"{reduction_pct:.2f}%"
)


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    loss_history
)

plt.xlabel(
    "Optimization step"
)

plt.ylabel(
    "Masked SmoothL1 loss"
)

plt.title(
    "DP-SSD Tiny Overfit Test"
)

plt.grid(
    alpha=0.2
)

plt.show()


assert final_loss < initial_loss

assert torch.isfinite(
    torch.tensor(
        loss_history
    )
).all()


print(
    "\nDP-SSD can learn the masked reconstruction task ✅"
)


## 5. DP-SSD controlled training and evaluation


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C26
# SECTION: P3-S9 — DP-SSD L0 REAL TRAINING
#
# PURPOSE:
#   Train the baseline DP-SSD self-supervised despeckler.
#
# OBJECTIVE:
#   L0 = masked SmoothL1 reconstruction only.
#
# FROZEN CONFIG:
#   epochs=60, lr=3e-4, AdamW, wd=1e-5,
#   grad clip=5, ReduceLROnPlateau, early stop=10.
#
# IMPORTANT:
#   - fresh model
#   - canonical FloodShift validation/test are untouched
#   - internal DP-SSD validation only
# ============================================================

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch

print("=" * 84)
print("P3-C26 — DP-SSD L0 REAL TRAINING")
print("=" * 84)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

L0_SEED = 42
random.seed(L0_SEED)
np.random.seed(L0_SEED)
torch.manual_seed(L0_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(L0_SEED)

# Reset the train dataset/shuffle state because C22/C25 perform
# sanity reads before real training. L0 always starts from the
# same frozen stochastic sequence.
dpssd_train_dataset.rng = np.random.default_rng(
    TRAIN_SEED
)
loader_generator.manual_seed(
    TRAIN_SEED
)

# ------------------------------------------------------------
# Frozen training configuration
# ------------------------------------------------------------

L0_MAX_EPOCHS = 60
L0_LR = 3e-4
L0_WEIGHT_DECAY = 1e-5
L0_GRAD_CLIP = 5.0
L0_BETA = 1.0
L0_EARLY_STOP_PATIENCE = 10

DPSSD_L0_ROOT = DPSSD_ROOT / "L0_masked_smoothl1"
DPSSD_L0_ROOT.mkdir(parents=True, exist_ok=True)

L0_BEST_PATH = DPSSD_L0_ROOT / "dpssd_L0_best.pth"
L0_LAST_PATH = DPSSD_L0_ROOT / "dpssd_L0_last.pth"
L0_HISTORY_PATH = DPSSD_L0_ROOT / "dpssd_L0_history.csv"
L0_CONFIG_PATH = DPSSD_L0_ROOT / "dpssd_L0_config.json"

dpssd_L0_model = DPSSDNet().to(device)

optimizer_L0 = torch.optim.AdamW(
    dpssd_L0_model.parameters(),
    lr=L0_LR,
    weight_decay=L0_WEIGHT_DECAY
)

scheduler_L0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_L0,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

use_amp = bool(torch.cuda.is_available())

try:
    scaler_L0 = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )
except TypeError:
    scaler_L0 = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

print("Device      :", device)
print("Train pairs :", len(dpssd_train_dataset))
print("Val pairs   :", len(dpssd_val_dataset))
print("Epochs      :", L0_MAX_EPOCHS)
print("LR          :", L0_LR)
print("AMP         :", use_amp)

history_L0 = []
best_val_L0 = float("inf")
best_epoch_L0 = 0
epochs_without_improvement = 0

training_start = time.perf_counter()

for epoch in range(1, L0_MAX_EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    dpssd_L0_model.train()
    train_losses = []

    for batch_i in dpssd_train_loader:

        x = batch_i["input"].to(
            device,
            non_blocking=True
        )

        y = batch_i["target"].to(
            device,
            non_blocking=True
        )

        m = batch_i["mask"].to(
            device,
            non_blocking=True
        )

        optimizer_L0.zero_grad(
            set_to_none=True
        )

        if use_amp:
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                pred = dpssd_L0_model(x)

                train_loss = masked_smooth_l1_loss(
                    pred,
                    y,
                    m,
                    beta=L0_BETA
                )

            scaler_L0.scale(
                train_loss
            ).backward()

            scaler_L0.unscale_(
                optimizer_L0
            )

            torch.nn.utils.clip_grad_norm_(
                dpssd_L0_model.parameters(),
                L0_GRAD_CLIP
            )

            scaler_L0.step(
                optimizer_L0
            )

            scaler_L0.update()

        else:
            pred = dpssd_L0_model(x)

            train_loss = masked_smooth_l1_loss(
                pred,
                y,
                m,
                beta=L0_BETA
            )

            train_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                dpssd_L0_model.parameters(),
                L0_GRAD_CLIP
            )

            optimizer_L0.step()

        train_losses.append(
            float(train_loss.detach().item())
        )

    # --------------------------------------------------------
    # INTERNAL VALIDATION — FP32
    # --------------------------------------------------------

    dpssd_L0_model.eval()
    val_losses = []

    with torch.no_grad():

        for batch_i in dpssd_val_loader:

            x = batch_i["input"].to(
                device,
                non_blocking=True
            )

            y = batch_i["target"].to(
                device,
                non_blocking=True
            )

            m = batch_i["mask"].to(
                device,
                non_blocking=True
            )

            pred = dpssd_L0_model(
                x
            )

            val_loss = masked_smooth_l1_loss(
                pred,
                y,
                m,
                beta=L0_BETA
            )

            val_losses.append(
                float(val_loss.item())
            )

    mean_train_loss = float(
        np.mean(train_losses)
    )

    mean_val_loss = float(
        np.mean(val_losses)
    )

    current_lr = float(
        optimizer_L0.param_groups[0]["lr"]
    )

    scheduler_L0.step(
        mean_val_loss
    )

    history_L0.append({
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "val_loss": mean_val_loss,
        "lr": current_lr
    })

    improved = (
        mean_val_loss
        <
        best_val_L0
        -
        1e-8
    )

    if improved:

        best_val_L0 = mean_val_loss
        best_epoch_L0 = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict":
                    dpssd_L0_model.state_dict(),

                "epoch":
                    epoch,

                "val_loss":
                    mean_val_loss,

                "config": {
                    "objective":
                        "masked_SmoothL1",

                    "beta":
                        L0_BETA,

                    "lr":
                        L0_LR,

                    "weight_decay":
                        L0_WEIGHT_DECAY,

                    "crop_size":
                        DPSSD_CROP_SIZE,

                    "mask_ratio":
                        DPSSD_MASK_RATIO,
                }
            },
            L0_BEST_PATH
        )

    else:
        epochs_without_improvement += 1

    if (
        epoch == 1
        or
        epoch % 5 == 0
        or
        epoch == L0_MAX_EPOCHS
    ):
        print(
            f"Epoch {epoch:02d}/{L0_MAX_EPOCHS} | "
            f"train={mean_train_loss:.6f} | "
            f"val={mean_val_loss:.6f} | "
            f"best={best_val_L0:.6f} @ {best_epoch_L0:02d} | "
            f"lr={current_lr:.2e}"
        )

    if (
        epochs_without_improvement
        >=
        L0_EARLY_STOP_PATIENCE
    ):
        print(
            f"Early stopping at epoch {epoch}."
        )
        break

# ------------------------------------------------------------
# Save last + history/config
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict":
            dpssd_L0_model.state_dict(),

        "epoch":
            history_L0[-1]["epoch"],

        "val_loss":
            history_L0[-1]["val_loss"]
    },
    L0_LAST_PATH
)

history_L0_df = pd.DataFrame(
    history_L0
)

history_L0_df.to_csv(
    L0_HISTORY_PATH,
    index=False
)

L0_CONFIG = {
    "seed": L0_SEED,
    "max_epochs": L0_MAX_EPOCHS,
    "lr": L0_LR,
    "weight_decay": L0_WEIGHT_DECAY,
    "grad_clip": L0_GRAD_CLIP,
    "beta": L0_BETA,
    "mask_ratio": DPSSD_MASK_RATIO,
    "crop_size": DPSSD_CROP_SIZE,
    "train_pairs": len(dpssd_train_dataset),
    "val_pairs": len(dpssd_val_dataset),
    "canonical_validation_used": False,
    "canonical_test_used": False
}

with open(
    L0_CONFIG_PATH,
    "w"
) as f:
    json.dump(
        L0_CONFIG,
        f,
        indent=2
    )

elapsed_min = (
    time.perf_counter()
    -
    training_start
) / 60.0

# ------------------------------------------------------------
# Reload BEST model for all later evaluation
# ------------------------------------------------------------

best_checkpoint_L0 = torch.load(
    L0_BEST_PATH,
    map_location=device,
    weights_only=False
)

dpssd_L0_model.load_state_dict(
    best_checkpoint_L0[
        "model_state_dict"
    ]
)

dpssd_L0_model.eval()

print("\n" + "-" * 84)
print("DP-SSD L0 TRAINING COMPLETE")
print("-" * 84)
print("Elapsed minutes :", f"{elapsed_min:.2f}")
print("Best epoch      :", best_epoch_L0)
print("Best val loss   :", f"{best_val_L0:.10f}")
print("Best checkpoint :", L0_BEST_PATH)
print("Last checkpoint :", L0_LAST_PATH)
print("History         :", L0_HISTORY_PATH)
print("Config          :", L0_CONFIG_PATH)

assert L0_BEST_PATH.exists()
assert np.isfinite(best_val_L0)

print("\nP3-C26 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C27
# SECTION: P3-S10 — DP-SSD L0 BLIND/J-INVARIANT EVALUATION
#
# PURPOSE:
#   Evaluate the trained L0 model using deterministic
#   10-fold masked reconstruction rather than naive forward
#   inference.
#
# METRICS:
#   ENL gain, EGR, EGC, radiometric intensity ratio, runtime.
#
# IMPORTANT:
#   Same three frozen representative scenes / homogeneous
#   patches used for every SAR method.
# ============================================================

import time
import numpy as np
import pandas as pd
import rasterio
import torch

print("=" * 88)
print("P3-C27 — DP-SSD L0 BLIND/J-INVARIANT EVALUATION")
print("=" * 88)

# ============================================================
# HELPERS
# ============================================================

DPSSD_EVAL_FOLDS = 10
DPSSD_EVAL_EPS = 1e-8

def _interleave_2x2(
    array_2chw
):
    """Return EE/EO/OE/OO 256x256 views from [2,H,W]."""

    return {
        "EE": array_2chw[:, 0::2, 0::2],
        "EO": array_2chw[:, 0::2, 1::2],
        "OE": array_2chw[:, 1::2, 0::2],
        "OO": array_2chw[:, 1::2, 1::2],
    }


def _merge_2x2(
    views,
    height,
    width
):
    out = np.full(
        (2, height, width),
        np.nan,
        dtype=np.float32
    )

    out[:, 0::2, 0::2] = views["EE"]
    out[:, 0::2, 1::2] = views["EO"]
    out[:, 1::2, 0::2] = views["OE"]
    out[:, 1::2, 1::2] = views["OO"]

    return out


def _denormalize_log_sar(
    normalized
):
    out = normalized.astype(
        np.float32,
        copy=True
    )

    out[0] = (
        out[0] * VV_STD
        +
        VV_MEAN
    )

    out[1] = (
        out[1] * VH_STD
        +
        VH_MEAN
    )

    return out


def _deterministic_neighbour_fill(
    target,
    fold_mask
):
    """
    Replace masked pixels by a deterministic genuine neighbour
    from the same polarization.
    """

    C, H, W = target.shape

    yy, xx = np.where(
        fold_mask
    )

    # Direction depends on coordinates; deterministic.
    direction = (
        yy + 3 * xx
    ) % 4

    ny = yy.copy()
    nx = xx.copy()

    ny[direction == 0] -= 1
    ny[direction == 1] += 1
    nx[direction == 2] -= 1
    nx[direction == 3] += 1

    bad = ny < 0
    ny[bad] = yy[bad] + 1

    bad = ny >= H
    ny[bad] = yy[bad] - 1

    bad = nx < 0
    nx[bad] = xx[bad] + 1

    bad = nx >= W
    nx[bad] = xx[bad] - 1

    filled = target.copy()

    for c in range(C):
        filled[
            c,
            yy,
            xx
        ] = target[
            c,
            ny,
            nx
        ]

    return filled


def dpssd_j_invariant_reconstruct(
    model,
    raw_db,
    folds=10
):
    """
    raw_db: [2,H,W], dB.
    Returns reconstructed dB using deterministic masked
    J-invariant inference.
    """

    raw_db = raw_db.astype(
        np.float32
    )

    H, W = raw_db.shape[-2:]

    valid_full = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    linear = np.power(
        10.0,
        raw_db / 10.0
    ).astype(
        np.float32
    )

    log_sar = np.log(
        np.maximum(
            linear,
            DPSSD_EVAL_EPS
        )
    ).astype(
        np.float32
    )

    normalized = normalize_log_sar(
        log_sar
    )

    norm_views = _interleave_2x2(
        normalized
    )

    valid_views = _interleave_2x2(
        np.stack(
            [
                valid_full.astype(np.float32),
                valid_full.astype(np.float32)
            ],
            axis=0
        )
    )

    reconstructed_views = {}

    model.eval()

    for view_name, view_target in norm_views.items():

        view_valid = (
            valid_views[
                view_name
            ][0]
            >
            0.5
        )

        _, h, w = (
            view_target.shape
        )

        # Model input must never contain NaN/Inf. Invalid
        # footprint positions are a neutral normalized zero and
        # are restored to NaN only after reconstruction.
        view_target_clean = view_target.copy()
        view_target_clean[
            :,
            ~view_valid
        ] = 0.0

        yy, xx = np.indices(
            (h, w)
        )

        prediction_accum = np.zeros_like(
            view_target_clean,
            dtype=np.float32
        )

        prediction_written = np.zeros(
            (h, w),
            dtype=bool
        )

        for fold in range(
            folds
        ):

            fold_mask = (
                (
                    yy * w + xx
                )
                %
                folds
                ==
                fold
            )

            fold_mask &= (
                view_valid
            )

            if not fold_mask.any():
                continue

            masked_input = (
                _deterministic_neighbour_fill(
                    view_target_clean,
                    fold_mask
                )
            )

            x = torch.from_numpy(
                masked_input[
                    None
                ]
            ).to(
                device=device,
                dtype=torch.float32
            )

            with torch.no_grad():
                pred = model(
                    x
                )[0].detach().cpu().numpy()

            prediction_accum[
                :,
                fold_mask
            ] = pred[
                :,
                fold_mask
            ]

            prediction_written[
                fold_mask
            ] = True

        # Keep original values at invalid/unwritten locations.
        prediction_accum[
            :,
            ~prediction_written
        ] = view_target_clean[
            :,
            ~prediction_written
        ]

        reconstructed_views[
            view_name
        ] = prediction_accum

    reconstructed_norm = _merge_2x2(
        reconstructed_views,
        H,
        W
    )

    reconstructed_log = (
        _denormalize_log_sar(
            reconstructed_norm
        )
    )

    reconstructed_linear = np.exp(
        reconstructed_log
    ).astype(
        np.float32
    )

    reconstructed_db = (
        10.0
        *
        np.log10(
            np.maximum(
                reconstructed_linear,
                DPSSD_EVAL_EPS
            )
        )
    ).astype(
        np.float32
    )

    reconstructed_db[
        :,
        ~valid_full
    ] = np.nan

    return reconstructed_db


def evaluate_dpssd_on_frozen_scenes(
    model,
    method_name
):
    records = []

    for _, patch_row in (
        homogeneous_patches.iterrows()
    ):

        scene_id_i = patch_row[
            "scene_id"
        ]

        manifest_row = train_df[
            train_df["scene_id"]
            ==
            scene_id_i
        ].iloc[0]

        sar_path_i = locate_input_file(
            manifest_row[
                "sar_raw_path"
            ]
        )

        with rasterio.open(
            sar_path_i
        ) as src:
            raw_db = src.read().astype(
                np.float32
            )

        start = time.perf_counter()

        filtered_db = (
            dpssd_j_invariant_reconstruct(
                model,
                raw_db,
                folds=DPSSD_EVAL_FOLDS
            )
        )

        runtime_sec = (
            time.perf_counter()
            -
            start
        )

        x0 = int(
            patch_row["x"]
        )

        y0 = int(
            patch_row["y"]
        )

        ps = int(
            patch_row["patch_size"]
        )

        record = {
            "scene_id":
                scene_id_i,

            "category":
                patch_row[
                    "category"
                ],

            "method":
                method_name,

            "runtime_sec_total":
                runtime_sec
        }

        for band_idx, band_name in enumerate(
            ["VV", "VH"]
        ):

            prefix = (
                band_name.lower()
            )

            raw_patch_db = raw_db[
                band_idx,
                y0:y0+ps,
                x0:x0+ps
            ]

            filt_patch_db = filtered_db[
                band_idx,
                y0:y0+ps,
                x0:x0+ps
            ]

            raw_patch_linear = np.power(
                10.0,
                raw_patch_db / 10.0
            )

            filt_patch_linear = np.power(
                10.0,
                filt_patch_db / 10.0
            )

            raw_enl = patch_statistics(
                raw_patch_linear
            )[3]

            filt_enl = patch_statistics(
                filt_patch_linear
            )[3]

            edge_metrics = (
                edge_preservation_metrics(
                    raw_db[
                        band_idx
                    ],
                    filtered_db[
                        band_idx
                    ]
                )
            )

            valid_scene = (
                np.isfinite(
                    raw_db[
                        band_idx
                    ]
                )
                &
                np.isfinite(
                    filtered_db[
                        band_idx
                    ]
                )
            )

            raw_scene_linear = np.power(
                10.0,
                raw_db[
                    band_idx
                ][
                    valid_scene
                ]
                /
                10.0
            )

            filt_scene_linear = np.power(
                10.0,
                filtered_db[
                    band_idx
                ][
                    valid_scene
                ]
                /
                10.0
            )

            intensity_ratio = (
                float(
                    filt_scene_linear.mean()
                )
                /
                float(
                    raw_scene_linear.mean()
                )
            )

            record[
                f"{prefix}_raw_enl"
            ] = raw_enl

            record[
                f"{prefix}_filtered_enl"
            ] = filt_enl

            record[
                f"{prefix}_enl_gain"
            ] = (
                filt_enl
                /
                raw_enl
            )

            record[
                f"{prefix}_egr"
            ] = edge_metrics[
                "egr"
            ]

            record[
                f"{prefix}_egc"
            ] = edge_metrics[
                "egc"
            ]

            record[
                f"{prefix}_intensity_ratio"
            ] = intensity_ratio

        records.append(
            record
        )

    return pd.DataFrame(
        records
    )


# ============================================================
# RUN L0 EVALUATION
# ============================================================

dpssd_L0_results = (
    evaluate_dpssd_on_frozen_scenes(
        dpssd_L0_model,
        "DP-SSD L0"
    )
)

print(
    dpssd_L0_results.to_string(
        index=False
    )
)

print("\nSUMMARY")
print("-" * 88)

for metric in [
    "vv_enl_gain",
    "vh_enl_gain",
    "vv_egr",
    "vh_egr",
    "vv_egc",
    "vh_egc",
    "vv_intensity_ratio",
    "vh_intensity_ratio",
    "runtime_sec_total"
]:
    print(
        f"{metric:24s}: "
        f"{dpssd_L0_results[metric].mean():.6f}"
    )

L0_EVAL_PATH = (
    DPSSD_L0_ROOT
    /
    "dpssd_L0_frozen_scene_evaluation.csv"
)

dpssd_L0_results.to_csv(
    L0_EVAL_PATH,
    index=False
)

assert len(
    dpssd_L0_results
) == len(
    homogeneous_patches
)

assert np.isfinite(
    dpssd_L0_results.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print("\nSaved:", L0_EVAL_PATH)
print("\nP3-C27 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C28
# SECTION: P3-S11 — STRUCTURE-AWARE LOSS V2
#
# PURPOSE:
#   Define the single controlled loss ablation:
#
#       L1 = masked SmoothL1
#            + lambda * smoothed-gradient structure loss
#
# FROZEN:
#   Gaussian kernel = 5x5, sigma=1
#   Sobel normalized by 8
#   structure SmoothL1 beta=0.5
#   lambda=1.0
#
# IMPORTANT:
#   This is a loss ablation only. No architecture change.
# ============================================================

import torch
import torch.nn.functional as F

print("=" * 84)
print("P3-C28 — STRUCTURE-AWARE LOSS V2")
print("=" * 84)

STRUCTURE_LAMBDA = 1.0
STRUCTURE_BETA = 0.5
STRUCTURE_GAUSSIAN_SIZE = 5
STRUCTURE_GAUSSIAN_SIGMA = 1.0


def gaussian_kernel_2d(
    size=5,
    sigma=1.0,
    device=None,
    dtype=torch.float32
):
    coords = torch.arange(
        size,
        device=device,
        dtype=dtype
    ) - (size - 1) / 2.0

    g = torch.exp(
        -(
            coords ** 2
        )
        /
        (
            2.0
            *
            sigma ** 2
        )
    )

    g = (
        g
        /
        g.sum()
    )

    kernel = (
        g[:, None]
        *
        g[None, :]
    )

    return kernel


def gaussian_smooth_channels(
    x,
    size=5,
    sigma=1.0
):
    C = x.shape[1]

    kernel = gaussian_kernel_2d(
        size=size,
        sigma=sigma,
        device=x.device,
        dtype=x.dtype
    )

    kernel = (
        kernel[
            None,
            None
        ]
        .repeat(
            C,
            1,
            1,
            1
        )
    )

    return F.conv2d(
        x,
        kernel,
        padding=size // 2,
        groups=C
    )


def normalized_sobel_gradients(
    x
):
    sobel_x = torch.tensor(
        [
            [-1.0, 0.0, 1.0],
            [-2.0, 0.0, 2.0],
            [-1.0, 0.0, 1.0]
        ],
        device=x.device,
        dtype=x.dtype
    ) / 8.0

    sobel_y = sobel_x.t()

    C = x.shape[1]

    kx = sobel_x[
        None,
        None
    ].repeat(
        C,
        1,
        1,
        1
    )

    ky = sobel_y[
        None,
        None
    ].repeat(
        C,
        1,
        1,
        1
    )

    gx = F.conv2d(
        x,
        kx,
        padding=1,
        groups=C
    )

    gy = F.conv2d(
        x,
        ky,
        padding=1,
        groups=C
    )

    return gx, gy


def masked_structure_loss(
    prediction,
    target,
    mask,
    beta=STRUCTURE_BETA
):
    pred_s = gaussian_smooth_channels(
        prediction,
        size=STRUCTURE_GAUSSIAN_SIZE,
        sigma=STRUCTURE_GAUSSIAN_SIGMA
    )

    targ_s = gaussian_smooth_channels(
        target,
        size=STRUCTURE_GAUSSIAN_SIZE,
        sigma=STRUCTURE_GAUSSIAN_SIGMA
    )

    pgx, pgy = normalized_sobel_gradients(
        pred_s
    )

    tgx, tgy = normalized_sobel_gradients(
        targ_s
    )

    mask_2ch = mask.expand(
        -1,
        prediction.shape[1],
        -1,
        -1
    )

    loss_x = F.smooth_l1_loss(
        pgx,
        tgx,
        reduction="none",
        beta=beta
    )

    loss_y = F.smooth_l1_loss(
        pgy,
        tgy,
        reduction="none",
        beta=beta
    )

    selected = (
        loss_x[
            mask_2ch
        ]
        +
        loss_y[
            mask_2ch
        ]
    )

    if selected.numel() == 0:
        raise RuntimeError(
            "No masked pixels for structure loss."
        )

    return selected.mean()


def dpssd_L1_loss(
    prediction,
    target,
    mask,
    lambda_structure=STRUCTURE_LAMBDA
):
    reconstruction = masked_smooth_l1_loss(
        prediction,
        target,
        mask,
        beta=1.0
    )

    structure = masked_structure_loss(
        prediction,
        target,
        mask,
        beta=STRUCTURE_BETA
    )

    total = (
        reconstruction
        +
        lambda_structure
        *
        structure
    )

    return (
        total,
        reconstruction,
        structure
    )


# ------------------------------------------------------------
# Verify on a real batch
# ------------------------------------------------------------

verify_batch = next(
    iter(
        dpssd_train_loader
    )
)

verify_x = verify_batch[
    "input"
][:4].to(
    device
)

verify_y = verify_batch[
    "target"
][:4].to(
    device
)

verify_m = verify_batch[
    "mask"
][:4].to(
    device
)

verify_model = DPSSDNet().to(
    device
)

verify_prediction = verify_model(
    verify_x
)

(
    verify_total,
    verify_reconstruction,
    verify_structure
) = dpssd_L1_loss(
    verify_prediction,
    verify_y,
    verify_m
)

structure_ratio = float(
    verify_structure.item()
    /
    max(
        verify_reconstruction.item(),
        1e-12
    )
)

print(
    "Reconstruction :",
    float(
        verify_reconstruction.item()
    )
)

print(
    "Structure      :",
    float(
        verify_structure.item()
    )
)

print(
    "Total          :",
    float(
        verify_total.item()
    )
)

print(
    "Structure/recon:",
    f"{100.0 * structure_ratio:.2f}%"
)

# ------------------------------------------------------------
# Offset-invariance sanity:
# gradients should ignore a global constant shift.
# ------------------------------------------------------------

with torch.no_grad():

    base = verify_y[:1]
    offset = base + 0.75

    base_s = gaussian_smooth_channels(
        base
    )

    offset_s = gaussian_smooth_channels(
        offset
    )

    bgx, bgy = normalized_sobel_gradients(
        base_s
    )

    ogx, ogy = normalized_sobel_gradients(
        offset_s
    )

    # Ignore outer 3px where zero-padding can create boundary
    # effects from the added constant.
    interior = (
        slice(None),
        slice(None),
        slice(3, -3),
        slice(3, -3)
    )

    offset_gradient_difference = float(
        (
            (
                bgx[
                    interior
                ]
                -
                ogx[
                    interior
                ]
            ).abs().mean()
            +
            (
                bgy[
                    interior
                ]
                -
                ogy[
                    interior
                ]
            ).abs().mean()
        ).item()
    )

print(
    "Interior gradient difference after constant offset:",
    offset_gradient_difference
)

assert torch.isfinite(
    verify_total
)

assert torch.isfinite(
    verify_structure
)

assert offset_gradient_difference < 1e-5

print("\nP3-C28 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C28B
# SECTION: P3-S11 — CONTROLLED STRUCTURE-LOSS SANITY
#
# PURPOSE:
#   Confirm the frozen structure term behaves in the intended
#   order on controlled synthetic perturbations.
#
# EXPECTED:
#   perfect match ~ 0
#   constant offset ~ 0 (interior gradients)
#   spatial shift > 0
#   blur > 0
# ============================================================

import torch
import torch.nn.functional as F

print("=" * 84)
print("P3-C28B — CONTROLLED STRUCTURE-LOSS SANITY")
print("=" * 84)

torch.manual_seed(2026)

synthetic = torch.rand(
    1,
    2,
    96,
    96,
    device=device
)

synthetic_mask = torch.ones(
    1,
    1,
    96,
    96,
    dtype=torch.bool,
    device=device
)

perfect_loss = masked_structure_loss(
    synthetic,
    synthetic,
    synthetic_mask
)

# Constant offset: evaluate interior to avoid padding effects.
offset_target = synthetic + 0.5

smooth_a = gaussian_smooth_channels(
    synthetic
)

smooth_b = gaussian_smooth_channels(
    offset_target
)

agx, agy = normalized_sobel_gradients(
    smooth_a
)

bgx, bgy = normalized_sobel_gradients(
    smooth_b
)

offset_loss = (
    F.smooth_l1_loss(
        agx[:, :, 3:-3, 3:-3],
        bgx[:, :, 3:-3, 3:-3],
        beta=STRUCTURE_BETA
    )
    +
    F.smooth_l1_loss(
        agy[:, :, 3:-3, 3:-3],
        bgy[:, :, 3:-3, 3:-3],
        beta=STRUCTURE_BETA
    )
)

shifted = torch.roll(
    synthetic,
    shifts=(2, 2),
    dims=(2, 3)
)

shift_loss = masked_structure_loss(
    shifted,
    synthetic,
    synthetic_mask
)

blurred = F.avg_pool2d(
    synthetic,
    kernel_size=5,
    stride=1,
    padding=2
)

blur_loss = masked_structure_loss(
    blurred,
    synthetic,
    synthetic_mask
)

print(
    f"Perfect match : {perfect_loss.item():.8f}"
)

print(
    f"Constant offset: {offset_loss.item():.8f}"
)

print(
    f"Spatial shift : {shift_loss.item():.8f}"
)

print(
    f"Blur          : {blur_loss.item():.8f}"
)

assert perfect_loss.item() < 1e-8
assert offset_loss.item() < 1e-5
assert shift_loss.item() > perfect_loss.item()
assert blur_loss.item() > perfect_loss.item()

print("\nP3-C28B completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C29
# SECTION: P3-S12 — DP-SSD L1 FRESH TRAINING
#
# PURPOSE:
#   Train the single controlled structure-aware DP-SSD loss
#   ablation from scratch.
#
# L1:
#   masked SmoothL1 + 1.0 * structure term
#
# IMPORTANT:
#   Same model, data, optimizer, LR and schedule as L0.
# ============================================================

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch

print("=" * 84)
print("P3-C29 — DP-SSD L1 FRESH TRAINING")
print("=" * 84)

L1_SEED = 42
random.seed(L1_SEED)
np.random.seed(L1_SEED)
torch.manual_seed(L1_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(L1_SEED)

# Reset to exactly the same stochastic crop/mask/shuffle stream
# used by L0 for a controlled loss-only comparison.
dpssd_train_dataset.rng = np.random.default_rng(
    TRAIN_SEED
)
loader_generator.manual_seed(
    TRAIN_SEED
)

L1_MAX_EPOCHS = 60
L1_LR = 3e-4
L1_WEIGHT_DECAY = 1e-5
L1_GRAD_CLIP = 5.0
L1_EARLY_STOP_PATIENCE = 10

DPSSD_L1_ROOT = DPSSD_ROOT / "L1_structure_aware"
DPSSD_L1_ROOT.mkdir(parents=True, exist_ok=True)

L1_BEST_PATH = DPSSD_L1_ROOT / "dpssd_L1_best.pth"
L1_LAST_PATH = DPSSD_L1_ROOT / "dpssd_L1_last.pth"
L1_HISTORY_PATH = DPSSD_L1_ROOT / "dpssd_L1_history.csv"
L1_CONFIG_PATH = DPSSD_L1_ROOT / "dpssd_L1_config.json"

dpssd_L1_model = DPSSDNet().to(
    device
)

optimizer_L1 = torch.optim.AdamW(
    dpssd_L1_model.parameters(),
    lr=L1_LR,
    weight_decay=L1_WEIGHT_DECAY
)

scheduler_L1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_L1,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

use_amp_L1 = bool(
    torch.cuda.is_available()
)

try:
    scaler_L1 = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp_L1
    )
except TypeError:
    scaler_L1 = torch.cuda.amp.GradScaler(
        enabled=use_amp_L1
    )

history_L1 = []
best_val_L1 = float("inf")
best_epoch_L1 = 0
epochs_without_improvement = 0

training_start = time.perf_counter()

for epoch in range(
    1,
    L1_MAX_EPOCHS + 1
):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    dpssd_L1_model.train()

    train_total_losses = []
    train_recon_losses = []
    train_structure_losses = []

    for batch_i in dpssd_train_loader:

        x = batch_i["input"].to(
            device,
            non_blocking=True
        )

        y = batch_i["target"].to(
            device,
            non_blocking=True
        )

        m = batch_i["mask"].to(
            device,
            non_blocking=True
        )

        optimizer_L1.zero_grad(
            set_to_none=True
        )

        if use_amp_L1:
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                pred = dpssd_L1_model(
                    x
                )

                (
                    total_loss,
                    recon_loss,
                    structure_loss
                ) = dpssd_L1_loss(
                    pred,
                    y,
                    m,
                    lambda_structure=STRUCTURE_LAMBDA
                )

            scaler_L1.scale(
                total_loss
            ).backward()

            scaler_L1.unscale_(
                optimizer_L1
            )

            torch.nn.utils.clip_grad_norm_(
                dpssd_L1_model.parameters(),
                L1_GRAD_CLIP
            )

            scaler_L1.step(
                optimizer_L1
            )

            scaler_L1.update()

        else:
            pred = dpssd_L1_model(
                x
            )

            (
                total_loss,
                recon_loss,
                structure_loss
            ) = dpssd_L1_loss(
                pred,
                y,
                m,
                lambda_structure=STRUCTURE_LAMBDA
            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                dpssd_L1_model.parameters(),
                L1_GRAD_CLIP
            )

            optimizer_L1.step()

        train_total_losses.append(
            float(
                total_loss.detach().item()
            )
        )

        train_recon_losses.append(
            float(
                recon_loss.detach().item()
            )
        )

        train_structure_losses.append(
            float(
                structure_loss.detach().item()
            )
        )

    # --------------------------------------------------------
    # INTERNAL VALIDATION — FP32
    # --------------------------------------------------------

    dpssd_L1_model.eval()

    val_total_losses = []
    val_recon_losses = []
    val_structure_losses = []

    with torch.no_grad():

        for batch_i in dpssd_val_loader:

            x = batch_i["input"].to(
                device,
                non_blocking=True
            )

            y = batch_i["target"].to(
                device,
                non_blocking=True
            )

            m = batch_i["mask"].to(
                device,
                non_blocking=True
            )

            pred = dpssd_L1_model(
                x
            )

            (
                total_loss,
                recon_loss,
                structure_loss
            ) = dpssd_L1_loss(
                pred,
                y,
                m,
                lambda_structure=STRUCTURE_LAMBDA
            )

            val_total_losses.append(
                float(total_loss.item())
            )

            val_recon_losses.append(
                float(recon_loss.item())
            )

            val_structure_losses.append(
                float(structure_loss.item())
            )

    mean_train_total = float(
        np.mean(
            train_total_losses
        )
    )

    mean_val_total = float(
        np.mean(
            val_total_losses
        )
    )

    mean_val_recon = float(
        np.mean(
            val_recon_losses
        )
    )

    mean_val_structure = float(
        np.mean(
            val_structure_losses
        )
    )

    current_lr = float(
        optimizer_L1.param_groups[0]["lr"]
    )

    scheduler_L1.step(
        mean_val_total
    )

    history_L1.append({
        "epoch": epoch,
        "train_total": mean_train_total,
        "val_total": mean_val_total,
        "val_reconstruction": mean_val_recon,
        "val_structure": mean_val_structure,
        "lr": current_lr
    })

    improved = (
        mean_val_total
        <
        best_val_L1
        -
        1e-8
    )

    if improved:

        best_val_L1 = mean_val_total
        best_epoch_L1 = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict":
                    dpssd_L1_model.state_dict(),

                "epoch":
                    epoch,

                "val_total":
                    mean_val_total,

                "val_reconstruction":
                    mean_val_recon,

                "val_structure":
                    mean_val_structure,

                "lambda_structure":
                    STRUCTURE_LAMBDA
            },
            L1_BEST_PATH
        )

    else:
        epochs_without_improvement += 1

    if (
        epoch == 1
        or
        epoch % 5 == 0
        or
        epoch == L1_MAX_EPOCHS
    ):
        print(
            f"Epoch {epoch:02d}/{L1_MAX_EPOCHS} | "
            f"train={mean_train_total:.6f} | "
            f"val={mean_val_total:.6f} | "
            f"recon={mean_val_recon:.6f} | "
            f"struct={mean_val_structure:.6f} | "
            f"best={best_val_L1:.6f} @ {best_epoch_L1:02d}"
        )

    if (
        epochs_without_improvement
        >=
        L1_EARLY_STOP_PATIENCE
    ):
        print(
            f"Early stopping at epoch {epoch}."
        )
        break

torch.save(
    {
        "model_state_dict":
            dpssd_L1_model.state_dict(),

        "epoch":
            history_L1[-1]["epoch"],

        "val_total":
            history_L1[-1]["val_total"]
    },
    L1_LAST_PATH
)

history_L1_df = pd.DataFrame(
    history_L1
)

history_L1_df.to_csv(
    L1_HISTORY_PATH,
    index=False
)

L1_CONFIG = {
    "seed": L1_SEED,
    "max_epochs": L1_MAX_EPOCHS,
    "lr": L1_LR,
    "weight_decay": L1_WEIGHT_DECAY,
    "grad_clip": L1_GRAD_CLIP,
    "lambda_structure": STRUCTURE_LAMBDA,
    "structure_beta": STRUCTURE_BETA,
    "gaussian_size": STRUCTURE_GAUSSIAN_SIZE,
    "gaussian_sigma": STRUCTURE_GAUSSIAN_SIGMA,
    "train_pairs": len(dpssd_train_dataset),
    "val_pairs": len(dpssd_val_dataset),
    "canonical_validation_used": False,
    "canonical_test_used": False
}

with open(
    L1_CONFIG_PATH,
    "w"
) as f:
    json.dump(
        L1_CONFIG,
        f,
        indent=2
    )

elapsed_min = (
    time.perf_counter()
    -
    training_start
) / 60.0

best_checkpoint_L1 = torch.load(
    L1_BEST_PATH,
    map_location=device,
    weights_only=False
)

dpssd_L1_model.load_state_dict(
    best_checkpoint_L1[
        "model_state_dict"
    ]
)

dpssd_L1_model.eval()

print("\n" + "-" * 84)
print("DP-SSD L1 TRAINING COMPLETE")
print("-" * 84)
print("Elapsed minutes :", f"{elapsed_min:.2f}")
print("Best epoch      :", best_epoch_L1)
print("Best total val  :", f"{best_val_L1:.10f}")
print(
    "Best val recon  :",
    f"{best_checkpoint_L1['val_reconstruction']:.10f}"
)
print(
    "Best val struct :",
    f"{best_checkpoint_L1['val_structure']:.10f}"
)

assert L1_BEST_PATH.exists()
assert np.isfinite(best_val_L1)

print("\nP3-C29 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C30
# SECTION: P3-S13 — DP-SSD L1 FROZEN-SCENE EVALUATION
#
# PURPOSE:
#   Evaluate the structure-aware L1 loss ablation using the
#   exact same blind/J-invariant reconstruction protocol and
#   frozen scenes used for L0.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 88)
print("P3-C30 — DP-SSD L1 FROZEN-SCENE EVALUATION")
print("=" * 88)

dpssd_L1_results = (
    evaluate_dpssd_on_frozen_scenes(
        dpssd_L1_model,
        "DP-SSD L1"
    )
)

print(
    dpssd_L1_results.to_string(
        index=False
    )
)

print("\nSUMMARY")
print("-" * 88)

for metric in [
    "vv_enl_gain",
    "vh_enl_gain",
    "vv_egr",
    "vh_egr",
    "vv_egc",
    "vh_egc",
    "vv_intensity_ratio",
    "vh_intensity_ratio",
    "runtime_sec_total"
]:
    print(
        f"{metric:24s}: "
        f"{dpssd_L1_results[metric].mean():.6f}"
    )

L1_EVAL_PATH = (
    DPSSD_L1_ROOT
    /
    "dpssd_L1_frozen_scene_evaluation.csv"
)

dpssd_L1_results.to_csv(
    L1_EVAL_PATH,
    index=False
)

# Controlled interpretation: L1 is retained as a loss ablation
# only; it does not trigger more SAR method development.
l0_egr_mean = float(
    (
        dpssd_L0_results[
            "vv_egr"
        ].mean()
        +
        dpssd_L0_results[
            "vh_egr"
        ].mean()
    )
    /
    2.0
)

l1_egr_mean = float(
    (
        dpssd_L1_results[
            "vv_egr"
        ].mean()
        +
        dpssd_L1_results[
            "vh_egr"
        ].mean()
    )
    /
    2.0
)

l0_egc_mean = float(
    (
        dpssd_L0_results[
            "vv_egc"
        ].mean()
        +
        dpssd_L0_results[
            "vh_egc"
        ].mean()
    )
    /
    2.0
)

l1_egc_mean = float(
    (
        dpssd_L1_results[
            "vv_egc"
        ].mean()
        +
        dpssd_L1_results[
            "vh_egc"
        ].mean()
    )
    /
    2.0
)

print("\nL1 vs L0:")
print(
    "  mean EGR delta:",
    l1_egr_mean - l0_egr_mean
)
print(
    "  mean EGC delta:",
    l1_egc_mean - l0_egc_mean
)

assert len(
    dpssd_L1_results
) == len(
    homogeneous_patches
)

assert np.isfinite(
    dpssd_L1_results.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print("\nSaved:", L1_EVAL_PATH)
print(
    "\nL1 retained as controlled preprocessing/loss ablation only."
)
print("No further DP-SSD loss variants will be trained.")
print("\nP3-C30 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C31
# SECTION: P3-S14 — COMPLETE SAR SCREENING TABLE
#
# PURPOSE:
#   Assemble the complete seven-method preprocessing screening
#   table, including the mandatory Raw SAR control.
#
# IMPORTANT:
#   This is screening only. Final downstream performance is
#   decided by standardized segmentation experiments later.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 92)
print("P3-C31 — COMPLETE SAR SCREENING TABLE")
print("=" * 92)

SAR_SCREENING_ROOT = (
    OUTPUT_ROOT
    /
    "sar_screening"
)

SAR_SCREENING_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def _mean_metric(
    frame,
    column
):
    return float(
        frame[
            column
        ].mean()
    )


raw_summary = {
    "method": "Raw SAR",
    "vv_enl_gain": 1.0,
    "vh_enl_gain": 1.0,
    "vv_egr": 1.0,
    "vh_egr": 1.0,
    "vv_egc": 1.0,
    "vh_egc": 1.0,
    "vv_intensity_ratio": 1.0,
    "vh_intensity_ratio": 1.0,
    "runtime_sec_per_scene": 0.0
}

refined_summary_v2 = {
    "method": "Refined Lee",
    "vv_enl_gain":
        _mean_metric(
            refined_lee_results,
            "vv_enl_gain"
        ),
    "vh_enl_gain":
        _mean_metric(
            refined_lee_results,
            "vh_enl_gain"
        ),
    "vv_egr":
        _mean_metric(
            refined_lee_edge_results,
            "vv_egr"
        ),
    "vh_egr":
        _mean_metric(
            refined_lee_edge_results,
            "vh_egr"
        ),
    "vv_egc":
        _mean_metric(
            refined_lee_edge_results,
            "vv_egc"
        ),
    "vh_egc":
        _mean_metric(
            refined_lee_edge_results,
            "vh_egc"
        ),
    "vv_intensity_ratio": np.nan,
    "vh_intensity_ratio": np.nan,
    "runtime_sec_per_scene": np.nan
}

lee_sigma_summary_v2 = {
    "method": "Lee Sigma",
    "vv_enl_gain":
        _mean_metric(
            lee_sigma_results,
            "vv_enl_gain"
        ),
    "vh_enl_gain":
        _mean_metric(
            lee_sigma_results,
            "vh_enl_gain"
        ),
    "vv_egr":
        _mean_metric(
            lee_sigma_results,
            "vv_egr"
        ),
    "vh_egr":
        _mean_metric(
            lee_sigma_results,
            "vh_egr"
        ),
    "vv_egc":
        _mean_metric(
            lee_sigma_results,
            "vv_egc"
        ),
    "vh_egc":
        _mean_metric(
            lee_sigma_results,
            "vh_egc"
        ),
    "vv_intensity_ratio": np.nan,
    "vh_intensity_ratio": np.nan,
    "runtime_sec_per_scene": np.nan
}

sar2sar_summary_v2 = {
    "method": "SAR2SAR",
    "vv_enl_gain":
        _mean_metric(
            sar2sar_results,
            "vv_enl_gain"
        ),
    "vh_enl_gain":
        _mean_metric(
            sar2sar_results,
            "vh_enl_gain"
        ),
    "vv_egr":
        _mean_metric(
            sar2sar_results,
            "vv_egr"
        ),
    "vh_egr":
        _mean_metric(
            sar2sar_results,
            "vh_egr"
        ),
    "vv_egc":
        _mean_metric(
            sar2sar_results,
            "vv_egc"
        ),
    "vh_egc":
        _mean_metric(
            sar2sar_results,
            "vh_egc"
        ),
    "vv_intensity_ratio": np.nan,
    "vh_intensity_ratio": np.nan,
    "runtime_sec_per_scene":
        _mean_metric(
            sar2sar_results,
            "runtime_sec_total"
        )
}

bm3d_summary_v2 = {
    "method": "Homomorphic BM3D",
    "vv_enl_gain":
        _mean_metric(
            bm3d_results,
            "vv_enl_gain"
        ),
    "vh_enl_gain":
        _mean_metric(
            bm3d_results,
            "vh_enl_gain"
        ),
    "vv_egr":
        _mean_metric(
            bm3d_results,
            "vv_egr"
        ),
    "vh_egr":
        _mean_metric(
            bm3d_results,
            "vh_egr"
        ),
    "vv_egc":
        _mean_metric(
            bm3d_results,
            "vv_egc"
        ),
    "vh_egc":
        _mean_metric(
            bm3d_results,
            "vh_egc"
        ),
    "vv_intensity_ratio": np.nan,
    "vh_intensity_ratio": np.nan,
    "runtime_sec_per_scene":
        _mean_metric(
            bm3d_results,
            "runtime_sec_total"
        )
}


def _dpssd_summary(
    frame,
    method
):
    return {
        "method": method,
        "vv_enl_gain":
            _mean_metric(
                frame,
                "vv_enl_gain"
            ),
        "vh_enl_gain":
            _mean_metric(
                frame,
                "vh_enl_gain"
            ),
        "vv_egr":
            _mean_metric(
                frame,
                "vv_egr"
            ),
        "vh_egr":
            _mean_metric(
                frame,
                "vh_egr"
            ),
        "vv_egc":
            _mean_metric(
                frame,
                "vv_egc"
            ),
        "vh_egc":
            _mean_metric(
                frame,
                "vh_egc"
            ),
        "vv_intensity_ratio":
            _mean_metric(
                frame,
                "vv_intensity_ratio"
            ),
        "vh_intensity_ratio":
            _mean_metric(
                frame,
                "vh_intensity_ratio"
            ),
        "runtime_sec_per_scene":
            _mean_metric(
                frame,
                "runtime_sec_total"
            )
    }


dpssd_L0_summary = _dpssd_summary(
    dpssd_L0_results,
    "DP-SSD L0"
)

dpssd_L1_summary = _dpssd_summary(
    dpssd_L1_results,
    "DP-SSD L1"
)


sar_screening_all_methods = pd.DataFrame(
    [
        raw_summary,
        refined_summary_v2,
        lee_sigma_summary_v2,
        sar2sar_summary_v2,
        bm3d_summary_v2,
        dpssd_L0_summary,
        dpssd_L1_summary
    ]
)

print(
    sar_screening_all_methods.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Initial downstream shortlist before L0 calibration
# ------------------------------------------------------------

INITIAL_DOWNSTREAM_SHORTLIST = [
    "Raw SAR",
    "Lee Sigma",
    "Homomorphic BM3D",
    "DP-SSD L0",
    "DP-SSD L1"
]

sar_downstream_shortlist = (
    sar_screening_all_methods[
        sar_screening_all_methods[
            "method"
        ].isin(
            INITIAL_DOWNSTREAM_SHORTLIST
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

SAR_SCREENING_ALL_PATH = (
    SAR_SCREENING_ROOT
    /
    "sar_despeckling_screening_all_methods.csv"
)

SAR_DOWNSTREAM_SHORTLIST_PATH = (
    SAR_SCREENING_ROOT
    /
    "sar_downstream_shortlist.csv"
)

sar_screening_all_methods.to_csv(
    SAR_SCREENING_ALL_PATH,
    index=False
)

sar_downstream_shortlist.to_csv(
    SAR_DOWNSTREAM_SHORTLIST_PATH,
    index=False
)

print("\nSaved:")
print(" ", SAR_SCREENING_ALL_PATH)
print(" ", SAR_DOWNSTREAM_SHORTLIST_PATH)

assert len(
    sar_screening_all_methods
) == 7

assert sar_screening_all_methods[
    "method"
].is_unique

print("\nP3-C31 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C32
# SECTION: P3-S15 — FINAL DP-SSD L0 RADIOMETRIC + RESIDUAL CALIBRATION
#
# PURPOSE:
#   Perform the single pre-registered final calibration of the
#   already-trained L0 DP-SSD output.
#
# PROCEDURE:
#   1) Blind/J-invariant DP-SSD L0 reconstruction.
#   2) Per-scene/per-band linear-intensity mean correction.
#   3) Frozen residual blend:
#
#      I_final = alpha * I_dpssd_corrected
#                + (1-alpha) * I_raw
#
# GRID:
#   alpha = [0.25, 0.50, 0.75, 1.00]
#
# ACCEPTANCE:
#   Must meet/exceed Refined Lee mean VV/VH EGR + EGC.
#   Among passing candidates, maximize mean ENL gain.
#
# IMPORTANT:
#   This is the FINAL SAR method-development experiment.
#   No post-hoc alpha retuning is allowed.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import rasterio

print("=" * 96)
print("P3-C32 — FINAL DP-SSD L0 RADIOMETRIC + RESIDUAL CALIBRATION")
print("=" * 96)

CALIBRATION_ALPHAS = [
    0.25,
    0.50,
    0.75,
    1.00
]

SAR_CALIBRATION_ROOT = (
    DPSSD_ROOT
    /
    "L0_final_calibration"
)

SAR_CALIBRATION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CALIBRATION_PER_SCENE_PATH = (
    SAR_CALIBRATION_ROOT
    /
    "dpssd_calibration_per_scene.csv"
)

CALIBRATION_GRID_PATH = (
    SAR_CALIBRATION_ROOT
    /
    "dpssd_calibration_grid_summary.csv"
)

CALIBRATION_DECISION_PATH = (
    SAR_CALIBRATION_ROOT
    /
    "dpssd_calibration_decision.json"
)

# ------------------------------------------------------------
# Frozen structural thresholds = Refined Lee means
# ------------------------------------------------------------

CALIBRATION_THRESHOLDS = {
    "vv_egr":
        float(
            refined_lee_edge_results[
                "vv_egr"
            ].mean()
        ),

    "vh_egr":
        float(
            refined_lee_edge_results[
                "vh_egr"
            ].mean()
        ),

    "vv_egc":
        float(
            refined_lee_edge_results[
                "vv_egc"
            ].mean()
        ),

    "vh_egc":
        float(
            refined_lee_edge_results[
                "vh_egc"
            ].mean()
        )
}

print("\nStructural acceptance thresholds:")
for k, v in CALIBRATION_THRESHOLDS.items():
    print(
        f"  {k}: {v:.6f}"
    )


def radiometric_mean_correct_linear(
    raw_db,
    dpssd_db
):
    raw_linear = np.power(
        10.0,
        raw_db / 10.0
    ).astype(
        np.float32
    )

    dpssd_linear = np.power(
        10.0,
        dpssd_db / 10.0
    ).astype(
        np.float32
    )

    corrected = (
        dpssd_linear.copy()
    )

    factors = []

    for band_idx in range(2):

        valid = (
            np.isfinite(
                raw_db[
                    band_idx
                ]
            )
            &
            np.isfinite(
                dpssd_db[
                    band_idx
                ]
            )
        )

        raw_mean = float(
            raw_linear[
                band_idx
            ][valid].mean()
        )

        pred_mean = float(
            dpssd_linear[
                band_idx
            ][valid].mean()
        )

        factor = (
            raw_mean
            /
            max(
                pred_mean,
                1e-12
            )
        )

        corrected[
            band_idx,
            valid
        ] *= factor

        corrected[
            band_idx,
            ~valid
        ] = np.nan

        factors.append(
            float(factor)
        )

    return (
        raw_linear,
        corrected,
        factors
    )


calibration_records = []

# ------------------------------------------------------------
# Reconstruct each frozen scene ONCE, then evaluate all alphas
# ------------------------------------------------------------

for _, patch_row in (
    homogeneous_patches.iterrows()
):

    scene_id_i = patch_row[
        "scene_id"
    ]

    manifest_row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row[
            "sar_raw_path"
        ]
    )

    with rasterio.open(
        sar_path_i
    ) as src:
        raw_db = src.read().astype(
            np.float32
        )

    start = time.perf_counter()

    l0_db = dpssd_j_invariant_reconstruct(
        dpssd_L0_model,
        raw_db,
        folds=DPSSD_EVAL_FOLDS
    )

    base_runtime = (
        time.perf_counter()
        -
        start
    )

    (
        raw_linear,
        l0_corrected_linear,
        correction_factors
    ) = radiometric_mean_correct_linear(
        raw_db,
        l0_db
    )

    x0 = int(
        patch_row["x"]
    )

    y0 = int(
        patch_row["y"]
    )

    ps = int(
        patch_row["patch_size"]
    )

    for alpha in (
        CALIBRATION_ALPHAS
    ):

        final_linear = (
            alpha
            *
            l0_corrected_linear
            +
            (
                1.0
                -
                alpha
            )
            *
            raw_linear
        )

        final_db = (
            10.0
            *
            np.log10(
                np.maximum(
                    final_linear,
                    1e-12
                )
            )
        ).astype(
            np.float32
        )

        final_db[
            ~np.isfinite(
                raw_db
            )
        ] = np.nan

        record = {
            "scene_id":
                scene_id_i,

            "category":
                patch_row[
                    "category"
                ],

            "alpha":
                float(alpha),

            "vv_correction_factor":
                correction_factors[0],

            "vh_correction_factor":
                correction_factors[1],

            "runtime_sec_total":
                base_runtime
        }

        for band_idx, band_name in enumerate(
            ["VV", "VH"]
        ):

            prefix = (
                band_name.lower()
            )

            raw_patch = raw_linear[
                band_idx,
                y0:y0+ps,
                x0:x0+ps
            ]

            filt_patch = final_linear[
                band_idx,
                y0:y0+ps,
                x0:x0+ps
            ]

            raw_enl = patch_statistics(
                raw_patch
            )[3]

            filt_enl = patch_statistics(
                filt_patch
            )[3]

            edge_metrics = (
                edge_preservation_metrics(
                    raw_db[
                        band_idx
                    ],
                    final_db[
                        band_idx
                    ]
                )
            )

            valid = (
                np.isfinite(
                    raw_db[
                        band_idx
                    ]
                )
                &
                np.isfinite(
                    final_db[
                        band_idx
                    ]
                )
            )

            intensity_ratio = (
                float(
                    final_linear[
                        band_idx
                    ][valid].mean()
                )
                /
                float(
                    raw_linear[
                        band_idx
                    ][valid].mean()
                )
            )

            record[
                f"{prefix}_enl_gain"
            ] = (
                filt_enl
                /
                raw_enl
            )

            record[
                f"{prefix}_egr"
            ] = edge_metrics[
                "egr"
            ]

            record[
                f"{prefix}_egc"
            ] = edge_metrics[
                "egc"
            ]

            record[
                f"{prefix}_intensity_ratio"
            ] = intensity_ratio

        calibration_records.append(
            record
        )


dpssd_calibration_per_scene = pd.DataFrame(
    calibration_records
)

summary_records = []

for alpha, group in (
    dpssd_calibration_per_scene.groupby(
        "alpha",
        sort=True
    )
):

    summary = {
        "alpha":
            float(alpha),

        "vv_enl_gain":
            float(
                group[
                    "vv_enl_gain"
                ].mean()
            ),

        "vh_enl_gain":
            float(
                group[
                    "vh_enl_gain"
                ].mean()
            ),

        "vv_egr":
            float(
                group[
                    "vv_egr"
                ].mean()
            ),

        "vh_egr":
            float(
                group[
                    "vh_egr"
                ].mean()
            ),

        "vv_egc":
            float(
                group[
                    "vv_egc"
                ].mean()
            ),

        "vh_egc":
            float(
                group[
                    "vh_egc"
                ].mean()
            ),

        "vv_intensity_ratio":
            float(
                group[
                    "vv_intensity_ratio"
                ].mean()
            ),

        "vh_intensity_ratio":
            float(
                group[
                    "vh_intensity_ratio"
                ].mean()
            ),
    }

    summary[
        "passes_structure"
    ] = bool(
        summary[
            "vv_egr"
        ]
        >=
        CALIBRATION_THRESHOLDS[
            "vv_egr"
        ]
        and
        summary[
            "vh_egr"
        ]
        >=
        CALIBRATION_THRESHOLDS[
            "vh_egr"
        ]
        and
        summary[
            "vv_egc"
        ]
        >=
        CALIBRATION_THRESHOLDS[
            "vv_egc"
        ]
        and
        summary[
            "vh_egc"
        ]
        >=
        CALIBRATION_THRESHOLDS[
            "vh_egc"
        ]
    )

    summary[
        "mean_enl_gain"
    ] = float(
        (
            summary[
                "vv_enl_gain"
            ]
            +
            summary[
                "vh_enl_gain"
            ]
        )
        /
        2.0
    )

    summary_records.append(
        summary
    )


dpssd_calibration_grid_summary = (
    pd.DataFrame(
        summary_records
    )
)

print(
    "\n"
    +
    dpssd_calibration_grid_summary.to_string(
        index=False
    )
)

passing = (
    dpssd_calibration_grid_summary[
        dpssd_calibration_grid_summary[
            "passes_structure"
        ]
    ]
)

if len(
    passing
) == 0:

    raise RuntimeError(
        "No calibration alpha passed all pre-registered "
        "Refined-Lee structural thresholds."
    )

selected_row = (
    passing.sort_values(
        [
            "mean_enl_gain",
            "alpha"
        ],
        ascending=[
            False,
            True
        ]
    ).iloc[0]
)

DPSSD_CALIBRATED_ALPHA = float(
    selected_row[
        "alpha"
    ]
)

# This project decision was pre-registered and previously
# validated as alpha = 0.50. A different value on rerun means
# reproducibility must be investigated rather than retuned.
if not np.isclose(
    DPSSD_CALIBRATED_ALPHA,
    0.50
):
    raise RuntimeError(
        "Frozen calibration reproducibility check failed: "
        f"expected alpha=0.50, got {DPSSD_CALIBRATED_ALPHA}. "
        "Do NOT retune; inspect the rerun."
    )

dpssd_calibration_per_scene.to_csv(
    CALIBRATION_PER_SCENE_PATH,
    index=False
)

dpssd_calibration_grid_summary.to_csv(
    CALIBRATION_GRID_PATH,
    index=False
)

calibration_decision = {
    "status":
        "FROZEN",

    "method":
        "DP-SSD L0",

    "radiometric_correction":
        "per-scene per-band linear-intensity mean preservation",

    "blend_equation":
        "I_final = alpha*I_dpssd_corrected + (1-alpha)*I_raw",

    "alpha_grid":
        CALIBRATION_ALPHAS,

    "selected_alpha":
        DPSSD_CALIBRATED_ALPHA,

    "structural_threshold_reference":
        "Refined Lee mean metrics",

    "structural_thresholds":
        CALIBRATION_THRESHOLDS,

    "selection_rule":
        "among candidates passing all 4 structural thresholds, maximize mean VV/VH ENL gain",

    "retuning_allowed":
        False
}

with open(
    CALIBRATION_DECISION_PATH,
    "w"
) as f:
    json.dump(
        calibration_decision,
        f,
        indent=2
    )

print("\nSelected alpha:", DPSSD_CALIBRATED_ALPHA)
print("\nSaved:")
print(" ", CALIBRATION_PER_SCENE_PATH)
print(" ", CALIBRATION_GRID_PATH)
print(" ", CALIBRATION_DECISION_PATH)

print(
    "\nP3-C32 completed successfully ✅"
)
print(
    "SAR method development is now CLOSED."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C33
# SECTION: P3-S16 — FINAL SAR PREPROCESSING LOCK
#
# PURPOSE:
#   Freeze the SAR preprocessing registry used by every later
#   FloodShift segmentation experiment.
#
# FINAL DOWNSTREAM CANDIDATES:
#   1) Raw SAR
#   2) Lee Sigma
#   3) Homomorphic BM3D
#   4) DP-SSD L0
#   5) DP-SSD Calibrated (alpha=0.50)
#
# SCREENING/ABLATION ONLY:
#   Refined Lee, SAR2SAR, DP-SSD L1
#
# IMPORTANT:
#   No final SAR winner is selected here.
#   Downstream segmentation decides comparative utility.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

print("=" * 96)
print("P3-C33 — FINAL SAR PREPROCESSING LOCK")
print("=" * 96)

SAR_FINAL_ROOT = (
    OUTPUT_ROOT
    /
    "sar_final"
)

SAR_FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SAR_FINAL_SHORTLIST_PATH = (
    SAR_FINAL_ROOT
    /
    "sar_downstream_shortlist_FINAL.csv"
)

SAR_ABLATION_ONLY_PATH = (
    SAR_FINAL_ROOT
    /
    "sar_preprocessing_ablation_only.csv"
)

SAR_PREPROCESSING_LOCK_PATH = (
    SAR_FINAL_ROOT
    /
    "sar_preprocessing_lock.json"
)

DPSSD_CALIBRATED_RECIPE_PATH = (
    SAR_FINAL_ROOT
    /
    "dpssd_calibrated_recipe.json"
)

# ------------------------------------------------------------
# Build calibrated summary from C32
# ------------------------------------------------------------

calibrated_row = (
    dpssd_calibration_grid_summary[
        np.isclose(
            dpssd_calibration_grid_summary[
                "alpha"
            ],
            DPSSD_CALIBRATED_ALPHA
        )
    ].iloc[0]
)

final_shortlist_records = [
    {
        "method":
            "Raw SAR",

        "role":
            "mandatory raw control",

        "alpha":
            np.nan
    },

    {
        "method":
            "Lee Sigma",

        "role":
            "classical edge-preserving comparator",

        "alpha":
            np.nan
    },

    {
        "method":
            "Homomorphic BM3D",

        "role":
            "non-local comparator",

        "alpha":
            np.nan
    },

    {
        "method":
            "DP-SSD L0",

        "role":
            "proposed uncalibrated learned candidate",

        "alpha":
            1.0
    },

    {
        "method":
            "DP-SSD Calibrated",

        "role":
            "proposed operational calibrated candidate",

        "alpha":
            float(
                DPSSD_CALIBRATED_ALPHA
            )
    }
]

sar_downstream_shortlist_FINAL = pd.DataFrame(
    final_shortlist_records
)

# Attach available screening metrics.
metric_lookup = (
    sar_screening_all_methods
    .set_index(
        "method"
    )
)

for column in [
    "vv_enl_gain",
    "vh_enl_gain",
    "vv_egr",
    "vh_egr",
    "vv_egc",
    "vh_egc",
    "vv_intensity_ratio",
    "vh_intensity_ratio",
    "runtime_sec_per_scene"
]:
    values = []

    for method in (
        sar_downstream_shortlist_FINAL[
            "method"
        ]
    ):

        if method == "DP-SSD Calibrated":
            source_col = (
                "runtime_sec_per_scene"
                if column
                ==
                "runtime_sec_per_scene"
                else column
            )

            if source_col == "runtime_sec_per_scene":
                value = float(
                    dpssd_L0_results[
                        "runtime_sec_total"
                    ].mean()
                )
            else:
                value = float(
                    calibrated_row[
                        source_col
                    ]
                )

        elif method in metric_lookup.index:
            value = metric_lookup.loc[
                method,
                column
            ]

        else:
            value = np.nan

        values.append(
            value
        )

    sar_downstream_shortlist_FINAL[
        column
    ] = values


sar_preprocessing_ablation_only = pd.DataFrame(
    [
        {
            "method":
                "Refined Lee",

            "reason":
                "screening comparator; not retained in downstream shortlist"
        },

        {
            "method":
                "SAR2SAR",

            "reason":
                "strong smoothing but poor structural metrics in screening"
        },

        {
            "method":
                "DP-SSD L1",

            "reason":
                "controlled loss ablation; did not repair structure sufficiently"
        }
    ]
)

sar_downstream_shortlist_FINAL.to_csv(
    SAR_FINAL_SHORTLIST_PATH,
    index=False
)

sar_preprocessing_ablation_only.to_csv(
    SAR_ABLATION_ONLY_PATH,
    index=False
)

dpssd_calibrated_recipe = {
    "status":
        "FROZEN",

    "base_model":
        "DP-SSD L0",

    "checkpoint":
        str(
            L0_BEST_PATH
        ),

    "inference":
        "10-fold deterministic J-invariant reconstruction",

    "radiometric_correction":
        "per-scene per-band linear-intensity mean preservation",

    "alpha":
        float(
            DPSSD_CALIBRATED_ALPHA
        ),

    "blend_equation":
        "I_final = 0.5*I_dpssd_corrected + 0.5*I_raw",

    "retuning_allowed":
        False
}

with open(
    DPSSD_CALIBRATED_RECIPE_PATH,
    "w"
) as f:
    json.dump(
        dpssd_calibrated_recipe,
        f,
        indent=2
    )

sar_preprocessing_lock = {
    "status":
        "PERMANENTLY_FROZEN",

    "method_development_closed":
        True,

    "channels":
        [
            "VV",
            "VH"
        ],

    "downstream_candidates":
        sar_downstream_shortlist_FINAL[
            "method"
        ].tolist(),

    "proposed_operational_candidate":
        "DP-SSD Calibrated",

    "dpssd_calibrated_alpha":
        float(
            DPSSD_CALIBRATED_ALPHA
        ),

    "final_winner_selected":
        False,

    "winner_selection":
        "standardized downstream segmentation benchmark",

    "ablation_only":
        sar_preprocessing_ablation_only[
            "method"
        ].tolist(),

    "future_rules": {
        "retrain_dpssd":
            False,

        "change_dpssd_loss":
            False,

        "retune_alpha":
            False,

        "add_new_sar_filter":
            False,

        "execute_frozen_recipes":
            True,

        "cache_frozen_outputs":
            True
    }
}

with open(
    SAR_PREPROCESSING_LOCK_PATH,
    "w"
) as f:
    json.dump(
        sar_preprocessing_lock,
        f,
        indent=2
    )

print(
    sar_downstream_shortlist_FINAL.to_string(
        index=False
    )
)

print("\nSaved:")
print(" ", SAR_FINAL_SHORTLIST_PATH)
print(" ", SAR_ABLATION_ONLY_PATH)
print(" ", SAR_PREPROCESSING_LOCK_PATH)
print(" ", DPSSD_CALIBRATED_RECIPE_PATH)

assert list(
    sar_downstream_shortlist_FINAL[
        "method"
    ]
) == [
    "Raw SAR",
    "Lee Sigma",
    "Homomorphic BM3D",
    "DP-SSD L0",
    "DP-SSD Calibrated"
]

assert np.isclose(
    sar_downstream_shortlist_FINAL.loc[
        sar_downstream_shortlist_FINAL[
            "method"
        ]
        ==
        "DP-SSD Calibrated",
        "alpha"
    ].iloc[0],
    0.50
)

print("\nP3-C33 completed successfully ✅")
print(
    "SAR PREPROCESSING DEVELOPMENT IS PERMANENTLY CLOSED."
)


## 6. Sentinel-2 optical preprocessing


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C34
# SECTION: P3-S17 — RAW SENTINEL-2 SAMPLE AUDIT
#
# PURPOSE:
#   Inspect one canonical Sentinel-2 chip before designing
#   optical preprocessing.
# ============================================================

from pathlib import Path
import numpy as np
import rasterio

print("=" * 100)
print("P3-C34 — RAW SENTINEL-2 SAMPLE AUDIT")
print("=" * 100)

OPTICAL_ROOT = (
    OUTPUT_ROOT
    /
    "optical"
)

OPTICAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXPECTED_S2_BANDS = [
    "B1", "B2", "B3", "B4", "B5", "B6", "B7",
    "B8", "B8A", "B9", "B10", "B11", "B12"
]

candidate_columns = [
    "optical_raw_path",
    "optical_path",
    "s2_raw_path",
    "s2_path"
]

OPTICAL_PATH_COLUMN = next(
    (
        column
        for column in candidate_columns
        if column in train_df.columns
    ),
    None
)

if OPTICAL_PATH_COLUMN is None:
    raise KeyError(
        "Could not find an optical path column in train_df."
    )

optical_sample_row = train_df.iloc[0]
optical_scene_id = optical_sample_row[
    "scene_id"
]

optical_sample_path = locate_input_file(
    optical_sample_row[
        OPTICAL_PATH_COLUMN
    ]
)

with rasterio.open(
    optical_sample_path
) as src:

    raw_optical = src.read().astype(
        np.float32
    )

    optical_descriptions = list(
        src.descriptions
    )

    optical_nodata = src.nodata

    optical_crs = src.crs

    optical_dataset_mask = src.dataset_mask()

print("Scene             :", optical_scene_id)
print("Path              :", optical_sample_path)
print("Shape             :", raw_optical.shape)
print("CRS               :", optical_crs)
print("Nodata            :", optical_nodata)
print("Band descriptions :", optical_descriptions)

assert raw_optical.shape == (
    13,
    512,
    512
)

assert optical_descriptions == (
    EXPECTED_S2_BANDS
)

print("\nPer-band raw audit:")

for idx, band_name in enumerate(
    EXPECTED_S2_BANDS
):
    band = raw_optical[idx]

    finite = np.isfinite(
        band
    )

    zero_fraction = float(
        np.mean(
            band == 0
        )
    )

    negative_fraction = float(
        np.mean(
            band < 0
        )
    )

    print(
        f"{band_name:4s} | "
        f"finite={100.0*finite.mean():6.2f}% | "
        f"zero={100.0*zero_fraction:6.2f}% | "
        f"negative={100.0*negative_fraction:6.2f}% | "
        f"min={np.nanmin(band):.1f} | "
        f"max={np.nanmax(band):.1f}"
    )

global_values = raw_optical[
    np.isfinite(
        raw_optical
    )
]

print("\nGlobal raw distribution:")
print("  median :", float(np.median(global_values)))
print("  p99    :", float(np.percentile(global_values, 99)))
print("  maximum:", float(np.max(global_values)))

unique_mask, mask_counts = np.unique(
    optical_dataset_mask,
    return_counts=True
)

print("\nRasterio dataset mask:")
for value, count in zip(
    unique_mask,
    mask_counts
):
    print(
        f"  {int(value)} -> {int(count):,}"
    )

print(
    "\nP3-C34 completed successfully ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C35
# SECTION: P3-S18 — FULL TRAIN SENTINEL-2 AUDIT
#
# PURPOSE:
#   Audit every canonical TRAIN Sentinel-2 scene, quantify
#   strict validity, and estimate train-only raw distributions.
#
# IMPORTANT:
#   - no validation/test used
#   - no clipping
#   - no cloud method selected here
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

print("=" * 104)
print("P3-C35 — FULL TRAIN SENTINEL-2 AUDIT")
print("=" * 104)

MAX_SAMPLES_PER_SCENE = 50000
OPTICAL_SAMPLE_SEED = 42
rng_optical = np.random.default_rng(
    OPTICAL_SAMPLE_SEED
)

scene_audit_records = []
band_valid_counts = np.zeros(
    13,
    dtype=np.int64
)
band_total_counts = np.zeros(
    13,
    dtype=np.int64
)
sample_chunks = {
    band_name: []
    for band_name in EXPECTED_S2_BANDS
}

dataset_mask_matches_all = 0
dataset_mask_matches_any = 0

for position, (_, row) in enumerate(
    train_df.iterrows(),
    start=1
):

    scene_id_i = row[
        "scene_id"
    ]

    optical_path_i = locate_input_file(
        row[
            OPTICAL_PATH_COLUMN
        ]
    )

    with rasterio.open(
        optical_path_i
    ) as src:

        raw = src.read().astype(
            np.float32
        )

        descriptions = list(
            src.descriptions
        )

        nodata = src.nodata

        dataset_mask = (
            src.dataset_mask()
            >
            0
        )

    assert raw.shape == (
        13,
        512,
        512
    )

    if descriptions != EXPECTED_S2_BANDS:
        raise RuntimeError(
            f"{scene_id_i}: unexpected Sentinel-2 band order."
        )

    if nodata != 0:
        raise RuntimeError(
            f"{scene_id_i}: expected nodata=0, got {nodata}."
        )

    finite_mask = np.isfinite(
        raw
    )

    band_valid = (
        finite_mask
        &
        (
            raw != 0
        )
    )

    all_band_valid = np.all(
        band_valid,
        axis=0
    )

    any_band_valid = np.any(
        band_valid,
        axis=0
    )

    rasterio_valid = dataset_mask

    if np.array_equal(
        rasterio_valid,
        all_band_valid
    ):
        dataset_mask_matches_all += 1

    if np.array_equal(
        rasterio_valid,
        any_band_valid
    ):
        dataset_mask_matches_any += 1

    band_valid_counts += band_valid.reshape(
        13,
        -1
    ).sum(
        axis=1
    )

    band_total_counts += (
        raw.shape[1]
        *
        raw.shape[2]
    )

    valid_positions = np.flatnonzero(
        all_band_valid.ravel()
    )

    if len(valid_positions) > 0:

        if (
            len(valid_positions)
            >
            MAX_SAMPLES_PER_SCENE
        ):
            selected_positions = rng_optical.choice(
                valid_positions,
                size=MAX_SAMPLES_PER_SCENE,
                replace=False
            )
        else:
            selected_positions = valid_positions

        raw_flat = raw.reshape(
            13,
            -1
        )

        for band_idx, band_name in enumerate(
            EXPECTED_S2_BANDS
        ):
            sample_chunks[
                band_name
            ].append(
                raw_flat[
                    band_idx,
                    selected_positions
                ].astype(
                    np.float32
                )
            )

    disagreement = float(
        100.0
        *
        np.mean(
            any_band_valid
            !=
            all_band_valid
        )
    )

    scene_audit_records.append({
        "scene_id":
            scene_id_i,

        "region":
            row[
                "region"
            ],

        "all_band_valid_pct":
            float(
                100.0
                *
                all_band_valid.mean()
            ),

        "any_band_valid_pct":
            float(
                100.0
                *
                any_band_valid.mean()
            ),

        "rasterio_valid_pct":
            float(
                100.0
                *
                rasterio_valid.mean()
            ),

        "band_validity_disagreement_pct":
            disagreement,

        "nonfinite_pixels":
            int(
                np.size(raw)
                -
                np.isfinite(raw).sum()
            ),

        "negative_pixels":
            int(
                np.sum(
                    raw < 0
                )
            )
    })

    if (
        position == 1
        or
        position % 50 == 0
        or
        position == len(train_df)
    ):
        print(
            f"[{position:03d}/{len(train_df)}] "
            f"{scene_id_i:24s} | "
            f"ALL={100*all_band_valid.mean():6.2f}% | "
            f"ANY={100*any_band_valid.mean():6.2f}%"
        )

optical_train_scene_audit = pd.DataFrame(
    scene_audit_records
)

coverage_records = []

for band_idx, band_name in enumerate(
    EXPECTED_S2_BANDS
):
    coverage_records.append({
        "band_name":
            band_name,

        "valid_pixels":
            int(
                band_valid_counts[
                    band_idx
                ]
            ),

        "total_pixels":
            int(
                band_total_counts[
                    band_idx
                ]
            ),

        "valid_pct":
            float(
                100.0
                *
                band_valid_counts[
                    band_idx
                ]
                /
                band_total_counts[
                    band_idx
                ]
            )
    })

optical_train_band_coverage = pd.DataFrame(
    coverage_records
)

distribution_records = []

for band_name in EXPECTED_S2_BANDS:

    values = np.concatenate(
        sample_chunks[
            band_name
        ]
    ).astype(
        np.float64
    )

    p1, p50, p95, p99, p999 = np.percentile(
        values,
        [
            1,
            50,
            95,
            99,
            99.9
        ]
    )

    distribution_records.append({
        "band_name":
            band_name,

        "sample_count":
            int(
                values.size
            ),

        "minimum":
            float(
                values.min()
            ),

        "maximum":
            float(
                values.max()
            ),

        "mean":
            float(
                values.mean()
            ),

        "std":
            float(
                values.std()
            ),

        "p1":
            float(p1),

        "p50":
            float(p50),

        "p95":
            float(p95),

        "p99":
            float(p99),

        "p99_9":
            float(p999)
    })

optical_train_distribution = pd.DataFrame(
    distribution_records
)

print("\n" + "=" * 104)
print("TRAIN SCENE VALIDITY SUMMARY")
print("=" * 104)

print(
    optical_train_scene_audit[
        [
            "all_band_valid_pct",
            "any_band_valid_pct",
            "rasterio_valid_pct",
            "band_validity_disagreement_pct"
        ]
    ].describe().to_string()
)

print(
    "\nRasterio mask matches ALL-band mask:",
    dataset_mask_matches_all,
    "/",
    len(train_df)
)

print(
    "Rasterio mask matches ANY-band mask:",
    dataset_mask_matches_any,
    "/",
    len(train_df)
)

print("\nBand coverage:")
print(
    optical_train_band_coverage.to_string(
        index=False
    )
)

print("\nValid-only sampled raw distribution:")
print(
    optical_train_distribution.to_string(
        index=False
    )
)

OPTICAL_SCENE_AUDIT_PATH = (
    OPTICAL_ROOT
    /
    "optical_train_scene_audit.csv"
)

OPTICAL_BAND_COVERAGE_PATH = (
    OPTICAL_ROOT
    /
    "optical_train_band_coverage.csv"
)

OPTICAL_RAW_DISTRIBUTION_PATH = (
    OPTICAL_ROOT
    /
    "optical_train_valid_distribution.csv"
)

optical_train_scene_audit.to_csv(
    OPTICAL_SCENE_AUDIT_PATH,
    index=False
)

optical_train_band_coverage.to_csv(
    OPTICAL_BAND_COVERAGE_PATH,
    index=False
)

optical_train_distribution.to_csv(
    OPTICAL_RAW_DISTRIBUTION_PATH,
    index=False
)

assert len(
    optical_train_scene_audit
) == 314

assert (
    optical_train_scene_audit[
        "nonfinite_pixels"
    ].sum()
    ==
    0
)

assert (
    optical_train_scene_audit[
        "negative_pixels"
    ].sum()
    ==
    0
)

print("\nSaved:")
print(" ", OPTICAL_SCENE_AUDIT_PATH)
print(" ", OPTICAL_BAND_COVERAGE_PATH)
print(" ", OPTICAL_RAW_DISTRIBUTION_PATH)

print("\nP3-C35 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C36
# SECTION: P3-S19 — OPTICAL SCALING + VALIDITY POLICY LOCK
#
# PURPOSE:
#   Permanently freeze numerical Sentinel-2 preprocessing.
#
# POLICY:
#   scale      = raw / 10000
#   nodata     = 0
#   validity   = all 13 bands finite + nonzero
#   clipping   = none
#   bands      = all 13 preserved at source stage
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import rasterio

print("=" * 100)
print("P3-C36 — OPTICAL SCALING + VALIDITY POLICY LOCK")
print("=" * 100)

OPTICAL_SCALE_FACTOR = 10000.0
OPTICAL_NODATA_VALUE = 0.0
OPTICAL_BAND_NAMES = EXPECTED_S2_BANDS.copy()
OPTICAL_VALIDITY_RULE = (
    "finite_and_nonzero_in_all_13_bands"
)

print("Scale factor   :", OPTICAL_SCALE_FACTOR)
print("Raw nodata     :", OPTICAL_NODATA_VALUE)
print("Validity rule  :", OPTICAL_VALIDITY_RULE)
print("Clipping       :", False)

assert len(
    optical_train_scene_audit
) == 314

assert list(
    optical_train_distribution[
        "band_name"
    ]
) == EXPECTED_S2_BANDS

print("\nC35 consistency checks passed ✅")

print("\nRasterio dataset mask:")
print(
    "  matches ANY-band mask:",
    dataset_mask_matches_any,
    "/ 314"
)
print(
    "  matches ALL-band mask:",
    dataset_mask_matches_all,
    "/ 314"
)

print("\nDecision:")
print(
    "  DO NOT use dataset_mask as the model-validity mask."
)
print(
    "  Use ALL-13-band nonzero validity instead."
)


def load_optical_reflectance(
    row
):
    optical_path = locate_input_file(
        row[
            OPTICAL_PATH_COLUMN
        ]
    )

    with rasterio.open(
        optical_path
    ) as src:

        raw = src.read().astype(
            np.float32
        )

        descriptions = list(
            src.descriptions
        )

        nodata = src.nodata

    if raw.shape != (
        13,
        512,
        512
    ):
        raise RuntimeError(
            f"Unexpected optical shape: {raw.shape}"
        )

    if descriptions != OPTICAL_BAND_NAMES:
        raise RuntimeError(
            "Unexpected Sentinel-2 band order."
        )

    if nodata != OPTICAL_NODATA_VALUE:
        raise RuntimeError(
            f"Expected nodata=0, got {nodata}."
        )

    valid_mask = np.all(
        np.isfinite(
            raw
        )
        &
        (
            raw
            !=
            OPTICAL_NODATA_VALUE
        ),
        axis=0
    )

    reflectance = (
        raw
        /
        OPTICAL_SCALE_FACTOR
    ).astype(
        np.float32
    )

    return (
        reflectance,
        valid_mask
    )


sample_reflectance, sample_valid = (
    load_optical_reflectance(
        optical_sample_row
    )
)

print("\nSample scene:", optical_scene_id)
print(
    "Reflectance shape:",
    sample_reflectance.shape
)
print(
    "Reflectance dtype:",
    sample_reflectance.dtype
)
print(
    "Strict valid pixels:",
    f"{int(sample_valid.sum()):,}",
    f"({100.0*sample_valid.mean():.2f}%)"
)

sample_values = (
    sample_reflectance[
        :,
        sample_valid
    ]
)

print("\nSample valid reflectance:")
print(
    "  min:",
    float(
        sample_values.min()
    )
)
print(
    "  max:",
    float(
        sample_values.max()
    )
)
print(
    "  mean:",
    float(
        sample_values.mean()
    )
)

optical_train_scaled_distribution = (
    optical_train_distribution.copy()
)

for column in [
    "minimum",
    "maximum",
    "mean",
    "std",
    "p1",
    "p50",
    "p95",
    "p99",
    "p99_9"
]:
    optical_train_scaled_distribution[
        column
    ] = (
        optical_train_scaled_distribution[
            column
        ]
        /
        OPTICAL_SCALE_FACTOR
    )

print("\n" + "=" * 100)
print("TRAIN VALID-PIXEL REFLECTANCE DISTRIBUTION")
print("=" * 100)

print(
    optical_train_scaled_distribution.to_string(
        index=False
    )
)

above_one = (
    optical_train_scaled_distribution[
        optical_train_scaled_distribution[
            "maximum"
        ]
        >
        1.0
    ][
        [
            "band_name",
            "maximum"
        ]
    ]
)

print(
    "\nBands with observed valid values > 1.0:"
)
print(
    above_one.to_string(
        index=False
    )
)

print("\nPolicy:")
print(
    "  Values > 1.0 are PRESERVED."
)
print(
    "  No reflectance clipping is applied."
)

b10_row = (
    optical_train_scaled_distribution[
        optical_train_scaled_distribution[
            "band_name"
        ]
        ==
        "B10"
    ].iloc[0]
)

print("\n" + "=" * 100)
print("B10 AFTER SCALING")
print("=" * 100)
print("median :", b10_row["p50"])
print("p95    :", b10_row["p95"])
print("p99    :", b10_row["p99"])
print("maximum:", b10_row["maximum"])
print("\nB10 remains in the 13-band stack.")

validity_cost = (
    optical_train_scene_audit[
        "any_band_valid_pct"
    ]
    -
    optical_train_scene_audit[
        "all_band_valid_pct"
    ]
)

print("\n" + "=" * 100)
print("STRICT ALL-BAND VALIDITY COST")
print("=" * 100)
print(
    "Mean coverage difference:",
    f"{validity_cost.mean():.4f}%"
)
print(
    "Maximum difference:",
    f"{validity_cost.max():.4f}%"
)
print(
    "Scenes affected:",
    int(
        (
            validity_cost > 0
        ).sum()
    ),
    "/ 314"
)

OPTICAL_POLICY_PATH = (
    OPTICAL_ROOT
    /
    "optical_numerical_preprocessing_policy_v1.json"
)

OPTICAL_REFLECTANCE_DISTRIBUTION_PATH = (
    OPTICAL_ROOT
    /
    "optical_train_reflectance_distribution.csv"
)

optical_numerical_policy = {
    "status":
        "FROZEN",

    "scale_factor":
        OPTICAL_SCALE_FACTOR,

    "raw_nodata":
        OPTICAL_NODATA_VALUE,

    "validity_rule":
        OPTICAL_VALIDITY_RULE,

    "clipping":
        False,

    "source_band_count":
        13,

    "source_band_order":
        OPTICAL_BAND_NAMES,

    "dataset_mask_used_for_model_validity":
        False,

    "cloud_shadow_policy":
        "NOT YET LOCKED",

    "normalization":
        "NOT YET LOCKED"
}

with open(
    OPTICAL_POLICY_PATH,
    "w"
) as f:
    json.dump(
        optical_numerical_policy,
        f,
        indent=2
    )

optical_train_scaled_distribution.to_csv(
    OPTICAL_REFLECTANCE_DISTRIBUTION_PATH,
    index=False
)

print("\nSaved:")
print(" ", OPTICAL_POLICY_PATH)
print(" ", OPTICAL_REFLECTANCE_DISTRIBUTION_PATH)

print("\nP3-C36 completed successfully ✅")
print(
    "Sentinel-2 numerical preprocessing is now LOCKED."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C37
# SECTION: P3-S20 — OMNICLOUDMASK SINGLE-SCENE SANITY CHECK
#
# PURPOSE:
#   Verify pretrained cloud/cloud-shadow inference on one
#   canonical Sentinel-2 TRAIN chip.
#
# INPUT:
#   B4=Red, B3=Green, B8=NIR
#
# OUTPUT:
#   classes:
#     0 clear
#     1 thick cloud
#     2 thin cloud
#     3 cloud shadow
#
# FLOODSHIFT RELIABILITY CANDIDATE:
#   P(clear), invalid pixels forced to 0.
# ============================================================

import sys
import subprocess
import importlib
import time
import json

import numpy as np
import torch

print("=" * 100)
print("P3-C37 — OMNICLOUDMASK SINGLE-SCENE SANITY CHECK")
print("=" * 100)

try:
    import omnicloudmask
except ImportError:
    print("OmniCloudMask not found.")
    print("Installing...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "omnicloudmask"
        ]
    )

    importlib.invalidate_caches()
    import omnicloudmask

from omnicloudmask import predict_from_array

try:
    OCM_VERSION = omnicloudmask.__version__
except Exception:
    OCM_VERSION = "unknown"

OCM_CLASS_NAMES = [
    "clear",
    "thick_cloud",
    "thin_cloud",
    "cloud_shadow"
]

print("OmniCloudMask imported ✅")
print("Version:", OCM_VERSION)

for class_id, class_name in enumerate(
    OCM_CLASS_NAMES
):
    print(
        f"  {class_id} -> {class_name}"
    )

S2_INDEX = {
    band_name: index
    for index, band_name in enumerate(
        OPTICAL_BAND_NAMES
    )
}

sample_reflectance, sample_valid = (
    load_optical_reflectance(
        optical_sample_row
    )
)

ocm_input = np.stack(
    [
        sample_reflectance[
            S2_INDEX["B4"]
        ],
        sample_reflectance[
            S2_INDEX["B3"]
        ],
        sample_reflectance[
            S2_INDEX["B8"]
        ]
    ],
    axis=0
).astype(
    np.float32
)

ocm_input[
    :,
    ~sample_valid
] = 0.0

OCM_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nInference device:", OCM_DEVICE)

start = time.perf_counter()

ocm_probability = predict_from_array(
    input_array=ocm_input,
    inference_device=OCM_DEVICE,
    inference_dtype="fp32",
    export_confidence=True,
    softmax_output=True,
    no_data_value=0,
    apply_no_data_mask=True,
    pred_classes=4
)

ocm_runtime = (
    time.perf_counter()
    -
    start
)

ocm_probability = np.asarray(
    ocm_probability,
    dtype=np.float32
)

if (
    ocm_probability.ndim == 4
    and
    ocm_probability.shape[0] == 1
):
    ocm_probability = (
        ocm_probability[0]
    )

assert ocm_probability.shape == (
    4,
    512,
    512
)

# OmniCloudMask applies a tiny confidence floor, so normalize
# the four channels back to an exact probability simplex.
valid_prob = (
    ocm_probability[
        :,
        sample_valid
    ]
)

prob_sum = np.maximum(
    valid_prob.sum(
        axis=0,
        keepdims=True
    ),
    1e-8
)

valid_prob = (
    valid_prob
    /
    prob_sum
)

normalized_probability = np.zeros_like(
    ocm_probability,
    dtype=np.float32
)

normalized_probability[
    :,
    sample_valid
] = valid_prob

sample_optical_reliability = (
    normalized_probability[
        0
    ].copy()
)

sample_optical_reliability[
    ~sample_valid
] = 0.0

ocm_class_map = np.argmax(
    normalized_probability,
    axis=0
).astype(
    np.uint8
)

valid_classes = (
    ocm_class_map[
        sample_valid
    ]
)

counts = np.bincount(
    valid_classes,
    minlength=4
)

total_valid = int(
    sample_valid.sum()
)

print("\nInference runtime:", f"{ocm_runtime:.3f} sec")
print(
    "Probability shape:",
    normalized_probability.shape
)

print("\nVALID-PIXEL CLASS DISTRIBUTION")

for class_id, class_name in enumerate(
    OCM_CLASS_NAMES
):
    pct = (
        100.0
        *
        counts[
            class_id
        ]
        /
        total_valid
    )

    print(
        f"{class_id} {class_name:14s}: "
        f"{counts[class_id]:8,d} "
        f"({pct:6.2f}%)"
    )

valid_reliability = (
    sample_optical_reliability[
        sample_valid
    ]
)

percentiles = np.percentile(
    valid_reliability,
    [
        0,
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99,
        100
    ]
)

print("\nP(CLEAR) RELIABILITY DISTRIBUTION")

for p, value in zip(
    [
        0,
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99,
        100
    ],
    percentiles
):
    print(
        f"p{p:3d}: {value:.6f}"
    )

print(
    "\nMean P(clear):",
    float(
        valid_reliability.mean()
    )
)

OCM_ROOT = (
    OPTICAL_ROOT
    /
    "cloud_reliability"
)

OCM_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

OCM_SAMPLE_SUMMARY_PATH = (
    OCM_ROOT
    /
    "omnicloudmask_sample_summary.json"
)

with open(
    OCM_SAMPLE_SUMMARY_PATH,
    "w"
) as f:
    json.dump(
        {
            "scene_id":
                str(
                    optical_scene_id
                ),

            "model":
                "OmniCloudMask",

            "version":
                str(
                    OCM_VERSION
                ),

            "input_bands":
                [
                    "B4",
                    "B3",
                    "B8"
                ],

            "classes":
                OCM_CLASS_NAMES,

            "reliability_definition":
                "renormalized_P_clear",

            "strict_optical_validity":
                OPTICAL_VALIDITY_RULE,

            "mean_p_clear":
                float(
                    valid_reliability.mean()
                ),

            "runtime_sec":
                float(
                    ocm_runtime
                )
        },
        f,
        indent=2
    )

print("\nSaved:", OCM_SAMPLE_SUMMARY_PATH)
print("\nP3-C37 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C38
# SECTION: P3-S21 — FULL TRAIN OMNICLOUDMASK RELIABILITY AUDIT
#
# PURPOSE:
#   Apply frozen OmniCloudMask inference to all 314 TRAIN
#   scenes and save one continuous P(clear) map per scene.
#
# IMPORTANT:
#   - train only
#   - no threshold
#   - no morphology
#   - no flood-label tuning
# ============================================================

from pathlib import Path
import time
import json
import warnings

import numpy as np
import pandas as pd
import torch

print("=" * 104)
print("P3-C38 — FULL TRAIN OMNICLOUDMASK RELIABILITY AUDIT")
print("=" * 104)

OCM_TRAIN_MAP_ROOT = (
    OCM_ROOT
    /
    "train_pclear"
)

OCM_TRAIN_MAP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RED_INDEX = S2_INDEX[
    "B4"
]
GREEN_INDEX = S2_INDEX[
    "B3"
]
NIR_INDEX = S2_INDEX[
    "B8"
]


def run_omnicloud_reliability(
    reflectance,
    valid_mask,
    device
):
    model_input = np.stack(
        [
            reflectance[
                RED_INDEX
            ],
            reflectance[
                GREEN_INDEX
            ],
            reflectance[
                NIR_INDEX
            ]
        ],
        axis=0
    ).astype(
        np.float32
    )

    model_input[
        :,
        ~valid_mask
    ] = 0.0

    start = time.perf_counter()

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=UserWarning
        )

        probability = predict_from_array(
            input_array=model_input,
            inference_device=device,
            inference_dtype="fp32",
            export_confidence=True,
            softmax_output=True,
            no_data_value=0,
            apply_no_data_mask=True,
            pred_classes=4
        )

    runtime = (
        time.perf_counter()
        -
        start
    )

    probability = np.asarray(
        probability,
        dtype=np.float32
    )

    if (
        probability.ndim == 4
        and
        probability.shape[0] == 1
    ):
        probability = (
            probability[0]
        )

    expected_shape = (
        4,
        reflectance.shape[1],
        reflectance.shape[2]
    )

    if probability.shape != expected_shape:
        raise RuntimeError(
            "Unexpected OmniCloudMask output shape: "
            f"{probability.shape}"
        )

    valid_probability = (
        probability[
            :,
            valid_mask
        ]
    )

    probability_sum = np.maximum(
        valid_probability.sum(
            axis=0,
            keepdims=True
        ),
        1e-8
    )

    valid_probability = (
        valid_probability
        /
        probability_sum
    )

    normalized_probability = np.zeros_like(
        probability,
        dtype=np.float32
    )

    normalized_probability[
        :,
        valid_mask
    ] = valid_probability

    reliability = (
        normalized_probability[
            0
        ].copy()
    )

    reliability[
        ~valid_mask
    ] = 0.0

    hard_class = np.argmax(
        normalized_probability,
        axis=0
    ).astype(
        np.uint8
    )

    hard_class[
        ~valid_mask
    ] = 255

    return (
        normalized_probability,
        reliability,
        hard_class,
        runtime
    )


scene_records = []

total_start = time.perf_counter()

for position, (_, row) in enumerate(
    train_df.iterrows(),
    start=1
):

    scene_id_i = row[
        "scene_id"
    ]

    reflectance, valid_mask = (
        load_optical_reflectance(
            row
        )
    )

    valid_count = int(
        valid_mask.sum()
    )

    if valid_count == 0:
        raise RuntimeError(
            f"No valid optical pixels: {scene_id_i}"
        )

    (
        probability,
        reliability,
        hard_class,
        runtime_sec
    ) = run_omnicloud_reliability(
        reflectance,
        valid_mask,
        OCM_DEVICE
    )

    valid_prob = (
        probability[
            :,
            valid_mask
        ]
    )

    sums = valid_prob.sum(
        axis=0
    )

    assert np.allclose(
        sums,
        1.0,
        atol=1e-5
    )

    valid_classes = (
        hard_class[
            valid_mask
        ]
    )

    class_counts = np.bincount(
        valid_classes,
        minlength=4
    )

    class_pct = (
        100.0
        *
        class_counts
        /
        valid_count
    )

    valid_reliability = (
        reliability[
            valid_mask
        ]
    )

    p = np.percentile(
        valid_reliability,
        [
            1,
            5,
            10,
            25,
            50,
            75,
            90,
            95,
            99
        ]
    )

    reliability_path = (
        OCM_TRAIN_MAP_ROOT
        /
        f"{scene_id_i}_pclear.npy"
    )

    np.save(
        reliability_path,
        reliability.astype(
            np.float16
        )
    )

    scene_records.append({
        "scene_id":
            scene_id_i,

        "region":
            row[
                "region"
            ],

        "valid_pixels":
            valid_count,

        "valid_pct":
            float(
                100.0
                *
                valid_mask.mean()
            ),

        "clear_pct":
            float(
                class_pct[0]
            ),

        "thick_cloud_pct":
            float(
                class_pct[1]
            ),

        "thin_cloud_pct":
            float(
                class_pct[2]
            ),

        "cloud_shadow_pct":
            float(
                class_pct[3]
            ),

        "mean_p_clear":
            float(
                valid_reliability.mean()
            ),

        "std_p_clear":
            float(
                valid_reliability.std()
            ),

        "pclear_p1":
            float(p[0]),

        "pclear_p5":
            float(p[1]),

        "pclear_p10":
            float(p[2]),

        "pclear_p25":
            float(p[3]),

        "pclear_p50":
            float(p[4]),

        "pclear_p75":
            float(p[5]),

        "pclear_p90":
            float(p[6]),

        "pclear_p95":
            float(p[7]),

        "pclear_p99":
            float(p[8]),

        "runtime_sec":
            float(
                runtime_sec
            ),

        "reliability_path":
            str(
                reliability_path
            )
    })

    if (
        position == 1
        or
        position % 25 == 0
        or
        position == len(train_df)
    ):
        print(
            f"[{position:03d}/{len(train_df)}] "
            f"{scene_id_i:24s} | "
            f"valid {100*valid_mask.mean():6.2f}% | "
            f"clear {class_pct[0]:6.2f}% | "
            f"cloud {class_pct[1]+class_pct[2]:6.2f}% | "
            f"shadow {class_pct[3]:6.2f}% | "
            f"mean Pclear {valid_reliability.mean():.4f} | "
            f"{runtime_sec:.2f}s"
        )

optical_train_cloud_audit = pd.DataFrame(
    scene_records
)

total_runtime = (
    time.perf_counter()
    -
    total_start
)

print("\n" + "=" * 104)
print("TRAIN OMNICLOUDMASK AUDIT COMPLETE")
print("=" * 104)
print(
    "Scenes processed:",
    len(
        optical_train_cloud_audit
    )
)
print(
    "Total runtime:",
    f"{total_runtime/60.0:.2f} min"
)
print(
    "Mean inference runtime:",
    f"{optical_train_cloud_audit['runtime_sec'].mean():.3f} sec/scene"
)

weights = optical_train_cloud_audit[
    "valid_pixels"
].to_numpy(
    dtype=np.float64
)

total_valid_pixels = weights.sum()

def weighted_pct(
    column
):
    values = optical_train_cloud_audit[
        column
    ].to_numpy(
        dtype=np.float64
    )

    return float(
        np.sum(
            values
            *
            weights
        )
        /
        total_valid_pixels
    )

GLOBAL_CLEAR_PCT = weighted_pct(
    "clear_pct"
)
GLOBAL_THICK_CLOUD_PCT = weighted_pct(
    "thick_cloud_pct"
)
GLOBAL_THIN_CLOUD_PCT = weighted_pct(
    "thin_cloud_pct"
)
GLOBAL_SHADOW_PCT = weighted_pct(
    "cloud_shadow_pct"
)
GLOBAL_CLOUD_PCT = (
    GLOBAL_THICK_CLOUD_PCT
    +
    GLOBAL_THIN_CLOUD_PCT
)

print("\n" + "=" * 104)
print("PIXEL-WEIGHTED TRAIN CLASS DISTRIBUTION")
print("=" * 104)

print(
    f"Clear        : {GLOBAL_CLEAR_PCT:.2f}%"
)
print(
    f"Thick cloud  : {GLOBAL_THICK_CLOUD_PCT:.2f}%"
)
print(
    f"Thin cloud   : {GLOBAL_THIN_CLOUD_PCT:.2f}%"
)
print(
    f"Cloud shadow : {GLOBAL_SHADOW_PCT:.2f}%"
)
print(
    f"Any cloud    : {GLOBAL_CLOUD_PCT:.2f}%"
)

print("\n" + "=" * 104)
print("SCENE-LEVEL DISTRIBUTION")
print("=" * 104)

print(
    optical_train_cloud_audit[
        [
            "clear_pct",
            "thick_cloud_pct",
            "thin_cloud_pct",
            "cloud_shadow_pct",
            "mean_p_clear"
        ]
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    ).to_string()
)

sorted_reliability = (
    optical_train_cloud_audit
    .sort_values(
        "mean_p_clear"
    )
    .reset_index(
        drop=True
    )
)

low_scene = sorted_reliability.iloc[0]
medium_scene = sorted_reliability.iloc[
    len(
        sorted_reliability
    )
    //
    2
]
high_scene = sorted_reliability.iloc[-1]

optical_reliability_representatives = pd.DataFrame(
    [
        {
            "category":
                "LOW reliability",

            **low_scene.to_dict()
        },

        {
            "category":
                "MEDIUM reliability",

            **medium_scene.to_dict()
        },

        {
            "category":
                "HIGH reliability",

            **high_scene.to_dict()
        }
    ]
)

print("\n" + "=" * 104)
print("FROZEN OPTICAL RELIABILITY REPRESENTATIVE SCENES")
print("=" * 104)

print(
    optical_reliability_representatives[
        [
            "category",
            "scene_id",
            "region",
            "valid_pct",
            "clear_pct",
            "thick_cloud_pct",
            "thin_cloud_pct",
            "cloud_shadow_pct",
            "mean_p_clear"
        ]
    ].to_string(
        index=False
    )
)

optical_train_cloud_audit[
    "any_cloud_pct"
] = (
    optical_train_cloud_audit[
        "thick_cloud_pct"
    ]
    +
    optical_train_cloud_audit[
        "thin_cloud_pct"
    ]
)

print("\nTOP 10 CLOUDIEST TRAIN SCENES")
print("-" * 104)

print(
    optical_train_cloud_audit
    .sort_values(
        "any_cloud_pct",
        ascending=False
    )[
        [
            "scene_id",
            "region",
            "valid_pct",
            "any_cloud_pct",
            "cloud_shadow_pct",
            "mean_p_clear"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)

print("\nTOP 10 SHADOW-HEAVIEST TRAIN SCENES")
print("-" * 104)

print(
    optical_train_cloud_audit
    .sort_values(
        "cloud_shadow_pct",
        ascending=False
    )[
        [
            "scene_id",
            "region",
            "valid_pct",
            "any_cloud_pct",
            "cloud_shadow_pct",
            "mean_p_clear"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)

OCM_TRAIN_AUDIT_PATH = (
    OCM_ROOT
    /
    "omnicloudmask_train_scene_audit.csv"
)

OCM_REPRESENTATIVE_PATH = (
    OCM_ROOT
    /
    "omnicloudmask_train_representatives.csv"
)

OCM_TRAIN_SUMMARY_PATH = (
    OCM_ROOT
    /
    "omnicloudmask_train_summary.json"
)

optical_train_cloud_audit.to_csv(
    OCM_TRAIN_AUDIT_PATH,
    index=False
)

optical_reliability_representatives.to_csv(
    OCM_REPRESENTATIVE_PATH,
    index=False
)

with open(
    OCM_TRAIN_SUMMARY_PATH,
    "w"
) as f:
    json.dump(
        {
            "model":
                "OmniCloudMask",

            "version":
                str(
                    OCM_VERSION
                ),

            "split":
                "train",

            "scene_count":
                314,

            "input_bands":
                [
                    "B4",
                    "B3",
                    "B8"
                ],

            "class_order":
                OCM_CLASS_NAMES,

            "reliability_definition":
                "renormalized_P_clear",

            "invalid_reliability":
                0.0,

            "strict_validity":
                OPTICAL_VALIDITY_RULE,

            "global_clear_pct":
                GLOBAL_CLEAR_PCT,

            "global_thick_cloud_pct":
                GLOBAL_THICK_CLOUD_PCT,

            "global_thin_cloud_pct":
                GLOBAL_THIN_CLOUD_PCT,

            "global_cloud_shadow_pct":
                GLOBAL_SHADOW_PCT,

            "mean_scene_p_clear":
                float(
                    optical_train_cloud_audit[
                        "mean_p_clear"
                    ].mean()
                ),

            "mean_runtime_sec":
                float(
                    optical_train_cloud_audit[
                        "runtime_sec"
                    ].mean()
                ),

            "threshold":
                None,

            "morphology":
                None,

            "full_policy_locked":
                False
        },
        f,
        indent=2
    )

assert len(
    optical_train_cloud_audit
) == 314

print("\nSaved:")
print(" ", OCM_TRAIN_AUDIT_PATH)
print(" ", OCM_REPRESENTATIVE_PATH)
print(" ", OCM_TRAIN_SUMMARY_PATH)
print(" Reliability maps:")
print(" ", OCM_TRAIN_MAP_ROOT)

print("\nP3-C38 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C39
# SECTION: P3-S22 — OPTICAL RELIABILITY VISUAL VALIDATION
#
# PURPOSE:
#   Re-run the frozen LOW/MEDIUM/HIGH reliability scenes,
#   confirm exact numerical reproducibility, and save a compact
#   visual audit before locking the method.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

print("=" * 104)
print("P3-C39 — OPTICAL RELIABILITY VISUAL VALIDATION")
print("=" * 104)

expected_categories = {
    "LOW reliability",
    "MEDIUM reliability",
    "HIGH reliability"
}

assert set(
    optical_reliability_representatives[
        "category"
    ]
) == expected_categories

CLASS_CMAP = ListedColormap(
    [
        "#2ca02c",
        "#ffffff",
        "#87ceeb",
        "#4b4b4b"
    ]
)


def make_rgb_display(
    reflectance,
    valid_mask
):
    rgb = np.stack(
        [
            reflectance[
                S2_INDEX["B4"]
            ],
            reflectance[
                S2_INDEX["B3"]
            ],
            reflectance[
                S2_INDEX["B2"]
            ]
        ],
        axis=-1
    )

    valid_rgb = rgb[
        valid_mask
    ]

    low = np.percentile(
        valid_rgb,
        2
    )

    high = np.percentile(
        valid_rgb,
        98
    )

    rgb_display = np.clip(
        (
            rgb
            -
            low
        )
        /
        max(
            high
            -
            low,
            1e-6
        ),
        0.0,
        1.0
    )

    rgb_display[
        ~valid_mask
    ] = 0.0

    return rgb_display


visual_records = []
visual_data = []

for _, representative in (
    optical_reliability_representatives.iterrows()
):

    category = representative[
        "category"
    ]

    scene_id_i = representative[
        "scene_id"
    ]

    matching_rows = train_df[
        train_df[
            "scene_id"
        ]
        ==
        scene_id_i
    ]

    assert len(
        matching_rows
    ) == 1

    row = matching_rows.iloc[0]

    reflectance, valid_mask = (
        load_optical_reflectance(
            row
        )
    )

    (
        probability,
        reliability,
        hard_class,
        runtime_sec
    ) = run_omnicloud_reliability(
        reflectance,
        valid_mask,
        OCM_DEVICE
    )

    rgb_display = make_rgb_display(
        reflectance,
        valid_mask
    )

    obscuration = (
        1.0
        -
        reliability
    )

    obscuration[
        ~valid_mask
    ] = 0.0

    hard_display = hard_class.astype(
        np.float32
    )

    hard_display[
        ~valid_mask
    ] = np.nan

    valid_classes = hard_class[
        valid_mask
    ]

    counts = np.bincount(
        valid_classes,
        minlength=4
    )

    percentages = (
        100.0
        *
        counts
        /
        valid_mask.sum()
    )

    visual_records.append({
        "category":
            category,

        "scene_id":
            scene_id_i,

        "region":
            row[
                "region"
            ],

        "valid_pct":
            float(
                100.0
                *
                valid_mask.mean()
            ),

        "clear_pct":
            float(
                percentages[0]
            ),

        "thick_cloud_pct":
            float(
                percentages[1]
            ),

        "thin_cloud_pct":
            float(
                percentages[2]
            ),

        "cloud_shadow_pct":
            float(
                percentages[3]
            ),

        "mean_p_clear":
            float(
                reliability[
                    valid_mask
                ].mean()
            ),

        "median_p_clear":
            float(
                np.median(
                    reliability[
                        valid_mask
                    ]
                )
            ),

        "runtime_sec":
            float(
                runtime_sec
            )
    })

    visual_data.append({
        "category":
            category,

        "scene_id":
            scene_id_i,

        "rgb":
            rgb_display,

        "hard":
            hard_display,

        "reliability":
            reliability,

        "obscuration":
            obscuration
    })

optical_reliability_visual_audit = pd.DataFrame(
    visual_records
)

print("\n" + "=" * 104)
print("REPRESENTATIVE NUMERICAL CHECK")
print("=" * 104)

print(
    optical_reliability_visual_audit[
        [
            "category",
            "scene_id",
            "region",
            "valid_pct",
            "clear_pct",
            "thick_cloud_pct",
            "thin_cloud_pct",
            "cloud_shadow_pct",
            "mean_p_clear",
            "median_p_clear"
        ]
    ].to_string(
        index=False
    )
)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(
        20,
        15
    )
)

for row_idx, item in enumerate(
    visual_data
):

    axes[
        row_idx,
        0
    ].imshow(
        item[
            "rgb"
        ]
    )

    axes[
        row_idx,
        0
    ].set_title(
        f"{item['category']}\n"
        f"{item['scene_id']}\n"
        "Sentinel-2 RGB"
    )

    axes[
        row_idx,
        1
    ].imshow(
        item[
            "hard"
        ],
        cmap=CLASS_CMAP,
        vmin=0,
        vmax=3
    )

    axes[
        row_idx,
        1
    ].set_title(
        "OmniCloudMask classes"
    )

    pclear_image = axes[
        row_idx,
        2
    ].imshow(
        item[
            "reliability"
        ],
        vmin=0.0,
        vmax=1.0
    )

    axes[
        row_idx,
        2
    ].set_title(
        "Optical reliability\nP(clear)"
    )

    obscuration_image = axes[
        row_idx,
        3
    ].imshow(
        item[
            "obscuration"
        ],
        vmin=0.0,
        vmax=1.0
    )

    axes[
        row_idx,
        3
    ].set_title(
        "Optical obscuration\n1 - P(clear)"
    )

    for col_idx in range(4):
        axes[
            row_idx,
            col_idx
        ].axis(
            "off"
        )

fig.colorbar(
    pclear_image,
    ax=axes[:, 2],
    fraction=0.02,
    pad=0.02,
    label="P(clear)"
)

fig.colorbar(
    obscuration_image,
    ax=axes[:, 3],
    fraction=0.02,
    pad=0.02,
    label="1 - P(clear)"
)

plt.suptitle(
    "FloodShift — OmniCloudMask Reliability Visual Audit",
    fontsize=16,
    y=0.995
)

VISUAL_AUDIT_PATH = (
    OCM_ROOT
    /
    "omnicloudmask_representative_visual_audit.png"
)

plt.savefig(
    VISUAL_AUDIT_PATH,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

VISUAL_TABLE_PATH = (
    OCM_ROOT
    /
    "omnicloudmask_representative_visual_audit.csv"
)

optical_reliability_visual_audit.to_csv(
    VISUAL_TABLE_PATH,
    index=False
)

merged_check = (
    optical_reliability_visual_audit
    .merge(
        optical_reliability_representatives[
            [
                "scene_id",
                "mean_p_clear"
            ]
        ],
        on="scene_id",
        suffixes=(
            "_c39",
            "_c38"
        )
    )
)

max_mean_difference = float(
    np.max(
        np.abs(
            merged_check[
                "mean_p_clear_c39"
            ]
            -
            merged_check[
                "mean_p_clear_c38"
            ]
        )
    )
)

print(
    "\nMaximum C38 ↔ C39 mean P(clear) difference:",
    max_mean_difference
)

assert (
    max_mean_difference
    <
    1e-4
)

print("\nSaved:")
print(" ", VISUAL_AUDIT_PATH)
print(" ", VISUAL_TABLE_PATH)

print("\nP3-C39 completed successfully ✅")
print(
    "Numerical reproducibility confirmed."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C40
# SECTION: P3-S23 — OPTICAL RELIABILITY + 10-BAND ENCODER LOCK
#
# PURPOSE:
#   Permanently freeze:
#   A) optical reliability = renormalized P(clear)
#   B) 10-band FloodShift optical encoder input
#
# ENCODER BANDS:
#   B2 B3 B4 B5 B6 B7 B8 B8A B11 B12
#
# EXCLUDED FROM ENCODER:
#   B1 B9 B10
#
# Raw 13-band source data remains preserved.
# ============================================================

from pathlib import Path
import json

import numpy as np

print("=" * 104)
print("P3-C40 — OPTICAL RELIABILITY + 10-BAND ENCODER LOCK")
print("=" * 104)

OPTICAL_RELIABILITY_MODEL = (
    "OmniCloudMask"
)

OPTICAL_RELIABILITY_DEFINITION = (
    "renormalized_P_clear"
)

OPTICAL_RELIABILITY_INVALID_VALUE = 0.0
OPTICAL_RELIABILITY_HARD_THRESHOLD = None
OPTICAL_RELIABILITY_CLASSES = OCM_CLASS_NAMES.copy()

assert len(
    optical_train_cloud_audit
) == 314

assert max_mean_difference < 1e-4

OPTICAL_ENCODER_BANDS = [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12"
]

OPTICAL_ENCODER_EXCLUDED_BANDS = [
    "B1",
    "B9",
    "B10"
]

assert set(
    OPTICAL_ENCODER_BANDS
    +
    OPTICAL_ENCODER_EXCLUDED_BANDS
) == set(
    OPTICAL_BAND_NAMES
)

OPTICAL_ENCODER_INDICES = [
    OPTICAL_BAND_NAMES.index(
        band_name
    )
    for band_name in OPTICAL_ENCODER_BANDS
]

print("\nFINAL FLOODSHIFT OPTICAL ENCODER BANDS")
for encoder_channel, band_name in enumerate(
    OPTICAL_ENCODER_BANDS,
    start=1
):
    print(
        f"Encoder channel {encoder_channel:02d} -> {band_name}"
    )

print("\nExcluded from encoder:")
for band_name in OPTICAL_ENCODER_EXCLUDED_BANDS:
    print(" ", band_name)

print("\n13-band source indices:")
for band_name, source_index in zip(
    OPTICAL_ENCODER_BANDS,
    OPTICAL_ENCODER_INDICES
):
    print(
        f"  {band_name:4s} -> zero-based source index {source_index}"
    )


def load_optical_encoder_reflectance(
    row
):
    reflectance_13, valid_mask = (
        load_optical_reflectance(
            row
        )
    )

    optical_10 = reflectance_13[
        OPTICAL_ENCODER_INDICES
    ].astype(
        np.float32,
        copy=False
    )

    return (
        optical_10,
        valid_mask
    )


sample_optical_10, sample_valid_10 = (
    load_optical_encoder_reflectance(
        optical_sample_row
    )
)

assert sample_optical_10.shape == (
    10,
    512,
    512
)

assert np.array_equal(
    sample_valid_10,
    sample_valid
)

sample_13, _ = load_optical_reflectance(
    optical_sample_row
)

for new_idx, old_idx in enumerate(
    OPTICAL_ENCODER_INDICES
):
    assert np.array_equal(
        sample_optical_10[
            new_idx
        ],
        sample_13[
            old_idx
        ]
    )

print("\n10-band selection verified against 13-band source ✅")

pixel_weighted_summary = {
    "clear_pct":
        float(
            GLOBAL_CLEAR_PCT
        ),

    "thick_cloud_pct":
        float(
            GLOBAL_THICK_CLOUD_PCT
        ),

    "thin_cloud_pct":
        float(
            GLOBAL_THIN_CLOUD_PCT
        ),

    "cloud_shadow_pct":
        float(
            GLOBAL_SHADOW_PCT
        ),

    "any_cloud_pct":
        float(
            GLOBAL_CLOUD_PCT
        )
}

OPTICAL_LOCK_ROOT = (
    OPTICAL_ROOT
    /
    "locks"
)

OPTICAL_LOCK_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

OPTICAL_RELIABILITY_LOCK_PATH = (
    OPTICAL_LOCK_ROOT
    /
    "optical_reliability_lock.json"
)

OPTICAL_BAND_LOCK_PATH = (
    OPTICAL_LOCK_ROOT
    /
    "optical_encoder_band_lock.json"
)

optical_reliability_lock = {
    "status":
        "FROZEN",

    "model":
        OPTICAL_RELIABILITY_MODEL,

    "model_version":
        str(
            OCM_VERSION
        ),

    "model_input_bands":
        [
            "B4",
            "B3",
            "B8"
        ],

    "class_order":
        OPTICAL_RELIABILITY_CLASSES,

    "postprocessing":
        "per-pixel renormalization across four classes",

    "reliability_definition":
        OPTICAL_RELIABILITY_DEFINITION,

    "invalid_pixel_reliability":
        OPTICAL_RELIABILITY_INVALID_VALUE,

    "strict_optical_validity":
        OPTICAL_VALIDITY_RULE,

    "hard_threshold":
        None,

    "morphology":
        None,

    "cloud_inpainting":
        False,

    "train_pixel_weighted_summary":
        pixel_weighted_summary,

    "method_development_closed":
        True
}

optical_band_lock = {
    "status":
        "FROZEN",

    "raw_source_band_count":
        13,

    "raw_source_band_order":
        OPTICAL_BAND_NAMES,

    "raw_source_preserved":
        True,

    "encoder_band_count":
        10,

    "encoder_bands":
        OPTICAL_ENCODER_BANDS,

    "encoder_source_zero_based_indices":
        OPTICAL_ENCODER_INDICES,

    "excluded_from_encoder":
        {
            "B1":
                "coastal aerosol",

            "B9":
                "water vapour",

            "B10":
                "cirrus"
        },

    "reflectance_scaling":
        "raw / 10000",

    "clipping":
        False,

    "strict_validity_derived_from":
        "all original 13 bands",

    "normalization":
        "NOT YET LOCKED",

    "band_selection_frozen":
        True
}

with open(
    OPTICAL_RELIABILITY_LOCK_PATH,
    "w"
) as f:
    json.dump(
        optical_reliability_lock,
        f,
        indent=2
    )

with open(
    OPTICAL_BAND_LOCK_PATH,
    "w"
) as f:
    json.dump(
        optical_band_lock,
        f,
        indent=2
    )

print("\nSaved:")
print(" ", OPTICAL_RELIABILITY_LOCK_PATH)
print(" ", OPTICAL_BAND_LOCK_PATH)

print("\nOPTICAL RELIABILITY METHOD LOCKED ✅")
print("OPTICAL ENCODER BANDS LOCKED ✅")
print("\nP3-C40 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C41
# SECTION: P3-S24 — TRAIN-ONLY OPTICAL NORMALIZATION
#
# PURPOSE:
#   Compute exact per-band normalization statistics for the
#   frozen 10-band Sentinel-2 optical encoder input.
#
# DATA:
#   canonical TRAIN only = 314 scenes
#
# PIXELS:
#   strict optical-valid pixels only
#   clouds/shadows included
#
# NORMALIZATION:
#   z = (reflectance - train_mean) / train_std
#
# INVALID PIXELS:
#   filled with 0 AFTER normalization.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

print("=" * 104)
print("P3-C41 — EXACT TRAIN-ONLY 10-BAND NORMALIZATION")
print("=" * 104)

assert len(
    train_df
) == 314

assert OPTICAL_ENCODER_BANDS == [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12"
]

N_OPTICAL_CHANNELS = 10

pixel_count = np.zeros(
    N_OPTICAL_CHANNELS,
    dtype=np.int64
)

pixel_sum = np.zeros(
    N_OPTICAL_CHANNELS,
    dtype=np.float64
)

pixel_sum_sq = np.zeros(
    N_OPTICAL_CHANNELS,
    dtype=np.float64
)

pixel_min = np.full(
    N_OPTICAL_CHANNELS,
    np.inf,
    dtype=np.float64
)

pixel_max = np.full(
    N_OPTICAL_CHANNELS,
    -np.inf,
    dtype=np.float64
)

scene_valid_records = []

for position, (_, row) in enumerate(
    train_df.iterrows(),
    start=1
):

    scene_id_i = row[
        "scene_id"
    ]

    optical_10, valid_mask = (
        load_optical_encoder_reflectance(
            row
        )
    )

    assert optical_10.shape == (
        10,
        512,
        512
    )

    valid_pixels_scene = int(
        valid_mask.sum()
    )

    if valid_pixels_scene == 0:
        raise RuntimeError(
            f"No strict-valid optical pixels in {scene_id_i}"
        )

    for band_idx in range(
        N_OPTICAL_CHANNELS
    ):

        values = optical_10[
            band_idx
        ][
            valid_mask
        ].astype(
            np.float64,
            copy=False
        )

        n = int(
            values.size
        )

        pixel_count[
            band_idx
        ] += n

        pixel_sum[
            band_idx
        ] += np.sum(
            values,
            dtype=np.float64
        )

        pixel_sum_sq[
            band_idx
        ] += np.sum(
            values
            *
            values,
            dtype=np.float64
        )

        pixel_min[
            band_idx
        ] = min(
            pixel_min[
                band_idx
            ],
            float(
                values.min()
            )
        )

        pixel_max[
            band_idx
        ] = max(
            pixel_max[
                band_idx
            ],
            float(
                values.max()
            )
        )

    scene_valid_records.append({
        "scene_id":
            scene_id_i,

        "region":
            row[
                "region"
            ],

        "strict_valid_pixels":
            valid_pixels_scene,

        "strict_valid_pct":
            float(
                100.0
                *
                valid_mask.mean()
            )
    })

    if (
        position == 1
        or
        position % 50 == 0
        or
        position == len(
            train_df
        )
    ):
        print(
            f"[{position:03d}/{len(train_df)}] "
            f"{scene_id_i:24s} | "
            f"valid {100.0*valid_mask.mean():6.2f}%"
        )

assert (
    pixel_count > 0
).all()

OPTICAL_NORM_MEAN = (
    pixel_sum
    /
    pixel_count
)

variance = (
    pixel_sum_sq
    /
    pixel_count
    -
    OPTICAL_NORM_MEAN
    **
    2
)

variance = np.maximum(
    variance,
    0.0
)

OPTICAL_NORM_STD = np.sqrt(
    variance
)

assert (
    OPTICAL_NORM_STD > 0
).all()

normalization_records = []

for band_idx, band_name in enumerate(
    OPTICAL_ENCODER_BANDS
):
    normalization_records.append({
        "channel":
            band_idx + 1,

        "band_name":
            band_name,

        "pixel_count":
            int(
                pixel_count[
                    band_idx
                ]
            ),

        "mean":
            float(
                OPTICAL_NORM_MEAN[
                    band_idx
                ]
            ),

        "std":
            float(
                OPTICAL_NORM_STD[
                    band_idx
                ]
            ),

        "minimum":
            float(
                pixel_min[
                    band_idx
                ]
            ),

        "maximum":
            float(
                pixel_max[
                    band_idx
                ]
            )
    })

optical_normalization_stats = pd.DataFrame(
    normalization_records
)

print("\n" + "=" * 104)
print("EXACT TRAIN-ONLY NORMALIZATION STATISTICS")
print("=" * 104)

print(
    optical_normalization_stats.to_string(
        index=False
    )
)

unique_counts = np.unique(
    pixel_count
)

assert len(
    unique_counts
) == 1

TOTAL_TRAIN_VALID_PIXELS = int(
    unique_counts[0]
)

print(
    "\nTotal strict-valid TRAIN pixels:",
    f"{TOTAL_TRAIN_VALID_PIXELS:,}"
)

# ------------------------------------------------------------
# Compare exact values to C35 sampled values.
# ------------------------------------------------------------

sampled_reference = (
    optical_train_scaled_distribution[
        optical_train_scaled_distribution[
            "band_name"
        ].isin(
            OPTICAL_ENCODER_BANDS
        )
    ][
        [
            "band_name",
            "mean",
            "std"
        ]
    ]
    .set_index(
        "band_name"
    )
    .loc[
        OPTICAL_ENCODER_BANDS
    ]
    .reset_index()
)

comparison = (
    optical_normalization_stats[
        [
            "band_name",
            "mean",
            "std"
        ]
    ]
    .merge(
        sampled_reference,
        on="band_name",
        suffixes=(
            "_exact",
            "_c35_sample"
        )
    )
)

comparison[
    "mean_abs_delta"
] = np.abs(
    comparison[
        "mean_exact"
    ]
    -
    comparison[
        "mean_c35_sample"
    ]
)

comparison[
    "std_abs_delta"
] = np.abs(
    comparison[
        "std_exact"
    ]
    -
    comparison[
        "std_c35_sample"
    ]
)

print("\n" + "=" * 104)
print("C41 EXACT vs C35 SAMPLED SANITY CHECK")
print("=" * 104)

print(
    comparison.to_string(
        index=False
    )
)

print(
    "\nMax mean difference:",
    float(
        comparison[
            "mean_abs_delta"
        ].max()
    )
)

print(
    "Max std difference :",
    float(
        comparison[
            "std_abs_delta"
        ].max()
    )
)


def normalize_optical_encoder(
    optical_10,
    valid_mask
):
    optical_10 = np.asarray(
        optical_10,
        dtype=np.float32
    )

    valid_mask = np.asarray(
        valid_mask,
        dtype=bool
    )

    if optical_10.shape[0] != 10:
        raise ValueError(
            "Expected 10 optical channels."
        )

    if valid_mask.shape != optical_10.shape[1:]:
        raise ValueError(
            "Validity-mask spatial shape mismatch."
        )

    means = OPTICAL_NORM_MEAN[
        :,
        None,
        None
    ].astype(
        np.float32
    )

    stds = OPTICAL_NORM_STD[
        :,
        None,
        None
    ].astype(
        np.float32
    )

    normalized = (
        (
            optical_10
            -
            means
        )
        /
        stds
    ).astype(
        np.float32
    )

    normalized[
        :,
        ~valid_mask
    ] = 0.0

    return normalized


sample_optical_10, sample_valid_norm = (
    load_optical_encoder_reflectance(
        optical_sample_row
    )
)

sample_normalized = normalize_optical_encoder(
    sample_optical_10,
    sample_valid_norm
)

print("\n" + "=" * 104)
print("NORMALIZED SAMPLE SANITY CHECK")
print("=" * 104)
print("Shape:", sample_normalized.shape)
print("Dtype:", sample_normalized.dtype)
print(
    "Finite:",
    np.isfinite(
        sample_normalized
    ).all()
)
print(
    "Invalid pixels exactly zero:",
    bool(
        (
            sample_normalized[
                :,
                ~sample_valid_norm
            ]
            ==
            0
        ).all()
    )
)
print(
    "Valid normalized min:",
    float(
        sample_normalized[
            :,
            sample_valid_norm
        ].min()
    )
)
print(
    "Valid normalized max:",
    float(
        sample_normalized[
            :,
            sample_valid_norm
        ].max()
    )
)

OPTICAL_NORMALIZATION_STATS_PATH = (
    OPTICAL_ROOT
    /
    "optical_10band_train_normalization_exact.csv"
)

OPTICAL_NORMALIZATION_LOCK_PATH = (
    OPTICAL_LOCK_ROOT
    /
    "optical_normalization_lock.json"
)

optical_normalization_stats.to_csv(
    OPTICAL_NORMALIZATION_STATS_PATH,
    index=False
)

normalization_payload = {
    "status":
        "FROZEN",

    "split_used":
        "train",

    "scene_count":
        314,

    "canonical_validation_used":
        False,

    "canonical_test_used":
        False,

    "encoder_band_count":
        10,

    "encoder_bands":
        OPTICAL_ENCODER_BANDS,

    "input_domain":
        "Sentinel-2 reflectance after raw / 10000",

    "validity_rule":
        "all_13_bands_finite_and_nonzero",

    "cloud_pixels_included":
        True,

    "cloud_shadow_pixels_included":
        True,

    "pclear_weighting":
        False,

    "normalization":
        "per-band z-score",

    "equation":
        "z = (x - train_mean) / train_std",

    "std_definition":
        "population_std",

    "clipping":
        False,

    "invalid_fill_after_normalization":
        0.0,

    "pixel_count_per_band":
        TOTAL_TRAIN_VALID_PIXELS,

    "mean": {
        band_name:
            float(
                OPTICAL_NORM_MEAN[
                    idx
                ]
            )
        for idx, band_name in enumerate(
            OPTICAL_ENCODER_BANDS
        )
    },

    "std": {
        band_name:
            float(
                OPTICAL_NORM_STD[
                    idx
                ]
            )
        for idx, band_name in enumerate(
            OPTICAL_ENCODER_BANDS
        )
    }
}

with open(
    OPTICAL_NORMALIZATION_LOCK_PATH,
    "w"
) as f:
    json.dump(
        normalization_payload,
        f,
        indent=2
    )

print("\nSaved:")
print(" ", OPTICAL_NORMALIZATION_STATS_PATH)
print(" ", OPTICAL_NORMALIZATION_LOCK_PATH)

assert np.isfinite(
    OPTICAL_NORM_MEAN
).all()

assert np.isfinite(
    OPTICAL_NORM_STD
).all()

print("\nOPTICAL NORMALIZATION LOCKED ✅")
print("\nP3-C41 completed successfully ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C42
# SECTION: P3-S25 — FROZEN OPTICAL PIPELINE ON VAL + TEST
#
# PURPOSE:
#   Apply the fully frozen optical preprocessing recipe to
#   canonical validation and test scenes.
#
# IMPORTANT:
#   - TRAIN statistics reused unchanged
#   - no holdout tuning
#   - no threshold selection
#   - no clipping introduced
# ============================================================

from pathlib import Path
import time
import json

import numpy as np
import pandas as pd

print("=" * 106)
print("P3-C42 — APPLY FROZEN OPTICAL PIPELINE TO VALIDATION + TEST")
print("=" * 106)

validation_df = (
    master[
        master[
            "split"
        ]
        ==
        "validation"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

test_df = (
    master[
        master[
            "split"
        ]
        ==
        "test"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

assert len(
    validation_df
) == 68

assert len(
    test_df
) == 58

print(
    "Validation scenes:",
    len(
        validation_df
    )
)
print(
    "Test scenes      :",
    len(
        test_df
    )
)

VAL_RELIABILITY_ROOT = (
    OCM_ROOT
    /
    "validation_pclear"
)

TEST_RELIABILITY_ROOT = (
    OCM_ROOT
    /
    "test_pclear"
)

VAL_RELIABILITY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

TEST_RELIABILITY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def process_frozen_optical_split(
    split_df,
    split_name,
    reliability_root
):
    records = []

    print("\n" + "=" * 106)
    print(
        f"PROCESSING {split_name.upper()}"
    )
    print("=" * 106)

    split_start = time.perf_counter()

    for position, (_, row) in enumerate(
        split_df.iterrows(),
        start=1
    ):

        scene_id_i = row[
            "scene_id"
        ]

        reflectance_13, valid_mask = (
            load_optical_reflectance(
                row
            )
        )

        valid_count = int(
            valid_mask.sum()
        )

        if valid_count == 0:
            raise RuntimeError(
                f"{scene_id_i}: zero strict-valid optical pixels"
            )

        valid_pct = float(
            100.0
            *
            valid_mask.mean()
        )

        optical_10 = reflectance_13[
            OPTICAL_ENCODER_INDICES
        ].astype(
            np.float32,
            copy=False
        )

        normalized_10 = normalize_optical_encoder(
            optical_10,
            valid_mask
        )

        assert np.isfinite(
            normalized_10
        ).all()

        assert (
            normalized_10[
                :,
                ~valid_mask
            ]
            ==
            0.0
        ).all()

        (
            probability,
            reliability,
            hard_class,
            runtime_sec
        ) = run_omnicloud_reliability(
            reflectance_13,
            valid_mask,
            OCM_DEVICE
        )

        assert np.isfinite(
            reliability
        ).all()

        assert (
            reliability[
                ~valid_mask
            ]
            ==
            0.0
        ).all()

        reliability_path = (
            reliability_root
            /
            f"{scene_id_i}_pclear.npy"
        )

        np.save(
            reliability_path,
            reliability.astype(
                np.float16
            )
        )

        valid_classes = hard_class[
            valid_mask
        ]

        class_counts = np.bincount(
            valid_classes,
            minlength=4
        )

        class_pct = (
            100.0
            *
            class_counts
            /
            valid_count
        )

        valid_reliability = reliability[
            valid_mask
        ]

        normalized_valid = normalized_10[
            :,
            valid_mask
        ]

        records.append({
            "scene_id":
                scene_id_i,

            "region":
                row[
                    "region"
                ],

            "split":
                split_name,

            "valid_pixels":
                valid_count,

            "valid_pct":
                valid_pct,

            "clear_pct":
                float(
                    class_pct[0]
                ),

            "thick_cloud_pct":
                float(
                    class_pct[1]
                ),

            "thin_cloud_pct":
                float(
                    class_pct[2]
                ),

            "cloud_shadow_pct":
                float(
                    class_pct[3]
                ),

            "mean_p_clear":
                float(
                    valid_reliability.mean()
                ),

            "median_p_clear":
                float(
                    np.median(
                        valid_reliability
                    )
                ),

            "normalized_min":
                float(
                    normalized_valid.min()
                ),

            "normalized_max":
                float(
                    normalized_valid.max()
                ),

            "normalized_mean":
                float(
                    normalized_valid.mean()
                ),

            "runtime_sec":
                float(
                    runtime_sec
                ),

            "reliability_path":
                str(
                    reliability_path
                )
        })

        if (
            position == 1
            or
            position % 20 == 0
            or
            position == len(
                split_df
            )
        ):
            print(
                f"[{position:03d}/{len(split_df)}] "
                f"{scene_id_i:24s} | "
                f"valid {valid_pct:6.2f}% | "
                f"Pclear {valid_reliability.mean():.4f} | "
                f"z [{normalized_valid.min():.2f}, "
                f"{normalized_valid.max():.2f}] | "
                f"{runtime_sec:.2f}s"
            )

    result_df = pd.DataFrame(
        records
    )

    split_runtime = (
        time.perf_counter()
        -
        split_start
    )

    print("\n" + "-" * 106)
    print(
        f"{split_name.upper()} COMPLETE"
    )
    print("-" * 106)
    print(
        "Scenes:",
        len(
            result_df
        )
    )
    print(
        "Total runtime:",
        f"{split_runtime:.2f}s"
    )
    print(
        "Mean valid coverage:",
        f"{result_df['valid_pct'].mean():.2f}%"
    )
    print(
        "Mean P(clear):",
        f"{result_df['mean_p_clear'].mean():.4f}"
    )

    return result_df


optical_validation_audit = (
    process_frozen_optical_split(
        validation_df,
        "validation",
        VAL_RELIABILITY_ROOT
    )
)

optical_test_audit = (
    process_frozen_optical_split(
        test_df,
        "test",
        TEST_RELIABILITY_ROOT
    )
)

optical_eval_audit = pd.concat(
    [
        optical_validation_audit,
        optical_test_audit
    ],
    ignore_index=True
)

assert len(
    optical_eval_audit
) == 126

summary_table = (
    optical_eval_audit
    .groupby(
        "split"
    )
    .agg(
        scenes=(
            "scene_id",
            "count"
        ),
        mean_valid_pct=(
            "valid_pct",
            "mean"
        ),
        min_valid_pct=(
            "valid_pct",
            "min"
        ),
        mean_p_clear=(
            "mean_p_clear",
            "mean"
        ),
        min_normalized_value=(
            "normalized_min",
            "min"
        ),
        max_normalized_value=(
            "normalized_max",
            "max"
        ),
        mean_runtime_sec=(
            "runtime_sec",
            "mean"
        )
    )
    .reset_index()
)

print("\n" + "=" * 106)
print("FROZEN OPTICAL PIPELINE — CROSS-SPLIT QA")
print("=" * 106)

print(
    summary_table.to_string(
        index=False
    )
)

train_ids = set(
    train_df[
        "scene_id"
    ]
)

validation_ids = set(
    validation_df[
        "scene_id"
    ]
)

test_ids = set(
    test_df[
        "scene_id"
    ]
)

assert train_ids.isdisjoint(
    validation_ids
)
assert train_ids.isdisjoint(
    test_ids
)
assert validation_ids.isdisjoint(
    test_ids
)

assert (
    len(train_ids)
    +
    len(validation_ids)
    +
    len(test_ids)
    ==
    440
)

print("\nSplit leakage check:")
print(
    "  TRAIN ∩ VALIDATION = 0"
)
print(
    "  TRAIN ∩ TEST       = 0"
)
print(
    "  VALIDATION ∩ TEST  = 0"
)
print(
    "  Total retained     = 440 ✅"
)

for _, row in optical_eval_audit.iterrows():
    assert Path(
        row[
            "reliability_path"
        ]
    ).exists()

OPTICAL_EVAL_ROOT = (
    OPTICAL_ROOT
    /
    "evaluation"
)

OPTICAL_EVAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

VAL_AUDIT_PATH = (
    OPTICAL_EVAL_ROOT
    /
    "optical_validation_frozen_pipeline_audit.csv"
)

TEST_AUDIT_PATH = (
    OPTICAL_EVAL_ROOT
    /
    "optical_test_frozen_pipeline_audit.csv"
)

CROSS_SPLIT_SUMMARY_PATH = (
    OPTICAL_EVAL_ROOT
    /
    "optical_cross_split_qa_summary.csv"
)

optical_validation_audit.to_csv(
    VAL_AUDIT_PATH,
    index=False
)

optical_test_audit.to_csv(
    TEST_AUDIT_PATH,
    index=False
)

summary_table.to_csv(
    CROSS_SPLIT_SUMMARY_PATH,
    index=False
)

OPTICAL_FROZEN_PIPELINE_PATH = (
    OPTICAL_LOCK_ROOT
    /
    "optical_pipeline_frozen_v1.json"
)

pipeline_payload = {
    "status":
        "FROZEN",

    "retained_scene_count":
        440,

    "splits":
        {
            "train":
                314,

            "validation":
                68,

            "test":
                58
        },

    "raw_bands":
        13,

    "encoder_bands":
        OPTICAL_ENCODER_BANDS,

    "scale":
        "raw / 10000",

    "clipping":
        False,

    "validity":
        "all_13_bands_finite_and_nonzero",

    "normalization":
        "per-band TRAIN-only z-score",

    "normalization_train_pixel_count":
        TOTAL_TRAIN_VALID_PIXELS,

    "reliability_model":
        "OmniCloudMask",

    "reliability_definition":
        "renormalized P(clear)",

    "invalid_reliability":
        0.0,

    "hard_cloud_threshold":
        None,

    "cloud_inpainting":
        False,

    "morphology":
        False,

    "train_reliability_maps":
        str(
            OCM_TRAIN_MAP_ROOT
        ),

    "validation_reliability_maps":
        str(
            VAL_RELIABILITY_ROOT
        ),

    "test_reliability_maps":
        str(
            TEST_RELIABILITY_ROOT
        ),

    "normalized_optical_storage":
        "generated_on_the_fly",

    "validation_used_for_preprocessing_tuning":
        False,

    "test_used_for_preprocessing_tuning":
        False
}

with open(
    OPTICAL_FROZEN_PIPELINE_PATH,
    "w"
) as f:
    json.dump(
        pipeline_payload,
        f,
        indent=2
    )

print("\nSaved:")
print(" ", VAL_AUDIT_PATH)
print(" ", TEST_AUDIT_PATH)
print(" ", CROSS_SPLIT_SUMMARY_PATH)
print(" ", OPTICAL_FROZEN_PIPELINE_PATH)

print("\n" + "=" * 106)
print("P3-C42 completed successfully ✅")
print("=" * 106)
print(
    "Frozen optical preprocessing applied to all 440 scenes."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C43
# SECTION: P3-S26 — FINAL GLOBAL PREPROCESSING PACKAGE + LOCK
#
# PURPOSE:
#   Produce the canonical preprocessing package consumed by
#   all later FloodShift segmentation experiments.
#
# THIS CELL PERMANENTLY CLOSES PREPROCESSING DEVELOPMENT.
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

print("=" * 110)
print("P3-C43 — FINAL GLOBAL PREPROCESSING PACKAGE + PERMANENT LOCK")
print("=" * 110)

EXPECTED_COUNTS = {
    "train":
        314,

    "validation":
        68,

    "test":
        58
}

assert len(
    master
) == 440

for split_name, expected_count in EXPECTED_COUNTS.items():

    actual_count = int(
        (
            master[
                "split"
            ]
            ==
            split_name
        ).sum()
    )

    assert actual_count == expected_count

print("\nCanonical retained dataset:")
print(
    "  TRAIN      :",
    EXPECTED_COUNTS[
        "train"
    ]
)
print(
    "  VALIDATION :",
    EXPECTED_COUNTS[
        "validation"
    ]
)
print(
    "  TEST       :",
    EXPECTED_COUNTS[
        "test"
    ]
)
print(
    "  TOTAL      :",
    len(master)
)

required_optical_lock_paths = [
    OPTICAL_POLICY_PATH,
    OPTICAL_RELIABILITY_LOCK_PATH,
    OPTICAL_BAND_LOCK_PATH,
    OPTICAL_NORMALIZATION_LOCK_PATH,
    OPTICAL_FROZEN_PIPELINE_PATH
]

for path in required_optical_lock_paths:
    assert Path(
        path
    ).exists()

print(
    "\nOptical lock artifacts verified:",
    len(
        required_optical_lock_paths
    )
)

SAR_FINAL_ROOT_EXPECTED = (
    OUTPUT_ROOT
    /
    "sar_final"
)

SAR_FINAL_SHORTLIST_PATH = (
    SAR_FINAL_ROOT_EXPECTED
    /
    "sar_downstream_shortlist_FINAL.csv"
)

SAR_GLOBAL_LOCK_PATH = (
    SAR_FINAL_ROOT_EXPECTED
    /
    "sar_preprocessing_lock.json"
)

SAR_DPSSD_RECIPE_PATH = (
    SAR_FINAL_ROOT_EXPECTED
    /
    "dpssd_calibrated_recipe.json"
)

SAR_ABLATION_PATH = (
    SAR_FINAL_ROOT_EXPECTED
    /
    "sar_preprocessing_ablation_only.csv"
)

required_sar_paths = [
    SAR_FINAL_SHORTLIST_PATH,
    SAR_GLOBAL_LOCK_PATH,
    SAR_DPSSD_RECIPE_PATH,
    SAR_ABLATION_PATH
]

missing_sar_paths = [
    str(path)
    for path in required_sar_paths
    if not Path(
        path
    ).exists()
]

if missing_sar_paths:
    raise RuntimeError(
        "P3-C33 SAR final lock is missing.\n"
        +
        "\n".join(
            missing_sar_paths
        )
    )

print(
    "SAR lock artifacts verified:",
    len(
        required_sar_paths
    )
)

final_sar_shortlist = pd.read_csv(
    SAR_FINAL_SHORTLIST_PATH
)

EXPECTED_SAR_METHODS = [
    "Raw SAR",
    "Lee Sigma",
    "Homomorphic BM3D",
    "DP-SSD L0",
    "DP-SSD Calibrated"
]

assert list(
    final_sar_shortlist[
        "method"
    ]
) == EXPECTED_SAR_METHODS

selected_alpha = float(
    final_sar_shortlist.loc[
        final_sar_shortlist[
            "method"
        ]
        ==
        "DP-SSD Calibrated",
        "alpha"
    ].iloc[0]
)

assert np.isclose(
    selected_alpha,
    0.50
)

print("\nFrozen downstream SAR candidates:")
for method in EXPECTED_SAR_METHODS:
    print(
        " ",
        method
    )

print(
    "DP-SSD calibrated alpha:",
    selected_alpha
)

# Resolve canonical JRC column name robustly across manifest
# versions produced by Notebook 01.
JRC_PATH_COLUMN = next(
    (
        column
        for column in [
            "jrc_water_path",
            "jrc_path",
            "JRCWater_path"
        ]
        if column in master.columns
    ),
    None
)

if JRC_PATH_COLUMN is None:
    raise KeyError(
        "Could not locate the canonical JRC water path column."
    )

assert OPTICAL_ENCODER_BANDS == [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12"
]

assert (
    OPTICAL_RELIABILITY_DEFINITION
    ==
    "renormalized_P_clear"
)

assert (
    OPTICAL_RELIABILITY_HARD_THRESHOLD
    is None
)


def reliability_path_for_scene(
    scene_id,
    split_name
):
    if split_name == "train":
        root = OCM_TRAIN_MAP_ROOT

    elif split_name == "validation":
        root = VAL_RELIABILITY_ROOT

    elif split_name == "test":
        root = TEST_RELIABILITY_ROOT

    else:
        raise ValueError(
            f"Unknown split: {split_name}"
        )

    return Path(
        root
    ) / f"{scene_id}_pclear.npy"


integrity_records = []
package_rows = []

print("\n" + "=" * 110)
print("FINAL 440-SCENE INTEGRITY AUDIT")
print("=" * 110)

for position, (_, row) in enumerate(
    master.iterrows(),
    start=1
):

    scene_id_i = str(
        row[
            "scene_id"
        ]
    )

    split_name = str(
        row[
            "split"
        ]
    )

    sar_path = Path(
        locate_input_file(
            row[
                "sar_raw_path"
            ]
        )
    )

    optical_path = Path(
        locate_input_file(
            row[
                OPTICAL_PATH_COLUMN
            ]
        )
    )

    flood_label_path = Path(
        locate_input_file(
            row[
                "flood_label_path"
            ]
        )
    )

    jrc_path = Path(
        locate_input_file(
            row[
                JRC_PATH_COLUMN
            ]
        )
    )

    reliability_path = (
        reliability_path_for_scene(
            scene_id_i,
            split_name
        )
    )

    assert sar_path.exists()
    assert optical_path.exists()
    assert flood_label_path.exists()
    assert jrc_path.exists()
    assert reliability_path.exists()

    reliability = np.load(
        reliability_path
    )

    assert reliability.shape == (
        512,
        512
    )

    assert np.isfinite(
        reliability
    ).all()

    reliability_min = float(
        reliability.min()
    )

    reliability_max = float(
        reliability.max()
    )

    assert reliability_min >= 0.0
    assert reliability_max <= 1.0

    package_rows.append({
        "scene_id":
            scene_id_i,

        "region":
            row[
                "region"
            ],

        "split":
            split_name,

        "sar_raw_path":
            str(
                sar_path
            ),

        "optical_raw_path":
            str(
                optical_path
            ),

        "flood_label_path":
            str(
                flood_label_path
            ),

        "jrc_water_path":
            str(
                jrc_path
            ),

        "optical_reliability_path":
            str(
                reliability_path
            ),

        "optical_band_set":
            "B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12",

        "optical_scale":
            "raw/10000",

        "optical_normalization_id":
            "train_zscore_v1",

        "optical_reliability_id":
            "omnicloudmask_pclear_v1",

        "sar_recipe_registry":
            "sar_preprocessing_lock_v1",

        "target_source":
            "flood_label_plus_jrc",

        "target_materialization":
            "deferred_to_segmentation_loader",

        "qc_status":
            "PASS"
    })

    integrity_records.append({
        "scene_id":
            scene_id_i,

        "split":
            split_name,

        "sar_exists":
            True,

        "optical_exists":
            True,

        "flood_label_exists":
            True,

        "jrc_exists":
            True,

        "reliability_exists":
            True,

        "reliability_finite":
            True,

        "reliability_min":
            reliability_min,

        "reliability_max":
            reliability_max,

        "status":
            "PASS"
    })

    if (
        position == 1
        or
        position % 100 == 0
        or
        position == len(
            master
        )
    ):
        print(
            f"[{position:03d}/440] "
            f"{scene_id_i:24s} | "
            f"{split_name:10s} | PASS"
        )

preprocessing_manifest = pd.DataFrame(
    package_rows
)

integrity_audit = pd.DataFrame(
    integrity_records
)

assert len(
    preprocessing_manifest
) == 440

assert (
    integrity_audit[
        "status"
    ]
    ==
    "PASS"
).all()

train_ids = set(
    preprocessing_manifest.loc[
        preprocessing_manifest[
            "split"
        ]
        ==
        "train",
        "scene_id"
    ]
)

validation_ids = set(
    preprocessing_manifest.loc[
        preprocessing_manifest[
            "split"
        ]
        ==
        "validation",
        "scene_id"
    ]
)

test_ids = set(
    preprocessing_manifest.loc[
        preprocessing_manifest[
            "split"
        ]
        ==
        "test",
        "scene_id"
    ]
)

assert train_ids.isdisjoint(
    validation_ids
)

assert train_ids.isdisjoint(
    test_ids
)

assert validation_ids.isdisjoint(
    test_ids
)

print("\nSplit leakage:")
print(
    "  TRAIN ∩ VALIDATION = 0"
)
print(
    "  TRAIN ∩ TEST       = 0"
)
print(
    "  VALIDATION ∩ TEST  = 0"
)

FINAL_PREPROCESSING_ROOT = (
    OUTPUT_ROOT
    /
    "final_preprocessing"
)

FINAL_PREPROCESSING_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MANIFEST_PATH = (
    FINAL_PREPROCESSING_ROOT
    /
    "floodshift_preprocessing_manifest_v1.csv"
)

FINAL_INTEGRITY_PATH = (
    FINAL_PREPROCESSING_ROOT
    /
    "floodshift_preprocessing_integrity_audit_v1.csv"
)

FINAL_REGISTRY_PATH = (
    FINAL_PREPROCESSING_ROOT
    /
    "floodshift_preprocessing_registry_v1.json"
)

GLOBAL_LOCK_PATH = (
    FINAL_PREPROCESSING_ROOT
    /
    "floodshift_preprocessing_GLOBAL_LOCK_v1.json"
)

preprocessing_manifest.to_csv(
    FINAL_MANIFEST_PATH,
    index=False
)

integrity_audit.to_csv(
    FINAL_INTEGRITY_PATH,
    index=False
)

normalization_registry = {
    band_name:
        {
            "mean":
                float(
                    OPTICAL_NORM_MEAN[
                        idx
                    ]
                ),

            "std":
                float(
                    OPTICAL_NORM_STD[
                        idx
                    ]
                )
        }
    for idx, band_name in enumerate(
        OPTICAL_ENCODER_BANDS
    )
}

preprocessing_registry = {
    "version":
        "FloodShift_Preprocessing_v1",

    "dataset":
        "Sen1Floods11 v1.1 HandLabeled",

    "retained_scenes":
        440,

    "splits":
        {
            "train":
                314,

            "validation":
                68,

            "test":
                58
        },

    "sar":
        {
            "channels":
                [
                    "VV",
                    "VH"
                ],

            "input_domain":
                "dB",

            "downstream_candidates":
                EXPECTED_SAR_METHODS,

            "proposed_calibrated_method":
                {
                    "name":
                        "DP-SSD Calibrated",

                    "alpha":
                        0.50,

                    "radiometric_correction":
                        "per-scene per-band linear-intensity mean preservation",

                    "blend":
                        "I_final = 0.5 * I_dpssd_corrected + 0.5 * I_raw"
                },

            "screening_ablation_only":
                [
                    "Refined Lee",
                    "SAR2SAR",
                    "DP-SSD L1"
                ],

            "final_winner_selected":
                False,

            "winner_selection":
                "standardized downstream segmentation benchmark",

            "recipe_lock":
                str(
                    SAR_GLOBAL_LOCK_PATH
                ),

            "candidate_table":
                str(
                    SAR_FINAL_SHORTLIST_PATH
                )
        },

    "optical":
        {
            "raw_band_count":
                13,

            "raw_band_order":
                OPTICAL_BAND_NAMES,

            "raw_source_preserved":
                True,

            "scale":
                "raw / 10000",

            "clipping":
                False,

            "strict_validity":
                "all_13_bands_finite_and_nonzero",

            "encoder_band_count":
                10,

            "encoder_bands":
                OPTICAL_ENCODER_BANDS,

            "normalization":
                {
                    "method":
                        "per-band z-score",

                    "statistics_source":
                        "canonical TRAIN only",

                    "valid_train_pixels_per_band":
                        int(
                            TOTAL_TRAIN_VALID_PIXELS
                        ),

                    "cloud_pixels_included":
                        True,

                    "pclear_weighted":
                        False,

                    "invalid_fill":
                        0.0,

                    "values":
                        normalization_registry
                },

            "reliability":
                {
                    "model":
                        "OmniCloudMask",

                    "model_version":
                        str(
                            OCM_VERSION
                        ),

                    "model_input":
                        [
                            "B4",
                            "B3",
                            "B8"
                        ],

                    "definition":
                        "renormalized P(clear)",

                    "invalid_value":
                        0.0,

                    "hard_threshold":
                        None,

                    "morphology":
                        False,

                    "inpainting":
                        False
                }
        },

    "supervision":
        {
            "flood_label":
                "original Sen1Floods11 hand label",

            "permanent_water":
                "JRC water mask",

            "three_class_target":
                "constructed deterministically inside segmentation data loader",

            "target_precomputed_here":
                False
        }
}

with open(
    FINAL_REGISTRY_PATH,
    "w"
) as f:
    json.dump(
        preprocessing_registry,
        f,
        indent=2
    )


def sha256_file(
    path
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


critical_artifacts = {
    "final_manifest":
        FINAL_MANIFEST_PATH,

    "integrity_audit":
        FINAL_INTEGRITY_PATH,

    "sar_global_lock":
        SAR_GLOBAL_LOCK_PATH,

    "sar_shortlist":
        SAR_FINAL_SHORTLIST_PATH,

    "sar_dpssd_recipe":
        SAR_DPSSD_RECIPE_PATH,

    "optical_numerical_policy":
        OPTICAL_POLICY_PATH,

    "optical_reliability_lock":
        OPTICAL_RELIABILITY_LOCK_PATH,

    "optical_band_lock":
        OPTICAL_BAND_LOCK_PATH,

    "optical_normalization_lock":
        OPTICAL_NORMALIZATION_LOCK_PATH,

    "optical_pipeline_lock":
        OPTICAL_FROZEN_PIPELINE_PATH,

    "preprocessing_registry":
        FINAL_REGISTRY_PATH
}

artifact_hashes = {
    name:
        sha256_file(
            path
        )
    for name, path in critical_artifacts.items()
}

holdout_qa = {
    "validation":
        {
            "scene_count":
                68,

            "mean_valid_pct":
                float(
                    optical_validation_audit[
                        "valid_pct"
                    ].mean()
                ),

            "mean_p_clear":
                float(
                    optical_validation_audit[
                        "mean_p_clear"
                    ].mean()
                ),

            "min_z":
                float(
                    optical_validation_audit[
                        "normalized_min"
                    ].min()
                ),

            "max_z":
                float(
                    optical_validation_audit[
                        "normalized_max"
                    ].max()
                )
        },

    "test":
        {
            "scene_count":
                58,

            "mean_valid_pct":
                float(
                    optical_test_audit[
                        "valid_pct"
                    ].mean()
                ),

            "mean_p_clear":
                float(
                    optical_test_audit[
                        "mean_p_clear"
                    ].mean()
                ),

            "min_z":
                float(
                    optical_test_audit[
                        "normalized_min"
                    ].min()
                ),

            "max_z":
                float(
                    optical_test_audit[
                        "normalized_max"
                    ].max()
                )
        },

    "policy":
        "descriptive QA only; no holdout-driven retuning or clipping"
}

global_lock = {
    "lock_name":
        "FloodShift Global Preprocessing Lock",

    "version":
        "v1",

    "status":
        "PERMANENTLY_FROZEN",

    "preprocessing_development_closed":
        True,

    "retained_scene_count":
        440,

    "manifest":
        str(
            FINAL_MANIFEST_PATH
        ),

    "registry":
        str(
            FINAL_REGISTRY_PATH
        ),

    "integrity_audit":
        str(
            FINAL_INTEGRITY_PATH
        ),

    "artifact_sha256":
        artifact_hashes,

    "holdout_qa":
        holdout_qa,

    "future_rules":
        {
            "change_geographic_split":
                False,

            "change_sar_despeckling_method":
                False,

            "retrain_dpssd":
                False,

            "change_dpssd_loss":
                False,

            "retune_dpssd_alpha":
                False,

            "add_new_sar_filter":
                False,

            "change_optical_scale":
                False,

            "change_optical_validity_rule":
                False,

            "change_optical_encoder_bands":
                False,

            "change_cloud_model":
                False,

            "change_cloud_reliability_definition":
                False,

            "introduce_cloud_threshold":
                False,

            "change_optical_normalization":
                False,

            "recompute_normalization_with_validation":
                False,

            "recompute_normalization_with_test":
                False,

            "holdout_driven_clipping":
                False,

            "execute_frozen_preprocessing_recipes":
                True,

            "cache_frozen_sar_outputs":
                True,

            "construct_segmentation_target_from_frozen_label_jrc":
                True
        },

    "next_stage":
        "Flood segmentation methodology and model development"
}

with open(
    GLOBAL_LOCK_PATH,
    "w"
) as f:
    json.dump(
        global_lock,
        f,
        indent=2
    )

assert FINAL_MANIFEST_PATH.exists()
assert FINAL_INTEGRITY_PATH.exists()
assert FINAL_REGISTRY_PATH.exists()
assert GLOBAL_LOCK_PATH.exists()

assert len(
    preprocessing_manifest
) == 440

assert (
    preprocessing_manifest[
        "qc_status"
    ]
    ==
    "PASS"
).all()

print("\n" + "=" * 110)
print("FINAL FLOODSHIFT PREPROCESSING PACKAGE")
print("=" * 110)

print(
    "Retained scenes        : 440"
)
print(
    "Train / Val / Test     : 314 / 68 / 58"
)
print(
    "SAR channels           : VV, VH"
)
print(
    "SAR downstream methods : 5 frozen candidates"
)
print(
    "DP-SSD calibrated alpha: 0.50"
)
print(
    "Optical raw bands      : 13 preserved"
)
print(
    "Optical encoder bands  : 10"
)
print(
    "Optical normalization  : TRAIN-only z-score"
)
print(
    "Optical reliability    : renormalized P(clear)"
)
print(
    "Reliability maps       : 440 / 440"
)
print(
    "Input integrity        : 440 / 440 PASS"
)
print(
    "Split leakage          : NONE"
)

print("\nHoldout optical QA:")
print(
    "  Validation mean P(clear):",
    f"{holdout_qa['validation']['mean_p_clear']:.4f}"
)
print(
    "  Test mean P(clear)      :",
    f"{holdout_qa['test']['mean_p_clear']:.4f}"
)
print(
    "  Validation z range      :",
    f"[{holdout_qa['validation']['min_z']:.3f}, "
    f"{holdout_qa['validation']['max_z']:.3f}]"
)
print(
    "  Test z range            :",
    f"[{holdout_qa['test']['min_z']:.3f}, "
    f"{holdout_qa['test']['max_z']:.3f}]"
)

print("\nSaved:")
print(" ", FINAL_MANIFEST_PATH)
print(" ", FINAL_INTEGRITY_PATH)
print(" ", FINAL_REGISTRY_PATH)
print(" ", GLOBAL_LOCK_PATH)

print("\n" + "=" * 110)
print("P3-C43 COMPLETED SUCCESSFULLY ✅")
print("=" * 110)
print(
    "FLOODSHIFT PREPROCESSING IS NOW 100% COMPLETE."
)
print(
    "PREPROCESSING DEVELOPMENT IS PERMANENTLY CLOSED."
)
print(
    "\nNEXT PHASE → FLOOD SEGMENTATION."
)
